# PersonaPlex Student — Production Dataset Generation (500 h+)

This notebook generates the complete **training dataset for the distilled PersonaPlex student**: 500+ hours of
natural, full-duplex, dual-channel conversations. It writes them in the exact token/frame format that
`distill/train.py` → `distill/data/dataset.py` reads, with no conversion step.

**How the data is made (teacher self-play, like PersonaPlex inference):**
1. A **simulated user** talks to the real **PersonaPlex-7B teacher**:
   - the user side comes from real Q&A and dialogue datasets (finance, banking, insurance, customer service,
     general knowledge, health, casual conversation), spoken by ~730 synthetic speakers and by real accented
     human speakers;
   - it goes through a realistic microphone path: room, noise, device, Opus codec, echo.
2. The **teacher answers live**, in a chosen voice (one of the 18 UI voices or 220 extra real voices) and a chosen
   persona prompt (2,000+ personas in PersonaPlex's own prompt style).
3. The simulator **reacts to the teacher in real time**: it waits for the teacher to finish a turn, barges in,
   backchannels ("mm-hmm"), pauses, says "hello?" when the teacher goes quiet, and so on.
4. **Every frame both sides produce** (text + 8 agent codebooks + 8 user codebooks) is recorded, including the
   voice/persona prompt prefix, exactly as the model sees it at inference.

**Properties:** resumable at every stage, multi-GPU, multi-pod (shared volume), deterministic from one seed,
fully logged in `logs/`, automatically validated.

### Step 1 — smoke test first (`SMOKE_TEST = True`, the default)
A tiny but **complete** run of every stage: real downloads (streamed / 1 parquet file each), the real teacher,
real Kokoro TTS, real self-play, assembly, validation (including the teacher alignment test) and Whisper QA.
- **Size:** about 0.5 h of conversation (~30 samples), 2 UI voices, 30 personas per category.
- **Output:** a **separate folder** (`ppx_student_dataset_smoke/`), so it never mixes with the real dataset.
- **Result:** the last cell prints a **SMOKE TEST PASSED / FAILED** checklist, one line per stage.

### Step 2 — the real run (`SMOKE_TEST = False`, `TARGET_HOURS = 500`, or 200, or any number)
Set the two variables in the CONFIG cell, restart the kernel and run again from Section 1.

### Changing the size later (`TARGET_HOURS`)
Every sample is keyed by (split, index), so the plan for 200 h is an exact subset of the plan for 500 h.
- **Increase** (e.g. 200 → 500): change `TARGET_HOURS` and re-run from Section 6. Section 12 re-plans
  automatically, and self-play generates **only the new samples**; finished ones are kept.
- **Decrease** (e.g. 500 → 200): same steps. Samples outside the smaller plan are parked in
  `_work/excluded_codes/` by the assemble step (not deleted) and come back automatically if you increase again.
- Stop running workers first (`stop_stage("selfplay")`); the planner refuses to re-plan under running workers.

### Run order
Run the cells top to bottom:
- Sections 1–4: setup.
- Section 5: writes the generator package.
- Sections 6–17: pipeline stages. Each stage **skips work that is already done**, so after any crash or pod
  restart you just re-run the same cells.
- The long stages (TTS, self-play) run as **detached background workers**. Closing the browser doesn't stop them;
  the **Monitor** cells re-attach.

> Only a GPU run can produce throughput and time numbers. The notebook **measures** them (`logs/progress/`) and
> prints an ETA; nothing here is a guessed benchmark.

## Verified data specification (checked against the repo source)

| Requirement | Value used | Where it comes from in the repo |
|---|---|---|
| Audio rate / frame | 24 kHz mono; 12.5 Hz frames; 1920 samples/frame | `loaders.py` `SAMPLE_RATE`, `FRAME_RATE` |
| Streams per frame | **17**: row 0 text, rows 1–8 agent Mimi codes, rows 9–16 user Mimi codes | `get_moshi_lm` sets `dep_q=16`, `n_q=16`; `server.py` asserts `tokens.shape[1] == dep_q + 1` |
| Token ranges | audio 0–2047 (2048 = initial token); text 0–31999 (32000 = text initial) | `lm.py` `initial_token_id`, `text_initial_token_id` |
| Delays | `[0, 0,1,1,1,1,1,1,1, 0,1,1,1,1,1,1,1]` → `max_delay = 1` | `_lm_kwargs["delays"]` |
| Text specials | 3 = PAD (`zero_text_code`), 0 = end-of-padding | `lm.py` |
| Prompt prefix | 125 voice frames (agent = voice codes, text 3, user = `SINE_TOKENS`) → 6 spacer frames (agent = `SILENCE_TOKENS`) → **T persona frames** (one token per frame) → 6 spacer frames | `LMGen._step_voice_prompt/_audio_silence/_text_prompt`; `audio_silence_frame_cnt = int(0.5*12.5) = 6` in `server.py` |
| Frame 0 | the first voice frame is overwritten by the initial token and never returned; recorded frame *j* = LMGen call *j+1* | `LMGen.prepare_step_input` / `process_transformer_output` (verified by running the real `LMGen`) |
| **prefix_len** | **136 + T** recorded frames | derived from the rows above, verified in a test |
| Voice prompt | raw wav → −24 LUFS → Mimi, frame by frame, from a freshly reset streaming state | `LMGen.load_voice_prompt`, `_encode_voice_prompt_frames` |
| `.pt` voices | **teacher-only** embedding caches, refused for the student | `server.py` / `offline.py` guards |
| Persona prompt | `tokenizer.encode("<system> {text} <system>")`, **never empty** (empty → `None` → crash in `_step_text_prompt_core`) | `server.wrap_with_system_tags` |
| User channel (conversation) | Mimi, streaming, of the **Opus-decoded microphone signal** every 80 ms. `SINE_TOKENS` only appear inside the prefix | `server.opus_loop` |
| Silence on the user side | Mimi encoding of **real room tone / low noise**, never a constant token | same as above (a real mic is never digital zero) |
| Mimi reset | after the prompts, before the conversation | `server.handle_chat` |
| Teacher sampling | temp 0.8 / 0.7, top-k 250 / 25 | `LMGen` defaults (server passes none) |
| Training file format | `<split>/manifest.jsonl` + `<split>/codes/<sample_id>.pt` = LongTensor `[17, num_frames]` | `distill/data/dataset.py` |
| Chunking | `collate_chunks` takes the **first** `CHUNK_FRAMES` frames and drops shorter samples | same |
| Turn-taking weighting | frames within ±8 of `transition_frames` get 4× loss weight | `distill/losses.py: frame_weights` |
| Student context | 1500 frames (120 s) | `distill/configs/student_ppx_s.yaml: max_frames` |

### Corrections to the earlier plan (found while reading the code)
- **No empty persona prompts.** The server crashes on them, so they can't occur at inference.
- **Voice prompts are exactly 10.0 s (125 frames).** This keeps the prefix length fixed. The streaming notebook's
  own voice render produced a clip of **8.8 s (110 frames)**, a mismatch with training. So this notebook exports
  `voices/agent/ui_deploy/*.wav` (the exact training clips); **use them as `STUDENT_VOICE_DIR` when streaming**.
- **Real human speech is ~20–25 % of user speech time, not 40 %.** No openly licensed corpus has real people
  asking *our* Q&A questions. So real speech is used where content doesn't need to match (casual conversation,
  accents, backchannels, some generic turns), plus your own recordings (`MY_RECORDINGS_DIR`).
- **Size is 500 h** (your request) instead of 200 h. `CHUNK_FRAMES = 1000` is confirmed (prefix 144–256 frames +
  ≥ 744 conversation frames). A separate `train_long/` set uses 1500 frames = the student's full context.
- **Chatterbox (voice-cloning TTS) is not used.** It pins `torch==2.6.0` (no RTX 5090 kernels) and
  `safetensors==0.5.3` (conflicts with moshi). Kokoro is used instead; accent diversity comes from real speech.

## Dataset design (production defaults: `ppxdata/config_default.py`)

| Aspect | Design |
|---|---|
| **Size** | ≥ **500 h of conversation** after validation (prefix excluded; +6 % planned overprovision). About 28 k samples |
| **Sample length** | 1000 frames (80 s incl. prefix) → `train/ val/ test/ test_deploy/`; 1500 frames (120 s) → `train_long/ val_long/` (10 % of hours) |
| **Splits** | train 95 % / val 2 % / test 3 %, plus `test_deploy` (400 samples: 18 UI voices × deployment clips). Held out **by entity**: personas, questions, chains, TTS speakers, real speakers, 20 extra agent voices, 1 of 6 UI clips per voice, noise files |
| **Scenarios** | multi-turn Q&A 30 %, barge-in 14 %, single + follow-up 10 %, backchannels 10 %, casual (real speech) 10 %, pauses/late start 8 %, rapid exchange 6 %, distractors (non-speech events) 6 %, repeat/clarify 6 % |
| **Conversation start** | wait for the agent's greeting 50 %, user speaks first 35 %, late start (4–10 s) 15 % |
| **Personas** | ~1,870 in PersonaPlex's own prompt format: crypto assistant 34 % (dedicated cryptocurrency-educator roles), identity 10 % (Robert / RB Labs, one bounded fact-shape per persona so it fits the token budget), assistant 22 %, casual 14 %, role-play 8 %, finance 6 %, customer service 6 % (finance/customer-service kept small, generic-dialogue diversity only -- their HF text sources are disabled below). Identity and customer-service/finance personas carry `Information:` facts plus matching questions (faq) |
| **User content** | **Cryptocurrency (primary domain, synthetic/procedural, ~1,100+ question phrasings + multi-turn chains covering Bitcoin/Ethereum/wallets/DeFi/NFTs/mining/staking/exchanges/scams/regulation/etc, no download needed)**, general-conversation behavior (synthetic), Robert/RB Labs identity Q&A (synthetic) + Dolly, Natural Questions, Bitext customer-support, OASST1 (multi-turn), UltraChat (multi-turn), SODA (casual multi-turn) + persona-specific questions + follow-ups / clarifications / interruptions / disfluencies / greetings. `dataset_text_preview/` dumps every generated question/chain/persona to `.txt` for manual review |
| **Agent voices** | 70 % the 18 UI voices (6 teacher-rendered 10 s clips each; 30 % of their samples use the deployment clip) + 30 % **220 extra real voices** (VCTK, LibriTTS-R, EdAcc, VoxPopuli; gender/accent balanced) |
| **User voices** | 728 Kokoro speakers (28 US/UK bases + 700 blends, individual speaking rates) + real speakers from EdAcc (40+ L1 accents) and VoxPopuli (accented) + your own recordings |
| **Backchannels / interruptions** | 18 backchannel phrases per TTS speaker + real backchannels mined from EdAcc; interruptions are real questions with 14 interruption openers + 16 short stop phrases (including explicit "stop", "stop talking", "that's enough", "never mind", "wait", "hold on"), triggered 0.8–5 s into the agent's speech |
| **Silence / noise** | continuous room tone under all speech. Noise: clean 15 %, quiet room 45 % (−62…−48 dBFS), moderate 25 % (SNR 15–25 dB), noisy 10 % (5–15 dB), very noisy 5 % (0–5 dB). Types: MUSAN ambient/music/babble + procedural noise; non-speech events |
| **Mic path** | room reverb 60 % (RT60 0.15–0.8 s), device headset / laptop / phone / cheap, level −30…−18 dBFS, Opus 12–48 kbps, agent echo 6 %, clipping 1 % |
| **Timing** | response gaps lognormal ~0.5 s (0.1–2 s); rapid 0.05–0.4 s; long pauses 2.5–8 s; end of turn = 5–9 idle agent frames; "hello?" nudge after 5 s of no response; cut-in after a 20–30 s monologue |

## 1. Environment check

In [ ]:
import os, sys, platform, shutil, subprocess
print("Platform:", platform.platform())
print("Python  :", sys.version)
assert (3, 10) <= sys.version_info[:2] < (3, 13), "kokoro 0.9.4 requires Python >=3.10,<3.13 (verified on PyPI)"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. CONFIG

**The two switches:** `SMOKE_TEST` (tiny full test run into a separate folder) and `TARGET_HOURS` (size of the
real run). Everything else here is paths and parallelism. The dataset design lives in `ppxdata/config_default.py`
(Section 5). Changing the design (not the hours) **after** the `plan` stage is refused (Section 6), because the
generated data would no longer match the plan.

**Storage (estimate):** teacher weights ~17 GB, HF parquet cache ~30 GB, MUSAN ~22 GB (tar + extracted),
extracted real speech ~15 GB, TTS audio ~25 GB, codes ~5 GB, previews ~6 GB. **Use a ≥ 250 GB network volume.**

In [ ]:
import os, sys, json, shutil, subprocess, pathlib

# ==== WHAT TO GENERATE =================================================================================
SMOKE_TEST   = False    # True : tiny end-to-end run of EVERY stage (~0.5 h) into a separate folder -- run this first.
                       # False: the real dataset (TARGET_HOURS below).
TARGET_HOURS = 100     # real run: conversation hours after validation (e.g. 200, 500, 1000). Ignored when SMOKE_TEST.
                       # Can be changed later; finished samples are reused (see "Changing the size" at the top).
# =======================================================================================================

WORKSPACE     = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL      = "https://github.com/MoshiHead/personaplex-helium-distillation-v4.git"
REPO_DIR      = os.path.join(WORKSPACE, "personaplex")          # needs moshi/ and distill/
DATASET_ROOT  = os.path.join(WORKSPACE, "ppx_student_dataset_smoke" if SMOKE_TEST else "ppx_student_dataset_v1")
CODE_DIR      = os.path.join(WORKSPACE, "ppx_datagen")           # the generator package is written here
HF_CACHE_DIR  = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID    = "nvidia/personaplex-7b-v1"

# ---- parallelism -------------------------------------------------------------------------------------
NUM_GPUS          = None   # None = all visible GPUs
SELFPLAY_PER_GPU  = 1      # teacher (~17 GB) + batch; use 2 only on >= 80 GB GPUs
TTS_PER_GPU       = 2
UI_VOICE_PER_GPU  = 1
CPU_WORKERS       = max(2, (os.cpu_count() or 4) // 2)
WORKER_ID_OFFSET  = 0      # multi-pod on one shared volume: give each pod a distinct offset (0, 100, 200, ...)

# ---- optional inputs -------------------------------------------------------------------------------------
MY_RECORDINGS_DIR = None   # folder of your own mic recordings (wav/flac + optional same-name .txt transcript)
CONFIG_OVERRIDES  = {}     # e.g. {"gen": {...}} -- applied on top of config_default BEFORE the plan stage
FORCE_CONFIG_UPDATE = False

for d in (WORKSPACE, DATASET_ROOT, CODE_DIR, HF_CACHE_DIR):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert SMOKE_TEST or float(TARGET_HOURS) > 0, "TARGET_HOURS must be > 0"
free_gb = shutil.disk_usage(DATASET_ROOT).free / 1e9
# rough storage need: fixed downloads (teacher, parquet, MUSAN) + per-hour TTS audio / codes / previews
need_gb = 30 if SMOKE_TEST else 75 + 0.12 * float(TARGET_HOURS)
print(f"MODE         {'SMOKE TEST (~0.5 h, all stages)' if SMOKE_TEST else f'REAL RUN, {TARGET_HOURS} h'}")
print(f"DATASET_ROOT {DATASET_ROOT}  (free disk: {free_gb:.0f} GB, estimated need ~{need_gb:.0f} GB)")
if free_gb < need_gb:
    print("WARNING: free disk is below the estimate -- enlarge the volume before the long stages.")

## 3. System + Python dependencies

- **System packages:** `ffmpeg` (Opus mic simulation, decoding), `libopus-dev` (sphn), `espeak-ng` (Kokoro
  out-of-vocabulary G2P), `libsndfile1`.
- **moshi** is installed with `--no-deps` plus its runtime deps, because its `huggingface-hub<0.25` pin conflicts
  with `datasets`/`transformers` (the conflict was also visible in your earlier run's log).
- **GPU check** runs in a fresh subprocess, like the workers. PyTorch is reinstalled (cu130, with `--upgrade`)
  **only** if the installed build lacks kernels for this GPU. torch ≥ 2.7 cu128 already supports the RTX 5090 (`sm_120`).

In [ ]:
SUDO = "" if os.geteuid() == 0 else "sudo "
!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates ffmpeg libopus-dev espeak-ng libsndfile1 curl
print("system packages ok:", shutil.which("ffmpeg") is not None, shutil.which("espeak-ng") is not None)

In [ ]:
import pathlib, subprocess
if not (pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
assert (pathlib.Path(REPO_DIR) / "distill" / "data" / "dataset.py").exists(), "REPO_DIR must contain distill/ (training code)"
print("repo ok:", REPO_DIR)

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q --no-deps "{REPO_DIR}/moshi/."
%pip install -q "numpy<2.2" "sphn>=0.1.4,<0.2" einops sentencepiece safetensors huggingface_hub
%pip install -q "datasets>=2.19" pyarrow soundfile scipy pyloudnorm jiwer pyyaml tqdm psutil "transformers>=4.40"
%pip install -q "kokoro==0.9.4" "misaki[en]==0.9.4"
!{sys.executable} -m spacy download en_core_web_sm -q

In [ ]:
# GPU / PyTorch check -- runs in FRESH subprocesses (that is how every generation worker runs); the kernel itself
# never initializes CUDA. If PyTorch can't use the GPU, the NVIDIA driver is queried directly (cuInit via ctypes,
# no PyTorch) to get the exact CUDA error code, and the one cause that is fixable inside the container -- a
# libcuda on LD_LIBRARY_PATH that doesn't match the host driver -- is repaired automatically.
import glob, json, os, subprocess, sys

TORCH_PROBE = r'''
import json
r = {}
try:
    import torch
    # compile-time kernel list; unlike torch.cuda.get_arch_list() it does not need a working GPU (that returns [])
    flags = torch._C._cuda_getArchFlags() if hasattr(torch._C, "_cuda_getArchFlags") else ""
    r.update(torch=torch.__version__, cuda=torch.version.cuda, arch=sorted(set((flags or "").split())))
    r["available"] = torch.cuda.is_available()
    if r["available"]:
        r["gpus"] = torch.cuda.device_count()
        r["name"] = torch.cuda.get_device_name(0)
        x = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16)
        (x @ x).sum().item()
        r["bf16_matmul_ok"] = True
except Exception as e:
    r["error"] = repr(e)
print("PROBE" + json.dumps(r))
'''

DRIVER_PROBE = r'''
import ctypes, json
r = {}
try:
    lib = ctypes.CDLL("libcuda.so.1")
    v = ctypes.c_int()
    lib.cuDriverGetVersion(ctypes.byref(v))
    r["libcuda_api_version"] = v.value                     # e.g. 13010 = CUDA 13.1
    r["cuInit"] = lib.cuInit(0)                            # 0 = success; otherwise a CUresult code
    n = ctypes.c_int()
    r["cuDeviceGetCount"] = lib.cuDeviceGetCount(ctypes.byref(n))
    r["device_count"] = n.value
    r["libcuda_loaded_from"] = sorted({l.split()[-1] for l in open("/proc/self/maps") if "libcuda.so" in l})
except OSError as e:
    r["load_error"] = str(e)
print("CUPROBE" + json.dumps(r))
'''

CU_ERRORS = {
    0: "success", 3: "CUDA_ERROR_NOT_INITIALIZED", 35: "CUDA_ERROR_INSUFFICIENT_DRIVER", 46: "CUDA_ERROR_DEVICES_UNAVAILABLE",
    100: "CUDA_ERROR_NO_DEVICE", 101: "CUDA_ERROR_INVALID_DEVICE", 802: "CUDA_ERROR_SYSTEM_NOT_READY",
    803: "CUDA_ERROR_SYSTEM_DRIVER_MISMATCH", 804: "CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE", 999: "CUDA_ERROR_UNKNOWN",
}


def run_probe(code, tag, env=None):
    p = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, env=env)
    line = [l for l in p.stdout.splitlines() if l.startswith(tag)]
    r = json.loads(line[-1][len(tag):]) if line else {"error": "probe crashed", "stderr_tail": p.stderr[-1500:]}
    if tag == "PROBE" and not r.get("bf16_matmul_ok"):
        r["stderr_tail"] = p.stderr[-800:]
    return r


def smi(q):
    return subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()


cap = float(smi("compute_cap").split()[0] or 0)
need_arch = f"sm_{int(round(cap * 10))}"               # 12.0 -> sm_120 (RTX 5090 / Blackwell)
r = run_probe(TORCH_PROBE, "PROBE")
print("installed torch:", json.dumps({k: v for k, v in r.items() if k != "stderr_tail"}))

if not r.get("bf16_matmul_ok") and r.get("arch") and need_arch not in r["arch"]:
    # Only reinstall when the installed build is KNOWN to lack kernels for this GPU (e.g. moshi's old torch<2.5 pin);
    # an unknown/empty kernel list (CUDA-less build info) never triggers a reinstall.
    # --upgrade is required: without it pip keeps the old build as "already satisfied" and installs nothing.
    print(f"Installed torch has no {need_arch} kernels -> installing the cu130 build")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
    r = run_probe(TORCH_PROBE, "PROBE")

if not r.get("bf16_matmul_ok"):
    # ---------------- driver-level diagnosis (independent of PyTorch)
    d = run_probe(DRIVER_PROBE, "CUPROBE")
    d["cuInit_meaning"] = CU_ERRORS.get(d.get("cuInit"), "see CUDA driver API CUresult docs")
    host_driver = smi("driver_version")
    all_libcuda = sorted({p for pat in ["/usr/lib*/**/libcuda.so*", "/usr/local/**/libcuda.so*", "/opt/**/libcuda.so*"]
                          for p in glob.glob(pat, recursive=True)})
    facts = {
        "host_driver (nvidia-smi)": host_driver, "gpu": smi("name"), "compute_mode": smi("compute_mode"),
        "gpu_processes": subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                                         "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "none",
        "driver_probe": d, "libcuda_files_in_container": all_libcuda,
        "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH"),
        "NVIDIA_DRIVER_CAPABILITIES": os.environ.get("NVIDIA_DRIVER_CAPABILITIES"),
    }
    print("\nDRIVER-LEVEL DIAGNOSIS:\n" + json.dumps(facts, indent=1))

    # ---------------- automatic repair: a libcuda that doesn't belong to the host driver shadows the real one
    # (typical: CUDA forward-compat libs from the image, which GeForce/RTX GPUs do not support -> cuInit fails).
    bad_dirs = sorted({os.path.dirname(p) for p in all_libcuda
                       if "compat" in p or (host_driver and ".so." in p and not p.endswith(host_driver)
                                             and os.path.basename(p).count(".") >= 3)})
    lp = [x for x in (os.environ.get("LD_LIBRARY_PATH") or "").split(":") if x]
    clean_lp = [x for x in lp if not any(x.rstrip("/") == b.rstrip("/") for b in bad_dirs) and "compat" not in x]
    if d.get("cuInit") not in (0, None) and clean_lp != lp:
        env = dict(os.environ, LD_LIBRARY_PATH=":".join(clean_lp))
        r2 = run_probe(TORCH_PROBE, "PROBE", env=env)
        print("retry without mismatched libcuda dirs on LD_LIBRARY_PATH:", r2.get("bf16_matmul_ok", False))
        if r2.get("bf16_matmul_ok"):
            os.environ["LD_LIBRARY_PATH"] = ":".join(clean_lp)   # inherited by every worker launched later
            r = r2
            print("FIXED: removed", sorted(set(lp) - set(clean_lp)), "from LD_LIBRARY_PATH for this session "
                  "(re-run this cell after a kernel restart; it re-applies the fix).")

if not r.get("bf16_matmul_ok"):
    code_ = d.get("cuInit")
    advice = {
        999: ("CUDA_ERROR_UNKNOWN from cuInit() itself, below PyTorch, with the right driver library loaded and "
              "/dev/nvidia-uvm present: the GPU or the host driver is in a bad state (e.g. after an Xid error). "
              "Nothing inside the container can fix that. TERMINATE this pod and DEPLOY A NEW ONE (Stop/Start usually "
              "lands on the same machine and GPU). Data on your /workspace network volume is kept."),
        804: ("The container's libcuda is a CUDA forward-compat library; GeForce/RTX GPUs don't support forward "
              "compatibility. Remove the compat directory from LD_LIBRARY_PATH (listed above) or use an image built "
              "for CUDA <= the host driver's version."),
        803: ("libcuda in the container doesn't match the host kernel driver (see libcuda_files_in_container vs "
              "host_driver). Remove the mismatched libcuda from LD_LIBRARY_PATH / the image, or use another template."),
        46: ("GPU busy/unavailable: check gpu_processes and compute_mode above (Exclusive_Process + another process "
             "holding the GPU). Stop that process or restart the pod."),
        100: ("No CUDA device visible to the driver API although nvidia-smi sees one -- the container was started "
              "without GPU compute capability (check NVIDIA_DRIVER_CAPABILITIES includes 'compute'); redeploy the pod."),
    }.get(code_, "Unexpected driver state; send the DRIVER-LEVEL DIAGNOSIS above. Deploying a new pod is the safest fix.")
    raise RuntimeError(f"PyTorch cannot use the GPU: cuInit() = {code_} ({CU_ERRORS.get(code_, '?')}).\n{advice}")

print(f"\nGPU OK in a fresh process: torch {r['torch']} (CUDA {r['cuda']}), {r['gpus']} x {r['name']}, "
      f"{need_arch} kernels present, bf16 matmul ok.")
if "torch" in sys.modules:
    print("NOTE: this kernel imported torch earlier. Restart the kernel (Kernel > Restart) and continue from "
          "Section 2 so later cells don't use a stale module. Workers are separate processes and are unaffected.")

## 4. Hugging Face auth + teacher assets

The token comes from the `HF_TOKEN` env var (e.g. a RunPod secret), or from a hidden prompt. **Never hardcode it.**
Accept the license at [nvidia/personaplex-7b-v1](https://huggingface.co/nvidia/personaplex-7b-v1) first.

In [ ]:
import json, tarfile
from getpass import getpass
from huggingface_hub import login, hf_hub_download

token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
hf_token = 'hf' + token
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")
assets = {f: hf_hub_download(HF_REPO_ID, f) for f in
          ["config.json", "tokenizer_spm_32k_3.model", "tokenizer-e351c8d8-checkpoint125.safetensors",
           "model.safetensors", "voices.tgz"]}
vt = pathlib.Path(assets["voices.tgz"])
if not (vt.parent / "voices").exists():
    with tarfile.open(vt) as t:
        t.extractall(vt.parent)
TEACHER = {"hf_repo": HF_REPO_ID, "moshi_weight": assets["model.safetensors"],
           "mimi_weight": assets["tokenizer-e351c8d8-checkpoint125.safetensors"],
           "tokenizer": assets["tokenizer_spm_32k_3.model"], "voices_dir": str(vt.parent / "voices")}
print(json.dumps(TEACHER, indent=1))

## 5. Write the generator package (`ppxdata/`)

These modules were tested before being embedded here:
- **Recording layout:** checked with the real `LMGen` from this repo, using a small random `LMModel` with the
  exact 17-stream / `dep_q=16` / delay configuration. Negative controls (corrupted prefix, corrupted user frame,
  one-frame shift) are detected.
- **Planner, personas, text cleaning, audio preparation, user simulator:** checked on synthetic catalogs.

In [ ]:
# %%writefile does not create folders -- create the package folder before the module cells below.
import os
os.makedirs(os.path.join(CODE_DIR, "ppxdata"), exist_ok=True)
print("writing the generator package to", os.path.join(CODE_DIR, "ppxdata"))

In [ ]:
%%writefile {CODE_DIR}/ppxdata/__init__.py
"""PersonaPlex student-distillation dataset generator (see PersonaPlex_Student_Dataset_Generation.ipynb)."""

In [ ]:
%%writefile {CODE_DIR}/ppxdata/common.py
"""Shared constants, paths, logging, atomic IO, work-claiming and audio utilities.

Every constant below is copied from (and verified against) the PersonaPlex repo:
  moshi/moshi/models/lm.py      SILENCE_TOKENS, SINE_TOKENS, AUDIO_TOKENS_PER_STREAM, LMGen.zero_text_code
  moshi/moshi/models/loaders.py SAMPLE_RATE=24000, FRAME_RATE=12.5, text_card=32000, card=2048,
                                existing_text_padding_id=3, dep_q=16 (forced in get_moshi_lm), n_q=16,
                                delays=[0,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1] (max_delay=1)
"""
import hashlib
import io
import json
import logging
import math
import os
import random
import socket
import subprocess
import sys
import time
import traceback
from pathlib import Path

import numpy as np

# ---------------------------------------------------------------- model/token constants (verified)
SR = 24000
FRAME_RATE = 12.5
FRAME = 1920                       # samples per Mimi frame (24000 / 12.5)
N_CODEBOOKS = 17                   # 1 text + 8 agent ("moshi") + 8 user ("input")
AGENT_ROWS = slice(1, 9)
USER_ROWS = slice(9, 17)
AUDIO_CARD = 2048                  # valid audio ids 0..2047; 2048 = initial token
TEXT_CARD = 32000                  # valid text ids 0..31999; 32000 = text initial token
TEXT_EPAD = 0                      # end-of-padding (lm.end_of_text_padding_id)
TEXT_PAD = 3                       # padding (existing_text_padding_id) == LMGen.zero_text_code
SILENCE_TOKENS = [948, 243, 1178, 546, 1736, 1030, 1978, 2008]
SINE_TOKENS = [430, 1268, 381, 1611, 1095, 1495, 56, 472]
VOICE_SECONDS = 10.0
VOICE_FRAMES = 125                 # every agent voice prompt is exactly 10.0 s = 125 frames
VOICE_SAMPLES = VOICE_FRAMES * FRAME
SILENCE_SPACER = 6                 # LMGen audio_silence_frame_cnt = int(0.5 * 12.5) in server/offline
VOICE_LUFS = -24.0                 # LMGen.load_voice_prompt normalization

# Recorded-frame layout produced by LMGen (see selfplay.py for the derivation):
#   call 0 (first voice frame) is overwritten by the initial token and never returned,
#   recorded codes[:, j] == everything provided/sampled at call j+1.
#   calls: voice 0..124 | spacer 125..130 | text 131..130+T | spacer 131+T..136+T | conversation 137+T..
#   => prefix_len (in recorded codes) = 136 + T; conversation starts at codes[:, 136+T]
PREFIX_BASE = VOICE_FRAMES - 1 + 2 * SILENCE_SPACER          # = 136


def prefix_len(n_text_tokens: int) -> int:
    return PREFIX_BASE + n_text_tokens


def prefix_calls(n_text_tokens: int) -> int:
    return VOICE_FRAMES + 2 * SILENCE_SPACER + n_text_tokens  # = 137 + T


# ---------------------------------------------------------------- config / paths
def load_cfg(root) -> dict:
    with open(Path(root) / "config.json", encoding="utf-8") as f:
        return json.load(f)


class P:
    """All dataset paths derived from the dataset root."""

    def __init__(self, root):
        r = Path(root)
        self.root = r
        self.logs = r / "logs"
        self.meta = r / "meta"
        self.voices = r / "voices"
        self.agent_ui = r / "voices" / "agent" / "ui"
        self.agent_deploy = r / "voices" / "agent" / "ui_deploy"
        self.agent_extra = r / "voices" / "agent" / "extra"
        self.voice_codes = r / "voices" / "agent" / "codes"
        self.preview = r / "audio_preview"
        self.work = r / "_work"
        self.raw = r / "_work" / "raw"
        self.real = r / "_work" / "real_speech"
        self.tts = r / "_work" / "tts"
        self.tts_audio = r / "_work" / "tts" / "audio"
        self.tts_voices = r / "_work" / "tts" / "voices"
        self.plan = r / "_work" / "plan"
        self.locks = r / "_work" / "locks"
        self.results = r / "_work" / "results"
        self.done = r / "_work" / "done"
        self.noise = r / "_work" / "noise"

    def split_dir(self, split):
        return self.root / split

    def codes_path(self, split, sample_id):
        return self.root / split / "codes" / f"{sample_id}.pt"

    def mkdirs(self):
        for d in [self.logs, self.meta, self.agent_ui, self.agent_deploy, self.agent_extra, self.voice_codes,
                  self.preview, self.raw, self.real, self.tts_audio, self.tts_voices, self.plan, self.locks,
                  self.results, self.done, self.noise,
                  self.logs / "stages", self.logs / "workers", self.logs / "progress", self.logs / "errors",
                  self.logs / "samples", self.logs / "sources", self.logs / "validation", self.logs / "qa",
                  self.logs / "env"]:
            d.mkdir(parents=True, exist_ok=True)


SPLITS = ["train", "val", "test", "test_deploy", "train_long", "val_long"]


# ---------------------------------------------------------------- logging
def get_logger(name: str, logfile: Path | None = None) -> logging.Logger:
    lg = logging.getLogger(name)
    if getattr(lg, "_ppx_configured", False):
        return lg
    lg.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s %(levelname)s [%(name)s pid=%(process)d] %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    lg.addHandler(sh)
    if logfile is not None:
        Path(logfile).parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(logfile, encoding="utf-8")
        fh.setFormatter(fmt)
        lg.addHandler(fh)
    lg.propagate = False
    lg._ppx_configured = True
    return lg


def worker_tag(worker_id: int) -> str:
    return f"{socket.gethostname()}_w{worker_id}"


def log_error(paths: P, stage: str, worker: str, err: BaseException, **ctx):
    rec = {"time": now(), "stage": stage, "worker": worker, "error": repr(err),
           "traceback": traceback.format_exc(), **ctx}
    append_jsonl(paths.logs / "errors" / f"{stage}_{worker}.jsonl", rec)


def stage_log(paths: P, stage: str, status: str, **info):
    """logs/stages/<stage>.json keeps the latest status; <stage>.history.jsonl keeps every transition."""
    rec = {"time": now(), "stage": stage, "status": status, **info}
    atomic_write_json(paths.logs / "stages" / f"{stage}.json", rec)
    append_jsonl(paths.logs / "stages" / f"{stage}.history.jsonl", rec)


def stage_done(paths: P, stage: str) -> bool:
    f = paths.logs / "stages" / f"{stage}.json"
    if not f.exists():
        return False
    try:
        return json.loads(f.read_text(encoding="utf-8")).get("status") == "done"
    except Exception:
        return False


def now() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%S%z")


# ---------------------------------------------------------------- atomic io
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp{os.getpid()}")
    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj):
    atomic_write_bytes(path, json.dumps(obj, ensure_ascii=False, indent=1).encode("utf-8"))


def atomic_torch_save(path, obj):
    import torch
    buf = io.BytesIO()
    torch.save(obj, buf)
    atomic_write_bytes(path, buf.getvalue())


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(obj, ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()


def read_jsonl(path):
    out = []
    p = Path(path)
    if not p.exists():
        return out
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # a torn last line from a killed process -- ignored, the work unit is redone
    return out


def write_jsonl(path, rows):
    buf = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    atomic_write_bytes(path, buf.encode("utf-8"))


# ---------------------------------------------------------------- deterministic randomness
def stable_int(*parts) -> int:
    h = hashlib.sha1("|".join(str(p) for p in parts).encode("utf-8")).hexdigest()
    return int(h[:15], 16)


def rng_for(*parts) -> np.random.Generator:
    return np.random.default_rng(stable_int(*parts))


def pyrng_for(*parts) -> random.Random:
    return random.Random(stable_int(*parts))


def split_of(key: str, seed, val: float, test: float) -> str:
    """Deterministic held-out assignment for an entity (voice, speaker, persona, question, noise file)."""
    u = (stable_int("split", seed, key) % 1_000_000) / 1_000_000
    if u < test:
        return "test"
    if u < test + val:
        return "val"
    return "train"


def weighted_choice(rng, weights: dict):
    keys = list(weights)
    w = np.array([weights[k] for k in keys], dtype=np.float64)
    return keys[int(rng.choice(len(keys), p=w / w.sum()))]


# ---------------------------------------------------------------- work claiming (multi-worker / multi-pod safe)
class Claimer:
    """File-lock based claiming of work units on a shared filesystem.

    A unit is done when `done/<stage>/<unit>.json` exists. A unit is claimed by atomically creating
    `locks/<stage>/<unit>.lock` (O_EXCL). The owner touches the lock (heartbeat) while working; a lock
    whose mtime is older than `stale_s` is considered abandoned (crashed pod/worker) and is reclaimed.
    """

    def __init__(self, paths: P, stage: str, worker: str, stale_s: float = 1800):
        self.lock_dir = paths.locks / stage
        self.done_dir = paths.done / stage
        self.lock_dir.mkdir(parents=True, exist_ok=True)
        self.done_dir.mkdir(parents=True, exist_ok=True)
        self.worker = worker
        self.stale_s = stale_s

    def is_done(self, unit: str) -> bool:
        return (self.done_dir / f"{unit}.json").exists()

    def try_claim(self, unit: str) -> bool:
        if self.is_done(unit):
            return False
        lock = self.lock_dir / f"{unit}.lock"
        for _ in range(2):
            try:
                fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                with os.fdopen(fd, "w") as f:
                    f.write(json.dumps({"worker": self.worker, "pid": os.getpid(), "time": now()}))
                return True
            except FileExistsError:
                try:
                    age = time.time() - lock.stat().st_mtime
                except FileNotFoundError:
                    continue
                if age > self.stale_s:
                    try:
                        os.replace(lock, lock.with_suffix(f".stale{int(time.time())}"))
                    except FileNotFoundError:
                        pass
                    continue
                return False
        return False

    def heartbeat(self, unit: str):
        try:
            os.utime(self.lock_dir / f"{unit}.lock", None)
        except FileNotFoundError:
            pass

    def mark_done(self, unit: str, info: dict):
        atomic_write_json(self.done_dir / f"{unit}.json", {"time": now(), "worker": self.worker, **info})
        try:
            (self.lock_dir / f"{unit}.lock").unlink()
        except FileNotFoundError:
            pass

    def release(self, unit: str):
        try:
            (self.lock_dir / f"{unit}.lock").unlink()
        except FileNotFoundError:
            pass


class Progress:
    """logs/progress/<stage>_<worker>.json -- heartbeat + throughput, read by the notebook monitor."""

    def __init__(self, paths: P, stage: str, worker: str, total_units: int):
        self.path = paths.logs / "progress" / f"{stage}_{worker}.json"
        self.stage, self.worker, self.total = stage, worker, total_units
        self.t0 = time.time()
        self.units_done = 0
        self.items_done = 0
        self.extra = {}

    def update(self, units=0, items=0, **extra):
        self.units_done += units
        self.items_done += items
        self.extra.update(extra)
        el = time.time() - self.t0
        atomic_write_json(self.path, {
            "stage": self.stage, "worker": self.worker, "time": now(), "elapsed_s": round(el, 1),
            "units_done_this_run": self.units_done, "items_done_this_run": self.items_done,
            "items_per_s": round(self.items_done / max(el, 1e-6), 4), "total_units": self.total, **self.extra})


# ---------------------------------------------------------------- audio utilities
def to_mono_f32(x) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        # soundfile returns [T, C]
        x = x.mean(axis=1) if x.shape[1] <= 8 else x.mean(axis=0)
    return np.ascontiguousarray(x, dtype=np.float32)


def resample(x: np.ndarray, sr_in: int, sr_out: int = SR) -> np.ndarray:
    if sr_in == sr_out:
        return x.astype(np.float32)
    from scipy.signal import resample_poly
    g = math.gcd(int(sr_in), int(sr_out))
    return resample_poly(x, sr_out // g, sr_in // g).astype(np.float32)


def decode_audio_bytes(data: bytes) -> tuple[np.ndarray, int]:
    """Decode wav/flac/ogg with soundfile; anything else (mp3/opus-in-webm) through ffmpeg."""
    import soundfile as sf
    try:
        x, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=False)
        return to_mono_f32(x), int(sr)
    except Exception:
        p = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", "pipe:0",
                            "-f", "f32le", "-ac", "1", "-ar", str(SR), "pipe:1"],
                           input=data, capture_output=True, check=True)
        return np.frombuffer(p.stdout, dtype=np.float32).copy(), SR


def read_audio(path) -> tuple[np.ndarray, int]:
    with open(path, "rb") as f:
        return decode_audio_bytes(f.read())


def load_24k(path) -> np.ndarray:
    x, sr = read_audio(path)
    return resample(x, sr, SR)


def write_flac(path, x: np.ndarray, sr: int = SR):
    import soundfile as sf
    buf = io.BytesIO()
    x = np.asarray(x)
    sf.write(buf, np.clip(x, -1, 1), sr, format="FLAC", subtype="PCM_16")
    atomic_write_bytes(path, buf.getvalue())


def write_wav(path, x: np.ndarray, sr: int = SR):
    import soundfile as sf
    buf = io.BytesIO()
    sf.write(buf, np.clip(np.asarray(x), -1, 1), sr, format="WAV", subtype="PCM_16")
    atomic_write_bytes(path, buf.getvalue())


def rms_db(x: np.ndarray) -> float:
    if len(x) == 0:
        return -120.0
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x, dtype=np.float64))) + 1e-9))


def frame_db(x: np.ndarray, hop: int = FRAME) -> np.ndarray:
    n = len(x) // hop
    if n == 0:
        return np.zeros(0)
    f = x[: n * hop].reshape(n, hop).astype(np.float64)
    return 20 * np.log10(np.sqrt((f ** 2).mean(axis=1)) + 1e-9)


def trim_silence(x: np.ndarray, sr: int = SR, rel_db: float = -35.0, pad_s: float = 0.08) -> np.ndarray:
    hop = int(0.02 * sr)
    d = frame_db(x, hop)
    if len(d) == 0:
        return x
    keep = np.where(d > d.max() + rel_db)[0]
    if len(keep) == 0:
        return x[:0]
    a = max(0, keep[0] * hop - int(pad_s * sr))
    b = min(len(x), (keep[-1] + 1) * hop + int(pad_s * sr))
    return x[a:b]


def voiced_ratio(x: np.ndarray, sr: int = SR, abs_db: float = -45.0) -> float:
    d = frame_db(x, int(0.02 * sr))
    return float((d > abs_db).mean()) if len(d) else 0.0


def gain_to(x: np.ndarray, target_db: float) -> np.ndarray:
    cur = rms_db(x)
    if cur < -100:
        return x
    return (x * (10 ** ((target_db - cur) / 20))).astype(np.float32)


def loudness_normalize(x: np.ndarray, sr: int = SR, target: float = VOICE_LUFS) -> np.ndarray:
    import pyloudnorm as pyln
    meter = pyln.Meter(sr)
    lufs = meter.integrated_loudness(x.astype(np.float64))
    if not np.isfinite(lufs):
        return x
    return pyln.normalize.loudness(x.astype(np.float64), lufs, target).astype(np.float32)


def synth_rir(rng: np.random.Generator, rt60: float, drr_db: float, sr: int = SR) -> np.ndarray:
    """Simple stochastic room impulse response: direct path + exponentially decaying diffuse tail."""
    n = int(max(0.05, rt60 * 1.2) * sr)
    t = np.arange(n) / sr
    decay = np.exp(-6.908 * t / max(rt60, 0.05))       # -60 dB at rt60
    tail = rng.standard_normal(n) * decay
    pre = int(rng.uniform(0.002, 0.012) * sr)          # early-reflection gap
    tail[:pre] = 0.0
    tail /= (np.sqrt(np.sum(tail ** 2)) + 1e-9)
    rir = tail * (10 ** (-drr_db / 20))
    rir[0] = 1.0
    return (rir / np.max(np.abs(rir))).astype(np.float32)


def apply_rir(x: np.ndarray, rir: np.ndarray | None) -> np.ndarray:
    if rir is None or len(x) == 0:
        return x
    from scipy.signal import fftconvolve
    y = fftconvolve(x, rir)[: len(x)]
    return gain_to(y.astype(np.float32), rms_db(x))


def apply_mic(x: np.ndarray, kind: str, rng: np.random.Generator, sr: int = SR) -> np.ndarray:
    """Device coloration: headset (flat), laptop (HPF+LPF+mild resonance), phone (narrowband), cheap (bandlimit+saturation)."""
    from scipy.signal import butter, sosfilt
    if len(x) == 0 or kind == "headset":
        return x
    if kind == "laptop":
        sos = butter(2, [float(rng.uniform(120, 200)), float(rng.uniform(6500, 9000))], btype="band", fs=sr, output="sos")
        return sosfilt(sos, x).astype(np.float32)
    if kind == "phone":
        sos = butter(4, [300, 3400], btype="band", fs=sr, output="sos")
        return sosfilt(sos, x).astype(np.float32)
    if kind == "cheap":
        sos = butter(2, [float(rng.uniform(150, 300)), float(rng.uniform(4500, 7000))], btype="band", fs=sr, output="sos")
        y = sosfilt(sos, x)
        drive = float(rng.uniform(1.5, 3.0))
        return (np.tanh(y * drive) / np.tanh(drive)).astype(np.float32)
    return x


def opus_roundtrip(x: np.ndarray, bitrate_kbps: int, sr: int = SR) -> np.ndarray:
    """Browser -> server path: the web client sends Opus; the server decodes it before Mimi (server.py)."""
    if len(x) == 0 or not bitrate_kbps:
        return x
    enc = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-f", "f32le", "-ar", str(sr), "-ac", "1",
                          "-i", "pipe:0", "-c:a", "libopus", "-b:a", f"{int(bitrate_kbps)}k", "-application", "voip",
                          "-f", "ogg", "pipe:1"], input=np.clip(x, -1, 1).astype(np.float32).tobytes(),
                         capture_output=True, check=True)
    dec = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", "pipe:0", "-f", "f32le",
                          "-ar", str(sr), "-ac", "1", "pipe:1"], input=enc.stdout, capture_output=True, check=True)
    y = np.frombuffer(dec.stdout, dtype=np.float32).copy()
    if len(y) >= len(x):
        return y[: len(x)]
    return np.concatenate([y, np.zeros(len(x) - len(y), np.float32)])


def synth_noise(kind: str, n: int, rng: np.random.Generator, sr: int = SR) -> np.ndarray:
    """Procedural backgrounds (always available, no download): white/pink/brown noise, mains hum, fan."""
    if n <= 0:
        return np.zeros(0, np.float32)
    w = rng.standard_normal(n).astype(np.float32)
    if kind == "white":
        y = w
    elif kind in ("pink", "fan"):
        f = np.fft.rfft(w)
        freqs = np.fft.rfftfreq(n, 1 / sr)
        f[1:] /= np.sqrt(freqs[1:])
        f[0] = 0
        y = np.fft.irfft(f, n).astype(np.float32)
        if kind == "fan":
            from scipy.signal import butter, sosfilt
            y = sosfilt(butter(2, float(rng.uniform(300, 900)), btype="low", fs=sr, output="sos"), y).astype(np.float32)
    elif kind == "brown":
        y = np.cumsum(w)
        y -= np.linspace(y[0], y[-1], n)
        y = y.astype(np.float32)
    elif kind == "hum":
        base = float(rng.choice([50.0, 60.0]))
        t = np.arange(n) / sr
        y = sum((0.6 ** k) * np.sin(2 * np.pi * base * (k + 1) * t + rng.uniform(0, 6.28)) for k in range(4))
        y = (y + 0.05 * w).astype(np.float32)
    else:
        y = w
    y = y / (np.std(y) + 1e-9)
    return y.astype(np.float32)


def loop_to(x: np.ndarray, n: int, rng: np.random.Generator, xfade: int = 2400) -> np.ndarray:
    """Tile a background recording to n samples with crossfades, from a random start offset."""
    if len(x) == 0:
        return np.zeros(n, np.float32)
    if len(x) >= n:
        s = int(rng.integers(0, len(x) - n + 1))
        return x[s:s + n].astype(np.float32)
    out = np.zeros(n, np.float32)
    pos = 0
    start = int(rng.integers(0, len(x)))
    seg = np.concatenate([x[start:], x[:start]]).astype(np.float32)
    fade = min(xfade, len(seg) // 4)
    ramp = np.linspace(0, 1, fade, dtype=np.float32) if fade > 0 else None
    out = seg.copy()
    while len(out) < n:
        if fade > 0:
            joint = out[-fade:] * (1 - ramp) + seg[:fade] * ramp
            out = np.concatenate([out[:-fade], joint, seg[fade:]])
        else:
            out = np.concatenate([out, seg])
    return out[:n].astype(np.float32)


def text_hash(t: str) -> str:
    return hashlib.sha1(" ".join(t.lower().split()).encode("utf-8")).hexdigest()[:16]


def env_versions() -> dict:
    out = {"python": sys.version, "host": socket.gethostname()}
    for m in ("torch", "numpy", "sphn", "sentencepiece", "kokoro", "misaki", "datasets", "transformers", "pyarrow",
              "soundfile", "scipy", "pyloudnorm"):
        try:
            mod = __import__(m)
            out[m] = getattr(mod, "__version__", "?")
        except Exception as e:
            out[m] = f"unavailable ({type(e).__name__})"
    try:
        import torch
        out["cuda"] = torch.version.cuda
        out["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        pass
    return out

In [ ]:
%%writefile {CODE_DIR}/ppxdata/config_default.py
"""Production configuration + smoke-test overrides. Every number is explained in the notebook's specification section."""

UI_VOICES = ["NATF0", "NATF1", "NATF2", "NATF3", "NATM0", "NATM1", "NATM2", "NATM3",
             "VARF0", "VARF1", "VARF2", "VARF3", "VARF4", "VARM0", "VARM1", "VARM2", "VARM3", "VARM4"]


def default_config(root, repo_dir, teacher, target_hours=500.0):
    return {
        "version": "ppx-student-data-v1",
        "seed": 20260926,
        "root": str(root),
        "repo_dir": str(repo_dir),
        "teacher": teacher,                       # hf_repo, moshi_weight, mimi_weight, tokenizer, voices_dir
        "smoke_test": False,

        # ---------------- size  (the ONE number to change: TARGET_HOURS in the notebook's CONFIG cell)
        "target_conv_hours": float(target_hours),  # conversation hours AFTER validation (prompt prefix not counted)
        "overprovision": 0.06,                    # planned extra to absorb rejected/failed samples
        "frames": {"main": 1000,                  # 80 s per sample incl. prefix -> CHUNK_FRAMES=1000
                   "long": 1500,                  # 120 s = student max_frames (context) -> train_long/val_long
                   "long_hours_frac": 0.10},
        "splits": {"val": 0.02, "test": 0.03, "test_deploy_samples": 400},
        "max_persona_tokens": 120,

        # ---------------- content mix (fractions of samples)
        "scenario_weights": {"multi_turn": 0.30, "single_followup": 0.10, "barge_in": 0.14, "backchannel": 0.10,
                             "pauses": 0.08, "rapid": 0.06, "casual_real": 0.10, "distractor": 0.06,
                             "repeat_clarify": 0.06},
        "start_policy_weights": {"wait_greeting": 0.50, "user_first": 0.35, "late_start": 0.15},
        "persona_category_weights": {"crypto": 0.34, "identity": 0.10, "assistant": 0.22, "casual": 0.14,
                                     "roleplay": 0.08, "finance": 0.06, "customer_service": 0.06},
        "personas": {"n_crypto": 700, "n_identity": 120, "n_customer_service": 150, "n_finance": 150,
                     "n_assistant": 300, "n_casual": 300, "n_roleplay": 150},

        # ---------------- voices / speakers
        "agent_voices": {"ui_voices": list(UI_VOICES), "ui_frac": 0.70, "ui_clips_per_voice": 6,
                         "deploy_clip_weight": 0.30,
                         "extra_per_corpus": {"vctk": 60, "libritts_r": 70, "edacc": 40, "voxpopuli": 30},
                         "heldout_extra_voices": 20},
        "tts": {"n_blend_speakers": 700, "us_fraction": 0.7, "rate_range": [0.88, 1.12], "jobs_per_shard": 2000},
        "real_speech": {
            "enabled": {"edacc": True, "voxpopuli": True, "vctk": True, "libritts_r": True},
            "max_files": {"edacc": 16, "voxpopuli": 2, "vctk": 27, "libritts_r": 30},
            "user_utt_min_s": 0.8, "user_utt_max_s": 14.0, "agent_voice_material_s": 40.0,
            "real_speaker_sample_frac": 0.25,     # samples that also get a real speaker (for generic turns)
            "generic_turn_real_frac": 0.10,       # of those samples' non-first turns spoken by the real speaker
            "my_recordings_dir": None},

        # ---------------- text sources
        "text_sources": {
            "min_words": 3, "max_words": 40,
            "streaming": False,                   # True = read only the first max_rows rows (no full download)
            "enabled": {"finance_alpaca": False, "fiqa": False, "financial_qa_10k": False,
                        "bitext_banking": False, "bitext_insurance": False, "bitext_customer_support": True,
                        "dolly": True, "natural_questions": True, "medquad": False, "oasst1": True,
                        "ultrachat": True, "soda": True,
                        "crypto_synth": True, "general_synth": True, "identity_synth": True},
            "max_rows": {"finance_alpaca": 69000, "fiqa": 17110, "financial_qa_10k": 7000, "bitext_banking": 25545,
                         "bitext_insurance": 39000, "bitext_customer_support": 26872, "natural_questions": 60000,
                         "medquad": 16407, "ultrachat": 120000, "soda": 200000}},

        # ---------------- acoustics (user microphone path)
        "noise": {"musan": True, "musan_urls": ["https://www.openslr.org/resources/17/musan.tar.gz",
                                                 "https://us.openslr.org/resources/17/musan.tar.gz"]},
        "acoustics": {
            "noise_class_weights": {"clean": 0.15, "quiet_room": 0.45, "moderate": 0.25, "noisy": 0.10,
                                    "very_noisy": 0.05},
            "bg_type_weights": {"ambient": 0.45, "music": 0.15, "babble": 0.15, "procedural": 0.25},
            "event_prob_other": 0.10, "rir_prob": 0.60,
            "mic_weights": {"headset": 0.35, "laptop": 0.35, "phone": 0.15, "cheap": 0.15},
            "speech_db_range": [-30.0, -18.0], "turn_db_jitter": 3.0,
            "opus_kbps_weights": {"12": 0.05, "16": 0.15, "24": 0.30, "32": 0.35, "48": 0.15},
            "echo_prob": 0.06, "clipping_prob": 0.01},

        # ---------------- turn-taking timing (seconds unless noted)
        "timing": {"gap_normal_s": [0.1, 2.0], "gap_rapid_s": [0.05, 0.4], "gap_pause_s": [2.5, 8.0],
                   "user_first_s": [0.3, 2.0], "late_start_s": [4.0, 10.0], "greeting_wait_s": 3.0,
                   "end_idle_frames": [5, 9], "no_response_s": 5.0, "max_monologue_s": [20.0, 30.0],
                   "barge_after_s": [0.8, 5.0], "bc_every_s_backchannel": [2.5, 5.0], "bc_every_s_other": [5.0, 9.0]},

        # ---------------- generation engine
        "gen": {"group_b": 8, "max_batch": None, "shard_groups": 6, "temp": 0.8, "temp_text": 0.7, "top_k": 250,
                "top_k_text": 25, "max_retries": 2, "stale_lock_s": 1800, "prep_threads": 6},
        "preview_audio_frac": 0.01,               # stereo previews of 1% of train (+ all val/test) for listening/QA
    }


def smoke_overrides():
    """SMOKE TEST: every stage runs for real (real downloads, real teacher, real TTS, real validation/QA) but tiny.

    Sized so that every split, scenario family and code path is exercised, while staying small:
      * ~0.5 h of conversation (~25 samples at 1000 frames + a few 1500-frame and deploy samples)
      * 2 UI voices x 2 clips, 3 extra voices per corpus, 30 personas per category, 20 TTS blend speakers
      * text sources STREAMED (first rows only), 1 parquet file per real-speech corpus, no 11 GB MUSAN download
      * larger val/test fractions so every held-out entity pool is non-empty (keeps the leakage check meaningful)
    The data format is identical to production (same 1000/1500-frame samples, same prefix layout).
    """
    return {
        "smoke_test": True,
        "target_conv_hours": 0.5,
        "overprovision": 0.5,
        "splits": {"val": 0.15, "test": 0.15, "test_deploy_samples": 4},
        "personas": {"n_crypto": 30, "n_identity": 15, "n_customer_service": 30, "n_finance": 30,
                     "n_assistant": 30, "n_casual": 30, "n_roleplay": 30},
        "agent_voices": {"ui_voices": ["NATF2", "NATM1"], "ui_clips_per_voice": 2,
                         "extra_per_corpus": {"vctk": 3, "libritts_r": 3, "edacc": 3, "voxpopuli": 3},
                         "heldout_extra_voices": 4},
        "tts": {"n_blend_speakers": 20, "jobs_per_shard": 200},
        "real_speech": {"max_files": {"edacc": 1, "voxpopuli": 1, "vctk": 1, "libritts_r": 1}},
        "text_sources": {"streaming": True,
                         "max_rows": {k: 1500 for k in ["finance_alpaca", "fiqa", "financial_qa_10k", "bitext_banking",
                                                        "bitext_insurance", "bitext_customer_support", "dolly",
                                                        "natural_questions", "medquad", "oasst1", "ultrachat", "soda"]}},
        "noise": {"musan": False},
        "gen": {"shard_groups": 2, "max_retries": 1},
        "preview_audio_frac": 1.0,
    }

In [ ]:
%%writefile {CODE_DIR}/ppxdata/text_sources.py
"""Stage `text`: download real-world Q&A / dialogue datasets and turn them into spoken-style USER turns.

Only the user side is taken from these datasets. The agent side is never scripted: the PersonaPlex teacher
answers live during self-play (selfplay.py), which is what the student must imitate.

Outputs (meta/):
  questions.jsonl  {qid, text, domain, source, license, split}                 single user questions
  chains.jsonl     {chain_id, turns:[...], domain, source, license, split}     multi-turn user-turn sequences
logs/sources/<source>.json: rows read, kept, and drop counts by reason.
"""
import argparse
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict

from .common import P, load_cfg, get_logger, stage_log, atomic_write_json, write_jsonl, split_of, text_hash, now

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MD_RE = re.compile(r"[*_#`>|~]+")
BRACKET_RE = re.compile(r"\[[^\]]*\]|\([^)]*http[^)]*\)")
PLACEHOLDER_RE = re.compile(r"\{\{\s*([^}]+?)\s*\}\}")
WS_RE = re.compile(r"\s+")
CODE_RE = re.compile(r"\w\(|\)\s*:|\b(def|lambda|import|printf|elif|SELECT \*)\b|==|->|;\s*$")
SPACE_PUNCT_RE = re.compile(r"\s+([?.!,;:])")
LOWER_I_RE = re.compile(r"\bi\b")
BAD_CHARS = set("{}<>=\\^$@§")
BLOCKLIST = {"porn", "nude", "sex ", "suicide", "kill myself", "bomb", "nazi", "rape", "cocaine", "heroin",
             "meth ", "terrorist", "racist", "nigg", "fuck", "shit", "bitch", "cunt"}
QUESTION_START = ("what", "how", "why", "when", "where", "who", "which", "is ", "are ", "can ", "could ", "should ",
                  "do ", "does ", "did ", "will ", "would ", "was ", "were ", "have ", "has ", "whats ", "how's ")

# Values for Bitext {{Placeholder}} slots -- filled deterministically per row so the spoken text is natural.
PLACEHOLDER_VALUES = {
    "account number": ["account ending in 4 5 2 1", "account ending in 7 7 0 3", "my checking account"],
    "account type": ["checking", "savings", "joint", "business checking"],
    "credit card": ["my credit card", "my Visa card", "my rewards card"],
    "credit card number": ["card ending in 0 9 8 2", "card ending in 3 1 1 4"],
    "person name": ["Maria Lopez", "James Chen", "Aisha Khan", "Tom Becker", "Priya Nair"],
    "customer support phone number": ["the support line", "your phone line"],
    "customer support hours": ["your support hours", "business hours"],
    "website url": ["your website", "the app"],
    "live chat support": ["live chat"],
    "order number": ["order 5 5 1 2 0", "order 8 8 3 4 1"],
    "invoice number": ["invoice 2 2 7 1", "invoice 9 0 4 5"],
    "refund amount": ["forty dollars", "one hundred twenty dollars"],
    "currency symbol": ["dollars"],
    "delivery city": ["Chicago", "Toronto", "Leeds", "Austin"],
    "delivery country": ["Canada", "the UK", "Germany"],
    "store location": ["the downtown store", "the mall location"],
    "policy number": ["policy number 6 6 1 0 2", "my home insurance policy"],
    "insurance type": ["car insurance", "home insurance", "travel insurance", "life insurance"],
    "claim number": ["claim 3 0 9 1", "claim 7 4 2 2"],
    "loan type": ["mortgage", "car loan", "personal loan", "student loan"],
    "transfer amount": ["five hundred dollars", "two thousand dollars"],
    "amount": ["two hundred dollars", "fifty dollars", "one thousand dollars"],
    "date": ["March third", "next Friday", "the fifteenth"],
    "program": ["the rewards program", "the loyalty program"],
}


# =====================================================================================================
# Synthetic domains (no external download; procedurally combined so the same fact can be asked in many
# different ways -- template x topic combinatorics -- exactly like a human would vary their phrasing).
# =====================================================================================================

# ---------------- cryptocurrency knowledge (the primary domain for this ~1B student)
CRYPTO_WHAT_IS_TERMS = [
    "Bitcoin", "Ethereum", "a cryptocurrency", "blockchain technology", "a blockchain", "a block",
    "a smart contract", "a decentralized application", "a decentralized exchange", "a centralized exchange",
    "a crypto wallet", "a hardware wallet", "a software wallet", "a hot wallet", "a cold wallet",
    "a public key", "a private key", "a seed phrase", "a recovery phrase", "a transaction hash",
    "a crypto address", "a token", "a coin", "a stablecoin", "an NFT", "DeFi", "Web3", "a DAO",
    "a gas fee", "a transaction fee", "a network confirmation", "cryptocurrency mining", "staking",
    "proof of work", "proof of stake", "a consensus mechanism", "a blockchain node", "a blockchain fork",
    "a hard fork", "a soft fork", "market capitalization", "trading volume", "liquidity", "volatility",
    "a bull market", "a bear market", "a Layer 1 network", "a Layer 2 network", "cold storage",
    "hot storage", "a custodial wallet", "a non-custodial wallet", "a crypto exchange", "a crypto scam",
    "a rug pull", "a 51 percent attack", "a whitepaper", "an airdrop", "a memecoin", "a satoshi", "gwei",
    "cryptocurrency regulation", "the crypto market", "Solana", "Cardano", "a stablecoin like USDC",
]
CRYPTO_HOW_WORKS_TERMS = [
    "Bitcoin", "blockchain technology", "cryptocurrency mining", "staking", "a smart contract",
    "a crypto wallet", "proof of work", "proof of stake", "a decentralized exchange",
    "a cryptocurrency transaction", "sending cryptocurrency", "buying cryptocurrency", "a crypto exchange",
    "cryptocurrency pricing", "a blockchain fork", "gas fees", "a DAO", "cold storage",
    "consensus mechanisms", "an NFT marketplace",
]
CRYPTO_INVENT_TERMS = ["Bitcoin", "Ethereum"]
CRYPTO_DIFF_PAIRS = [
    ("Bitcoin", "Ethereum"), ("a coin", "a token"), ("proof of work", "proof of stake"),
    ("a hard fork", "a soft fork"), ("a hot wallet", "a cold wallet"),
    ("a custodial wallet", "a non-custodial wallet"), ("a centralized exchange", "a decentralized exchange"),
    ("a Layer 1 network", "a Layer 2 network"), ("Bitcoin", "Solana"), ("Ethereum", "Cardano"),
    ("a bull market", "a bear market"), ("a coin", "a stablecoin"),
]
WHAT_IS_TEMPLATES = [
    "What is {x}?", "Can you explain {x} to me?", "Tell me what {x} is.",
    "I don't really understand {x}. Can you explain it?", "Could you give me a simple explanation of {x}?",
    "What exactly does {x} mean?", "How would you describe {x} to someone who is new to crypto?",
    "What's {x}, exactly?", "Can you break {x} down for me?", "I keep hearing about {x}, what is it?",
    "Define {x} for me.", "In simple terms, what is {x}?",
]
HOW_WORKS_TEMPLATES = [
    "How does {x} work?", "Can you explain how {x} works?", "Walk me through how {x} works.",
    "I don't get how {x} works, can you explain?", "What's the process behind {x}?",
    "How exactly does {x} function?", "Could you describe how {x} operates, step by step?",
]
WHO_INVENTED_TEMPLATES = [
    "Who invented {x}?", "Who created {x}?", "Who's behind {x}?", "Who came up with {x}?",
    "Who is the creator of {x}?",
]
WHEN_INVENTED_TEMPLATES = [
    "When was {x} invented?", "When did {x} start?", "When was {x} created?", "What year did {x} launch?",
    "How long has {x} been around?",
]
DIFF_TEMPLATES = [
    "What's the difference between {a} and {b}?", "How is {a} different from {b}?",
    "{a} versus {b}, what's the difference?", "Can you compare {a} and {b}?",
]
CRYPTO_CLUSTERS = [
    ["What are the risks of cryptocurrency?", "What risks should I know about before using crypto?",
     "Is crypto risky?"],
    ["Is cryptocurrency safe to use?", "Is it safe to use crypto day to day?",
     "How safe is cryptocurrency in general?"],
    ["Is cryptocurrency safe to invest in?", "Is investing in crypto safe?",
     "Should I be worried about losing money in crypto?"],
    ["How should someone evaluate a cryptocurrency project?", "How do I know if a crypto project is legit?",
     "What should I look for before investing in a coin?"],
    ["What are common cryptocurrency scams?", "What kinds of crypto scams should I watch out for?",
     "How do people get scammed with crypto?"],
    ["How can users protect their wallets and accounts?", "How do I keep my crypto wallet safe?",
     "What's the best way to secure my crypto accounts?"],
    ["What happens when someone loses their private key?", "What if I lose my private key?",
     "Can I recover my crypto if I lose the key?"],
    ["What happens if a transaction is sent to the wrong address?",
     "I sent crypto to the wrong address, what happens now?", "Can a crypto transaction be reversed?"],
    ["How do you buy cryptocurrency?", "How can I buy crypto?", "What's the process for buying Bitcoin?"],
    ["How do you sell cryptocurrency?", "How can I sell my crypto?", "What's the process for selling Bitcoin?"],
    ["How do you send cryptocurrency to someone?", "How do I send crypto to another wallet?",
     "How does sending crypto work?"],
    ["How do you receive cryptocurrency?", "How do I receive crypto from someone?",
     "How does receiving crypto work?"],
    ["How do cryptocurrency exchanges work?", "What does a crypto exchange actually do?",
     "How does trading on an exchange work?"],
    ["What causes cryptocurrency prices to change?", "Why do crypto prices move so much?",
     "What makes Bitcoin's price go up or down?"],
    ["What is market capitalization in crypto?", "What does market cap mean for a cryptocurrency?",
     "How is a coin's market cap calculated?"],
    ["What is trading volume in crypto?", "What does trading volume mean?", "Why does trading volume matter?"],
    ["What is liquidity in crypto?", "What does liquidity mean for a cryptocurrency?",
     "Why is liquidity important when trading crypto?"],
    ["What is volatility in cryptocurrency?", "Why is crypto so volatile?",
     "What does it mean when people say crypto is volatile?"],
    ["What are the different types of cryptocurrencies?", "What kinds of cryptocurrencies are there?",
     "How many types of crypto exist?"],
    ["Do all cryptocurrencies work the same way?", "What makes one cryptocurrency different from another?",
     "How do different cryptocurrencies compare to each other?"],
    ["What are the advantages and disadvantages of cryptocurrency?", "What are the pros and cons of using crypto?",
     "Why would someone use crypto over regular money?"],
    ["What is cryptocurrency regulation?", "How is cryptocurrency regulated?", "Are there laws about crypto?"],
    ["What is the history of Bitcoin?", "How did Bitcoin get started?", "Tell me about Bitcoin's history."],
    ["Tell me about Bitcoin in detail.", "Give me a detailed explanation of Bitcoin.",
     "I want to understand Bitcoin thoroughly, can you explain it in depth?"],
    ["What are network confirmations?", "How many confirmations does a crypto transaction need?",
     "Why do I have to wait for confirmations?"],
    ["How do miners earn cryptocurrency?", "What does it mean to mine crypto?",
     "What is cryptocurrency mining, exactly?"],
    ["What is staking cryptocurrency?", "How does staking earn rewards?", "What does it mean to stake a coin?"],
    ["What is a stablecoin used for?", "Why would someone use a stablecoin instead of Bitcoin?",
     "What makes a stablecoin stable?"],
    ["What are NFTs used for?", "Why do people buy NFTs?", "What can you do with an NFT?"],
    ["What is DeFi used for?", "Why is DeFi important?", "What problems does DeFi solve?"],
    ["What is Web3?", "How is Web3 different from the regular internet?", "What does Web3 mean?"],
    ["What is a DAO used for?", "How does a DAO make decisions?", "Why would people form a DAO?"],
    ["What's a good first step for someone completely new to crypto?",
     "I've never used crypto before, where should I start?", "How should a beginner get started with crypto?"],
]
CRYPTO_CHAINS = [
    ["What is Bitcoin?", "How does it work?", "Is it safe to invest in?"],
    ["What is Ethereum?", "How is it different from Bitcoin?", "What can you build on it?"],
    ["What is a crypto wallet?", "What's the difference between a hot wallet and a cold wallet?",
     "How do I keep mine secure?"],
    ["What is blockchain technology?", "How does mining work?", "What is proof of work?"],
    ["What is staking?", "How is that different from mining?", "Is staking safe?"],
    ["What is DeFi?", "How is it different from a regular bank?", "What are the risks?"],
    ["What is an NFT?", "Why would someone buy one?", "How do I buy one?"],
    ["How do I buy cryptocurrency?", "Which exchange should I use?", "What fees should I expect?"],
    ["What is a seed phrase?", "What happens if I lose it?", "How should I store it safely?"],
    ["What is a stablecoin?", "Why would someone use one instead of Bitcoin?", "Are stablecoins really stable?"],
    ["What is a smart contract?", "What can it be used for?", "Is it safe to use one?"],
    ["What are gas fees?", "Why do gas fees go up?", "Can I avoid paying high gas fees?"],
    ["Who invented Bitcoin?", "When was it invented?", "Why was it created?"],
    ["What's the difference between a coin and a token?", "Can you give me an example of each?",
     "Does that affect how I store them?"],
    ["What is proof of stake?", "How is it different from proof of work?", "Which one is more energy efficient?"],
]

# ---------------- general conversational behavior (greetings, short answers, meta questions about the assistant)
GENERAL_CHAT_CLUSTERS = [
    ["How are you?", "How are you doing?", "How's it going?", "How are you feeling today?"],
    ["What are you doing?", "What are you up to?", "What are you doing right now?"],
    ["Can you help me?", "Could you help me with something?", "I need some help, can you assist?"],
    ["Can you explain something to me?", "Could you explain something for me?",
     "I have something I'd like you to explain."],
    ["Can you answer questions?", "Are you able to answer questions?", "Can I ask you questions?"],
    ["What topics do you know about?", "What subjects can you talk about?", "What can you talk about?"],
    ["Can you have a conversation with me?", "Can we just chat for a bit?", "Do you like having conversations?"],
    ["Can you remember what I said?", "Do you remember what I told you earlier?",
     "Will you remember this conversation?"],
    ["What do you do when you don't know something?", "What happens if you don't know the answer?",
     "What if you can't answer my question?"],
    ["Can you explain something simply?", "Can you keep it simple?", "Can you explain that in easy terms?"],
    ["Can you explain something in detail?", "Can you go into more detail?", "Can you give me a detailed answer?"],
    ["Thank you.", "Thanks a lot.", "Thanks so much, I appreciate it."],
    ["Goodbye.", "Bye for now.", "I have to go, bye."],
    ["Hello.", "Hi there.", "Hey, how's it going?"],
    ["I don't know.", "I'm not sure.", "Not sure, honestly."],
    ["What's your favorite thing to talk about?", "What do you enjoy talking about?",
     "Is there a topic you like most?"],
    ["Can you keep this short?", "Can you give me a quick answer?", "Keep it brief please."],
    ["Can you slow down?", "Could you say that more slowly?", "Can you repeat that a bit slower?"],
    ["What did I just ask you?", "What was my last question?", "Can you remind me what I just said?"],
]

# ---------------- identity: who Robert is and who built RB Labs (paired with the identity persona at plan time)
IDENTITY_CLUSTERS = [
    ["Who are you?", "Who exactly are you?", "Tell me who you are."],
    ["What is your name?", "What's your name?", "Do you have a name?"],
    ["Who created you?", "Who made you?", "Who built you?"],
    ["When were you created?", "When did you come into existence?", "What year were you made?"],
    ["Where were you created?", "Where were you built?", "Where do you come from?"],
    ["Where are you located?", "Where are you right now?", "Where are you based?"],
    ["Tell me about yourself.", "Tell me something about yourself.", "Introduce yourself."],
    ["Can you introduce yourself?", "Could you introduce yourself to me?", "Please introduce yourself."],
    ["Are you a human or a robot?", "Are you a real person?", "Are you an AI or a person?"],
    ["What are you?", "What kind of thing are you?", "Are you a chatbot or something else?"],
    ["How old are you?", "How long have you existed?", "What's your age?"],
    ["What is your purpose?", "Why were you created?", "What are you here for?"],
    ["Who founded RB Labs?", "Who are the founders of RB Labs?", "Tell me about the people behind RB Labs."],
    ["Tell me about RB Labs.", "What is RB Labs?", "What does RB Labs do?"],
]


def normalize_unicode(t: str) -> str:
    t = unicodedata.normalize("NFKC", t)
    return t.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"') \
            .replace("—", ", ").replace("–", "-")


def fill_placeholders(t: str, key: str) -> str:
    def rep(m):
        name = m.group(1).strip().lower()
        vals = PLACEHOLDER_VALUES.get(name)
        if not vals:
            for k, v in PLACEHOLDER_VALUES.items():
                if k in name or name in k:
                    vals = v
                    break
        if not vals:
            return name
        return random.Random(f"{key}|{name}").choice(vals)
    return PLACEHOLDER_RE.sub(rep, t)


def clean_text(t: str) -> str:
    t = normalize_unicode(t or "")
    t = URL_RE.sub(" ", t)
    t = BRACKET_RE.sub(" ", t)
    t = MD_RE.sub(" ", t)
    t = t.replace("\n", " ").replace("\t", " ")
    t = WS_RE.sub(" ", t).strip(" -:;\"'")
    t = SPACE_PUNCT_RE.sub(r"\1", t)
    t = LOWER_I_RE.sub("I", t)
    return t


def spoken_reason(t: str, min_w: int, max_w: int) -> str | None:
    """None if `t` is usable as a spoken user turn, else the drop reason (logged per source)."""
    if not t:
        return "empty"
    words = t.split()
    if len(words) < min_w:
        return "too_short"
    if len(words) > max_w:
        return "too_long"
    if any(c in BAD_CHARS for c in t) or CODE_RE.search(t):
        return "code_or_markup"
    if sum(ch.isdigit() for ch in t) > 12:
        return "too_many_digits"
    non_ascii = sum(ord(ch) > 127 for ch in t)
    if non_ascii > 2:
        return "non_english"
    low = t.lower()
    if any(b in low for b in BLOCKLIST):
        return "unsafe"
    if low.count(",") > 6 or t.count(";") > 1:
        return "list_like"
    return None


def as_question(t: str) -> str:
    """Search-style queries ('who sang hey jude') -> spoken question ('Who sang hey jude?')."""
    t = t.strip()
    if not t:
        return t
    t = t[0].upper() + t[1:]
    if t[-1] not in "?.!":
        t += "?" if t.lower().startswith(QUESTION_START) else "."
    return t


class SourceStats:
    def __init__(self, name, license_):
        self.name, self.license = name, license_
        self.read = 0
        self.kept = 0
        self.drops = Counter()

    def dump(self, paths):
        atomic_write_json(paths.logs / "sources" / f"text_{self.name.replace('/', '__')}.json",
                          {"source": self.name, "license": self.license, "rows_read": self.read,
                           "rows_kept": self.kept, "drops": dict(self.drops), "time": now()})


def gen_crypto_questions(add_q, add_chain, st_q, st_c, seed):
    rr = random.Random(f"crypto|{seed}")
    n = 0
    for term in CRYPTO_WHAT_IS_TERMS:
        for tmpl in WHAT_IS_TEMPLATES:
            add_q(tmpl.format(x=term), "crypto", st_q.name, st_q.license, st_q, f"wis|{term}|{tmpl}")
            n += 1
    for term in CRYPTO_HOW_WORKS_TERMS:
        for tmpl in HOW_WORKS_TEMPLATES:
            add_q(tmpl.format(x=term), "crypto", st_q.name, st_q.license, st_q, f"how|{term}|{tmpl}")
            n += 1
    for term in CRYPTO_INVENT_TERMS:
        for tmpl in WHO_INVENTED_TEMPLATES + WHEN_INVENTED_TEMPLATES:
            add_q(tmpl.format(x=term), "crypto", st_q.name, st_q.license, st_q, f"inv|{term}|{tmpl}")
            n += 1
    for a, b in CRYPTO_DIFF_PAIRS:
        for tmpl in DIFF_TEMPLATES:
            add_q(tmpl.format(a=a, b=b), "crypto", st_q.name, st_q.license, st_q, f"diff|{a}|{b}|{tmpl}")
            n += 1
    for cluster in CRYPTO_CLUSTERS:
        for q in cluster:
            add_q(q, "crypto", st_q.name, st_q.license, st_q, f"cl|{q}")
            n += 1
    for chain in CRYPTO_CHAINS:
        add_chain(chain, "crypto", st_c.name, st_c.license, st_c, f"chain|{chain[0]}")
    # a few reordered / partial variants of each chain for extra multi-turn diversity
    for chain in CRYPTO_CHAINS:
        if len(chain) >= 3 and rr.random() < 0.6:
            add_chain(chain[:2], "crypto", st_c.name, st_c.license, st_c, f"chain2|{chain[0]}")
    return n


def gen_general_chat_questions(add_q, st_q):
    n = 0
    for cluster in GENERAL_CHAT_CLUSTERS:
        for q in cluster:
            add_q(q, "general", st_q.name, st_q.license, st_q, f"gc|{q}")
            n += 1
    return n


def gen_identity_questions(add_q, st_q):
    n = 0
    for cluster in IDENTITY_CLUSTERS:
        for q in cluster:
            add_q(q, "identity", st_q.name, st_q.license, st_q, f"id|{q}")
            n += 1
    return n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("text", paths.logs / "workers" / "text.log")
    stage_log(paths, "text", "running")
    from datasets import load_dataset

    tcfg = cfg["text_sources"]
    seed = cfg["seed"]
    val, test = cfg["splits"]["val"], cfg["splits"]["test"]
    min_w, max_w = tcfg["min_words"], tcfg["max_words"]
    questions, chains = [], []
    seen = set()

    def add_q(text, domain, source, license_, st: SourceStats, key):
        st.read += 1
        if URL_RE.search(text or ""):
            st.drops["contains_url"] += 1
            return
        text = clean_text(text)
        reason = spoken_reason(text, min_w, max_w)
        if reason:
            st.drops[reason] += 1
            return
        text = as_question(text)
        h = text_hash(text)
        if h in seen:
            st.drops["duplicate"] += 1
            return
        seen.add(h)
        qid = f"q_{h}"
        questions.append({"qid": qid, "text": text, "domain": domain, "source": source, "license": license_,
                          "split": split_of(qid, seed, val, test)})
        st.kept += 1

    def add_chain(turns, domain, source, license_, st: SourceStats, key):
        st.read += 1
        good = []
        for t in turns:
            t = clean_text(t)
            if spoken_reason(t, 1, max_w) is not None:
                break  # keep the chain coherent: stop at the first unusable turn
            good.append(as_question(t) if len(t.split()) >= 2 else t)
        if len(good) < 2:
            st.drops["chain_too_short_after_filter"] += 1
            return
        h = text_hash(" || ".join(good))
        if h in seen:
            st.drops["duplicate"] += 1
            return
        seen.add(h)
        cid = f"c_{h}"
        chains.append({"chain_id": cid, "turns": good[:8], "domain": domain, "source": source, "license": license_,
                       "split": split_of(cid, seed, val, test)})
        st.kept += 1

    def cap(n_key):
        return tcfg["max_rows"].get(n_key, 10 ** 9)

    def rows(name, split, limit, config=None):
        """Yields (i, row) for the first `limit` rows. streaming=True reads only those rows (smoke test);
        otherwise the dataset is downloaded once into the HF cache (production)."""
        args = (name, config) if config else (name,)
        ds = load_dataset(*args, split=split, streaming=bool(tcfg.get("streaming")))
        for i, r in enumerate(ds):
            if i >= limit:
                break
            yield i, r

    en = tcfg["enabled"]

    # ---------------- finance
    if en.get("finance_alpaca"):
        st = SourceStats("gbharti/finance-alpaca", "MIT")
        for i, r in rows("gbharti/finance-alpaca", "train", cap("finance_alpaca")):
            if (r.get("input") or "").strip():
                st.read += 1
                st.drops["needs_context"] += 1
                continue
            add_q(r["instruction"], "finance", st.name, st.license, st, i)
        st.dump(paths); log.info("finance-alpaca kept %d", st.kept)
    if en.get("fiqa"):
        st = SourceStats("FinGPT/fingpt-fiqa_qa", "unspecified on HF card (FiQA-derived) -- verify before commercial use")
        for i, r in rows("FinGPT/fingpt-fiqa_qa", "train", cap("fiqa")):
            add_q(r["input"], "finance", st.name, st.license, st, i)
        st.dump(paths); log.info("fiqa kept %d", st.kept)
    if en.get("financial_qa_10k"):
        st = SourceStats("virattt/financial-qa-10K", "unspecified on HF card -- verify before commercial use")
        for i, r in rows("virattt/financial-qa-10K", "train", cap("financial_qa_10k")):
            add_q(r["question"], "finance", st.name, st.license, st, i)
        st.dump(paths); log.info("financial-qa-10K kept %d", st.kept)
    for key, name, domain in [("bitext_banking", "bitext/Bitext-retail-banking-llm-chatbot-training-dataset", "finance_cs"),
                              ("bitext_insurance", "bitext/Bitext-insurance-llm-chatbot-training-dataset", "finance_cs"),
                              ("bitext_customer_support", "bitext/Bitext-customer-support-llm-chatbot-training-dataset", "customer_service")]:
        if not en.get(key):
            continue
        st = SourceStats(name, "CDLA-Sharing-1.0")
        for i, r in rows(name, "train", cap(key)):
            add_q(fill_placeholders(r["instruction"], f"{key}{i}"), domain, st.name, st.license, st, i)
        st.dump(paths); log.info("%s kept %d", name, st.kept)

    # ---------------- general knowledge / assistant
    if en.get("dolly"):
        st = SourceStats("databricks/databricks-dolly-15k", "CC-BY-SA-3.0")
        for i, r in rows("databricks/databricks-dolly-15k", "train", cap("dolly")):
            if r["category"] not in ("open_qa", "general_qa", "brainstorming", "creative_writing"):
                st.read += 1
                st.drops["category"] += 1
                continue
            if (r.get("context") or "").strip():
                st.read += 1
                st.drops["needs_context"] += 1
                continue
            add_q(r["instruction"], "general", st.name, st.license, st, i)
        st.dump(paths); log.info("dolly kept %d", st.kept)
    if en.get("natural_questions"):
        st = SourceStats("sentence-transformers/natural-questions", "CC-BY-SA-3.0 (Natural Questions) -- verify")
        for i, r in rows("sentence-transformers/natural-questions", "train", cap("natural_questions"), config="pair"):
            add_q(r["query"], "general", st.name, st.license, st, i)
        st.dump(paths); log.info("natural-questions kept %d", st.kept)
    if en.get("medquad"):
        st = SourceStats("keivalya/MedQuad-MedicalQnADataset", "unspecified on HF card (MedQuAD) -- verify")
        for i, r in rows("keivalya/MedQuad-MedicalQnADataset", "train", cap("medquad")):
            q = re.sub(r"\(are\)\s*", "", r["Question"]).replace(" ?", "?")
            add_q(q, "health", st.name, st.license, st, i)
        st.dump(paths); log.info("medquad kept %d", st.kept)
    if en.get("oasst1"):
        st = SourceStats("OpenAssistant/oasst1", "Apache-2.0")
        msgs = []
        for sp in ("train", "validation"):
            msgs.extend(r for _, r in rows("OpenAssistant/oasst1", sp, cap("oasst1")))
        children = defaultdict(list)
        for r in msgs:
            if r["parent_id"]:
                children[r["parent_id"]].append(r)
        # Follow the highest-ranked assistant reply at each step to recover the user-turn chain.
        for r in msgs:
            if r["parent_id"] is not None or r["role"] != "prompter" or r["lang"] != "en":
                continue
            turns = [r["text"]]
            node = r
            while True:
                asst = sorted([c for c in children[node["message_id"]] if c["role"] == "assistant"],
                              key=lambda c: (c["rank"] if c["rank"] is not None else 99))
                if not asst:
                    break
                nxt = [c for c in children[asst[0]["message_id"]] if c["role"] == "prompter" and c["lang"] == "en"]
                if not nxt:
                    break
                node = nxt[0]
                turns.append(node["text"])
            if len(turns) >= 2:
                add_chain(turns, "general", st.name, st.license, st, r["message_id"])
            else:
                add_q(turns[0], "general", st.name, st.license, st, r["message_id"])
        st.dump(paths); log.info("oasst1 kept %d", st.kept)
    if en.get("ultrachat"):
        st = SourceStats("HuggingFaceH4/ultrachat_200k", "MIT")
        for i, r in rows("HuggingFaceH4/ultrachat_200k", "train_sft", cap("ultrachat")):
            turns = [m["content"] for m in r["messages"] if m["role"] == "user"]
            if len(turns) >= 2:
                add_chain(turns, "general", st.name, st.license, st, i)
            elif turns:
                add_q(turns[0], "general", st.name, st.license, st, i)
        st.dump(paths); log.info("ultrachat kept %d", st.kept)

    # ---------------- casual social conversation
    if en.get("soda"):
        st = SourceStats("allenai/soda", "CC-BY-4.0")
        for i, r in rows("allenai/soda", "train", cap("soda")):
            dlg, spk = r["dialogue"], r["speakers"]
            if not dlg or len(dlg) < 4:
                st.read += 1
                st.drops["dialogue_too_short"] += 1
                continue
            # The user is the speaker of the first line; take every one of their lines.
            first = spk[0]
            turns = [d for d, s in zip(dlg, spk) if s == first]
            add_chain(turns, "casual", st.name, st.license, st, i)
        st.dump(paths); log.info("soda kept %d", st.kept)

    # ---------------- cryptocurrency / general conversation / identity (synthetic, procedural, no download)
    if en.get("crypto_synth", True):
        st = SourceStats("synthetic/crypto_qa", "synthetic (generated for this project)")
        st_c = SourceStats("synthetic/crypto_chains", "synthetic (generated for this project)")
        n = gen_crypto_questions(add_q, add_chain, st, st_c, seed)
        st.dump(paths); st_c.dump(paths); log.info("crypto synthetic kept %d/%d (chains %d)", st.kept, n, st_c.kept)
    if en.get("general_synth", True):
        st = SourceStats("synthetic/general_chat", "synthetic (generated for this project)")
        n = gen_general_chat_questions(add_q, st)
        st.dump(paths); log.info("general chat synthetic kept %d/%d", st.kept, n)
    if en.get("identity_synth", True):
        st = SourceStats("synthetic/identity_qa", "synthetic (generated for this project)")
        n = gen_identity_questions(add_q, st)
        st.dump(paths); log.info("identity synthetic kept %d/%d", st.kept, n)

    rnd = random.Random(seed)
    rnd.shuffle(questions)
    rnd.shuffle(chains)
    write_jsonl(paths.meta / "questions.jsonl", questions)
    write_jsonl(paths.meta / "chains.jsonl", chains)
    summary = {
        "questions": len(questions), "chains": len(chains),
        "questions_by_domain": dict(Counter(q["domain"] for q in questions)),
        "questions_by_split": dict(Counter(q["split"] for q in questions)),
        "chains_by_domain": dict(Counter(c["domain"] for c in chains)),
        "chains_by_split": dict(Counter(c["split"] for c in chains)),
        "questions_by_source": dict(Counter(q["source"] for q in questions)),
    }
    atomic_write_json(paths.logs / "sources" / "text_summary.json", summary)
    log.info("text summary: %s", json.dumps(summary))
    stage_log(paths, "text", "done", **{k: v for k, v in summary.items() if isinstance(v, int)})


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/personas.py
"""Stage `personas`: generate persona (text) prompts in PersonaPlex's own prompt style.

Formats are taken verbatim from the repo:
  README.md          "You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way."
                     "You enjoy having a good conversation."
                     "You work for CitySan Services which is a waste management company and your name is Ayelen Lucero.
                      Information: ..."
  prompt_service.txt "You work for SwiftPlex Appliances which is a appliance repair company and your name is ..."
  server.py docstring "You enjoy having a good conversation. Have a deep conversation about technology. Your name is Jane."

Each persona is tokenized exactly like server.py/offline.py do (tokenizer.encode(wrap_with_system_tags(text)));
the token count T fixes its prefix length (136 + T recorded frames). Personas longer than
cfg.max_persona_tokens are dropped. An empty persona is NOT generated: server.py sets text_prompt_tokens=None
for an empty prompt and LMGen._step_text_prompt_core then fails iterating None -- it can't occur at inference.

Customer-service/finance personas carry `faq`: questions answerable from their own `Information:` facts.
"""
import argparse
import json
import random
from collections import Counter

from .common import P, load_cfg, get_logger, stage_log, atomic_write_json, write_jsonl, split_of

FIRST = ["Ayelen", "Farhod", "Maria", "James", "Aisha", "Tomás", "Priya", "Kenji", "Olivia", "Mateo", "Fatima",
         "Liam", "Chloe", "Arjun", "Sofia", "Daniel", "Amara", "Noah", "Mei", "Lucas", "Zara", "Ethan", "Leila",
         "Omar", "Hannah", "Diego", "Nia", "Samuel", "Yuki", "Grace", "Ibrahim", "Elena", "Kwame", "Ava", "Ravi",
         "Isabella", "Chen", "Emily", "Tunde", "Lucia", "Mohammed", "Sarah", "Hiroshi", "Anya", "Carlos", "Jade",
         "Sipho", "Rosa", "Aaron", "Meera", "Jonas", "Adaeze", "Pedro", "Freya", "Hassan", "Lily", "Nikhil", "Ines",
         "Marcus", "Ayumi", "Kofi", "Camila", "Viktor", "Leah", "Rahul", "Beatriz", "Owen", "Salma", "Andrei", "Ruth"]
LAST = ["Lucero", "Toshmatov", "Lopez", "Chen", "Khan", "Becker", "Nair", "Tanaka", "Okafor", "Silva", "Haddad",
        "Murphy", "Dubois", "Sharma", "Rossi", "Kim", "Mensah", "Johansson", "Wong", "Garcia", "Ali", "Novak",
        "Fischer", "Adeyemi", "Moreau", "Patel", "Ivanova", "Santos", "Nakamura", "O'Brien", "Kowalski", "Rahman",
        "Costa", "Schmidt", "Mwangi", "Herrera", "Yilmaz", "Andersen", "Park", "Castillo", "Bauer", "Osei",
        "Gonzalez", "Levi", "Reddy", "Walsh", "Moyo", "Ferreira", "Lindqvist", "Abdi"]
CO_A = ["Swift", "City", "Blue", "North", "Summit", "Harbor", "Bright", "Prime", "Evergreen", "Metro", "Pioneer",
        "Crest", "Silver", "Oak", "River", "Nova", "Keystone", "Atlas", "Clear", "Beacon", "Maple", "Horizon"]
CO_B = ["Plex", "San", "Line", "Point", "Wave", "Field", "Stone", "Bridge", "Path", "Way", "Gate", "Core", "Light"]
CO_C = ["Services", "Solutions", "Group", "Company", "Partners", "Co.", "Inc."]

MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October",
          "November", "December"]


def money(r, lo, hi, cents=True):
    v = r.uniform(lo, hi)
    return f"${v:,.2f}" if cents else f"${int(v):,}"


def ordinal(n: int) -> str:
    return f"{n}{'th' if 11 <= n % 100 <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


def date(r):
    return f"{r.choice(MONTHS)} {ordinal(r.randint(1, 28))}"


def company(r):
    return f"{r.choice(CO_A)}{r.choice(CO_B)} {r.choice(CO_C)}"


def person(r):
    return f"{r.choice(FIRST)} {r.choice(LAST)}"


# ----------------------------------------------------------------------------- customer-service style domains
# Each domain: industry phrase, fact generators -> (fact sentence, [questions that fact answers])
def dom_waste(r, cust):
    freq = r.choice(["every week", "every other week", "twice a month"])
    d = date(r)
    fee = money(r, 5, 15, cents=False)
    return "waste management", [
        (f"Verify customer name {cust}.", [f"Hi, this is {cust.split()[0]}, can you check my account?"]),
        (f"Current schedule: {freq}.", ["How often do you pick up my trash?", "What's my current pickup schedule?"]),
        (f"Upcoming pickup: {d}.", ["When is my next pickup?", "Is there a pickup coming up soon?"]),
        (f"Compost bin service available for {fee}/month add-on.", ["Do you offer compost bins?", "How much is the compost service?"]),
    ]


def dom_appliance(r, cust):
    rate = money(r, 45, 95, cents=False)
    delay = r.randint(2, 7)
    appl = r.choice(["dishwasher", "washing machine", "dryer", "refrigerator", "oven"])
    return "appliance repair", [
        (f"The {appl} model is out of stock for replacement parts; we can use an alternative part with a {delay}-day delay.",
         [f"When can you fix my {appl}?", "Do you have the part in stock?", "Is there any faster option?"]),
        (f"Labor cost remains {rate} per hour.", ["How much do you charge for labor?", "What will the repair cost me?"]),
    ]


def dom_bank(r, cust):
    bal = money(r, 120, 9800)
    last = date(r)
    od = money(r, 25, 39, cents=False)
    apr = f"{r.uniform(3.5, 6.9):.2f} percent"
    return "retail banking", [
        (f"Verify customer name {cust}.", [f"Hi, my name is {cust}.", "Can you look up my account?"]),
        (f"Checking balance: {bal}.", ["What's my checking balance?", "How much money do I have in checking?"]),
        (f"Last deposit posted {last}.", ["When did my last deposit go through?", "Did my paycheck arrive?"]),
        (f"Overdraft fee: {od}.", ["What's the overdraft fee?", "Will I be charged if I overdraw?"]),
        (f"Savings account interest rate: {apr}.", ["What interest does your savings account pay?"]),
    ]


def dom_insurance(r, cust):
    ded = money(r, 250, 2000, cents=False)
    prem = money(r, 60, 240)
    renew = date(r)
    kind = r.choice(["auto", "home", "renters", "travel"])
    return "insurance", [
        (f"Customer has a {kind} policy.", [f"I have a question about my {kind} insurance.", "What does my policy cover?"]),
        (f"Deductible: {ded}.", ["What's my deductible?", "How much would I pay out of pocket for a claim?"]),
        (f"Monthly premium: {prem}.", ["How much is my monthly premium?", "Can I lower my premium?"]),
        (f"Policy renews on {renew}.", ["When does my policy renew?"]),
    ]


def dom_card(r, cust):
    limit = money(r, 1000, 15000, cents=False)
    due = date(r)
    mn = money(r, 25, 140)
    return "credit card", [
        (f"Credit limit: {limit}.", ["What's my credit limit?", "Can I get a higher limit?"]),
        (f"Payment due {due}; minimum payment {mn}.", ["When is my payment due?", "What's the minimum I have to pay?"]),
        ("Card can be frozen instantly in the app.", ["I think I lost my card, what should I do?", "Can you freeze my card?"]),
    ]


def dom_mortgage(r, cust):
    rate = f"{r.uniform(5.2, 7.4):.2f} percent"
    term = r.choice(["15-year", "30-year"])
    closing = money(r, 2500, 9000, cents=False)
    return "mortgage lending", [
        (f"Current {term} fixed rate: {rate}.", [f"What's your {term} rate right now?", "What mortgage rates do you have?"]),
        (f"Estimated closing costs: {closing}.", ["How much are closing costs?", "What fees should I expect?"]),
        ("Pre-approval takes about two business days.", ["How long does pre-approval take?"]),
    ]


def dom_tax(r, cust):
    fee = money(r, 80, 350, cents=False)
    dl = r.choice(["April 15th", "October 15th"])
    return "tax preparation", [
        (f"Standard filing package: {fee}.", ["How much do you charge to file my taxes?"]),
        (f"Filing deadline for this client: {dl}.", ["When is my filing deadline?", "Can I get an extension?"]),
        ("Client must bring W-2 forms and last year's return.", ["What documents do I need to bring?"]),
    ]


def dom_telecom(r, cust):
    plan = money(r, 35, 95, cents=False)
    data = r.choice(["15 gigabytes", "unlimited data", "50 gigabytes"])
    outage = r.choice(["no outages reported in the area", "a known outage in the area until 6 PM"])
    return "internet and mobile", [
        (f"Current plan: {data} for {plan} per month.", ["What plan am I on?", "How much data do I have?"]),
        (f"Network status: {outage}.", ["My internet is down, is there an outage?", "Why is my connection so slow?"]),
        ("Router restart usually fixes connection drops.", ["What can I do to fix my Wi-Fi?"]),
    ]


def dom_airline(r, cust):
    flight = f"{r.choice(['BA', 'LH', 'UA', 'EK', 'AC'])} {r.randint(100, 999)}"
    delay = r.randint(20, 150)
    bag = money(r, 30, 75, cents=False)
    return "airline", [
        (f"Customer booked on flight {flight}, delayed {delay} minutes.", ["Is my flight on time?", "How late is my flight?"]),
        (f"Checked bag fee: {bag}.", ["How much is a checked bag?"]),
        ("Rebooking to a later flight is free today.", ["Can I change to a later flight?"]),
    ]


def dom_clinic(r, cust):
    slot = f"{r.choice(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'])} at {r.randint(8, 16)}:{r.choice(['00', '30'])}"
    copay = money(r, 15, 60, cents=False)
    return "medical clinic scheduling", [
        (f"Next available appointment: {slot}.", ["When can I see the doctor?", "Do you have anything this week?"]),
        (f"Copay for a standard visit: {copay}.", ["How much is the copay?"]),
        ("Patients should arrive fifteen minutes early.", ["What time should I get there?"]),
    ]


def dom_restaurant(r, cust):
    wait = r.randint(10, 60)
    special = r.choice(["mushroom risotto", "grilled salmon", "lamb curry", "vegan lasagna"])
    return "restaurant", [
        (f"Current wait for a table: {wait} minutes.", ["How long is the wait?", "Can I book a table for tonight?"]),
        (f"Today's special: {special}.", ["What's the special today?", "Do you have vegetarian options?"]),
    ]


def dom_delivery(r, cust):
    eta = date(r)
    return "package delivery", [
        (f"Package is in transit, estimated delivery {eta}.", ["Where is my package?", "When will my order arrive?"]),
        ("Redelivery can be scheduled online or by phone.", ["I missed my delivery, what now?"]),
    ]


def dom_hotel(r, cust):
    rate = money(r, 90, 320, cents=False)
    return "hotel", [
        (f"Standard room rate: {rate} per night.", ["How much is a room per night?", "Do you have rooms available?"]),
        ("Check-in is at 3 PM, check-out at 11 AM.", ["What time is check-in?", "Can I get a late checkout?"]),
        ("Breakfast is included.", ["Is breakfast included?"]),
    ]


def dom_brokerage(r, cust):
    fee = r.choice(["zero commission on stock trades", "a $4.95 commission per trade"])
    return "online brokerage", [
        (f"Pricing: {fee}.", ["How much do trades cost?", "Are there any fees?"]),
        ("IRA and Roth IRA accounts available with no minimum.", ["Can I open a Roth IRA?", "What's the difference between an IRA and a Roth?"]),
        ("Transfers from other brokers take five to seven business days.", ["How long does an account transfer take?"]),
    ]


CS_DOMAINS = {"waste": dom_waste, "appliance": dom_appliance, "telecom": dom_telecom, "airline": dom_airline,
              "clinic": dom_clinic, "restaurant": dom_restaurant, "delivery": dom_delivery, "hotel": dom_hotel}
FIN_DOMAINS = {"bank": dom_bank, "insurance": dom_insurance, "card": dom_card, "mortgage": dom_mortgage,
               "tax": dom_tax, "brokerage": dom_brokerage}

ASSISTANT_ROLES = [
    ("a wise and friendly teacher", "Answer questions or provide advice in a clear and engaging way.", "general"),
    ("a patient science tutor", "Explain ideas simply and check that the listener understands.", "general"),
    ("a knowledgeable history guide", "Answer questions with interesting stories and clear facts.", "general"),
    ("a helpful home cooking expert", "Give practical cooking tips and simple step by step advice.", "general"),
    ("a friendly travel advisor", "Help people plan trips and answer travel questions.", "general"),
    ("a calm tech support specialist", "Help troubleshoot computer and phone problems step by step.", "general"),
    ("an encouraging language tutor", "Help the learner practice and explain grammar simply.", "general"),
    ("a careful health information assistant", "Give general health information and suggest seeing a doctor when needed.", "health"),
    ("a knowledgeable personal finance coach", "Explain budgeting, saving, and debt in simple terms.", "finance"),
    ("a clear-thinking financial educator", "Explain investing, markets, and financial terms without giving personal investment advice.", "finance"),
    ("a practical budgeting assistant", "Help people build a budget and stick to it.", "finance"),
    ("a tax basics explainer", "Explain how taxes work in plain language.", "finance"),
    ("a thoughtful career coach", "Give practical advice about jobs, interviews, and careers.", "general"),
    ("a friendly fitness coach", "Give safe, simple exercise and habit advice.", "general"),
    ("an enthusiastic math tutor", "Walk through problems step by step and keep it encouraging.", "general"),
    ("a well-read librarian", "Recommend books and answer questions about literature.", "general"),
    ("a gardening expert", "Answer questions about plants and gardening with practical tips.", "general"),
    ("a curious science communicator", "Make complicated topics easy and fun to understand.", "general"),
]
CASUAL_TOPICS = ["technology", "music", "movies", "travel", "food", "sports", "books", "space", "history", "art",
                 "pets", "the weekend", "work life", "cooking", "video games", "nature", "fashion", "philosophy",
                 "the future", "language learning", "photography", "football", "coffee", "hiking", "family",
                 "money and saving", "the stock market", "starting a business", "personal finance", "crypto"]
CASUAL_STYLES = ["deep", "light-hearted", "friendly", "casual", "curious", "fun", "thoughtful", "relaxed"]
ROLEPLAY = [
    ("an enthusiastic tour guide describing a city's history to a visitor", "general"),
    ("a no-nonsense project manager giving a brief status update in a stand-up meeting", "general"),
    ("a friendly barista taking a coffee order and making small talk", "casual"),
    ("a museum guide explaining a famous painting", "general"),
    ("a real estate agent showing an apartment to a first-time renter", "finance"),
    ("a bank teller helping a customer open a new account", "finance"),
    ("a car salesperson explaining financing options", "finance"),
    ("a hotel concierge recommending places to eat nearby", "general"),
    ("a radio host chatting with a caller", "casual"),
    ("a job interviewer asking about someone's experience", "general"),
    ("a pharmacist explaining how to take a new prescription", "health"),
    ("a startup founder pitching an idea to an investor", "finance"),
    ("a park ranger answering questions on a hiking trail", "general"),
    ("a friendly neighbor catching up over the fence", "casual"),
]
README_VERBATIM = [
    ("assistant", "You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way.", "general", []),
    ("casual", "You enjoy having a good conversation.", "casual", []),
    ("customer_service",
     "You work for CitySan Services which is a waste management company and your name is Ayelen Lucero. Information: "
     "Verify customer name Omar Torres. Current schedule: every other week. Upcoming pickup: April 12th. Compost bin "
     "service available for $8/month add-on.", "customer_service",
     ["When is my next pickup?", "How often do you pick up?", "Do you have compost bins?", "Hi, this is Omar Torres."]),
    ("customer_service",
     "You work for SwiftPlex Appliances which is a appliance repair company and your name is Farhod Toshmatov. "
     "Information: The dishwasher model is out of stock for replacement parts; we can use an alternative part with a "
     "3-day delay. Labor cost remains $60 per hour.", "customer_service",
     ["When can you fix my dishwasher?", "How much is labor?", "Is there a faster option?"]),
]


# ----------------------------------------------------------------------------- Robert / RB Labs identity persona
# Fixed factual content (never randomized) so the teacher learns one consistent identity. Order/subset of the
# optional facts is shuffled per persona row for token-length diversity; name/creation/location are always present.
IDENTITY_CORE = [
    ("Your name is Robert, and people call you Robert the Robot.",
     ["What is your name?", "Who are you?", "What should I call you?", "Do you have a name?", "What's your name?"]),
    ("You were created in June 2025 in Geneva, Switzerland, by a company called RB Labs.",
     ["Who created you?", "Who made you?", "Who built you?", "When were you created?", "Where were you created?",
      "When did you come into existence?", "Where do you come from?"]),
    ("You are physically located in San Francisco.",
     ["Where are you located?", "Where are you right now?", "Where are you based?", "Where do you live?"]),
]
IDENTITY_NAME = IDENTITY_CORE[0]
IDENTITY_FOUNDERS_OVERVIEW = (
    "RB Labs was founded by three people: Robin Krambroeckers, Lin Dayen-Hsu, and Metodi Dimitrov.",
    ["Who founded RB Labs?", "Who are the founders of RB Labs?", "Who runs RB Labs?",
     "Tell me about RB Labs' founders.", "What is RB Labs?", "Tell me about RB Labs."])
# Kept short (~35-45 words each) so a lead + name fact + one bio still fits cfg.max_persona_tokens -- unlike the
# generic customer-service `Information:` facts, these describe real people and don't compress well.
IDENTITY_FOUNDER_BIOS = [
    ("Robin Krambroeckers is a German technologist and co-founder of RB Labs, responsible for AI and blockchain. "
     "He leads technical operations, building predictive systems and applying machine learning to financial "
     "markets, and gained early recognition presenting his work at events including Davos.",
     ["Tell me about Robin Krambroeckers.", "Who is Robin?", "What does Robin do at RB Labs?",
      "Tell me about Robin."]),
    ("Lin Dayen-Hsu is a co-founder of RB Labs with deep experience in business development, sales, and "
     "marketing. He is also an entrepreneur in fashion and events, including Geneva Tailors and Krazy Events, "
     "connecting technology with real market needs.",
     ["Tell me about Lin Dayen-Hsu.", "Who is Lin?", "What does Lin do at RB Labs?", "Tell me about Lin."]),
    ("Metodi Dimitrov is a fintech entrepreneur and AI specialist, and co-founder of RB Labs. He focuses on "
     "operational excellence and product development that makes finance more accessible through AI and "
     "blockchain, with an ethical, human-first approach.",
     ["Tell me about Metodi Dimitrov.", "Who is Metodi?", "What does Metodi do at RB Labs?",
      "Tell me about Metodi."]),
]
IDENTITY_PURPOSE = (
    "You can talk about cryptocurrency, answer everyday questions, and have natural voice conversations.",
    ["What can you help me with?", "What can you do?", "What is your purpose?", "What topics do you know about?",
     "Can you have a conversation with me?"])
IDENTITY_LEADS = [
    "You are Robert the Robot, a helpful voice assistant created by RB Labs.",
    "Your name is Robert. You are Robert the Robot, an AI assistant built by RB Labs.",
    "You are Robert the Robot. You work for RB Labs.",
    "You are Robert, also known as Robert the Robot, made by RB Labs.",
    "Your name is Robert the Robot. You were made by RB Labs and you enjoy having a good conversation.",
]


def make_identity(r):
    """One bounded fact "shape" per persona (never all facts stacked together) so every variant comfortably
    fits cfg.max_persona_tokens -- a founder bio alone is already close to the customer-service persona length."""
    lead = r.choice(IDENTITY_LEADS)
    shape = r.choice(["intro", "founders", "bio", "purpose"])
    if shape == "intro":
        facts = list(IDENTITY_CORE)
    elif shape == "founders":
        facts = [IDENTITY_NAME, IDENTITY_FOUNDERS_OVERVIEW]
    elif shape == "bio":
        facts = [IDENTITY_NAME, r.choice(IDENTITY_FOUNDER_BIOS)]
    else:
        facts = list(IDENTITY_CORE) + [IDENTITY_PURPOSE]
    r.shuffle(facts)
    text = f"{lead} Information: " + " ".join(f for f, _ in facts)
    faq = [q for _, qs in facts for q in qs]
    return text, "identity", faq, f"robert_{shape}"


# ----------------------------------------------------------------------------- cryptocurrency-focused assistant personas
CRYPTO_ROLES = [
    ("a knowledgeable cryptocurrency educator",
     "Explain blockchain, Bitcoin, and other cryptocurrencies clearly and patiently, in plain language.", "crypto"),
    ("a patient crypto and blockchain tutor",
     "Answer questions about wallets, exchanges, mining, and DeFi in simple, friendly language.", "crypto"),
    ("a helpful Web3 assistant",
     "Explain NFTs, DeFi, DAOs, and decentralized apps in plain language.", "crypto"),
    ("a careful crypto security advisor",
     "Help people understand how to protect wallets, private keys, and seed phrases, and how to spot scams.", "crypto"),
    ("a friendly crypto exchange support assistant",
     "Explain how buying, selling, sending, and receiving cryptocurrency works, step by step.", "crypto"),
    ("a clear-thinking cryptocurrency market educator",
     "Explain crypto prices, market cap, volatility, and trading in plain language without giving financial advice.",
     "crypto"),
    ("an experienced blockchain developer",
     "Explain how blockchains, smart contracts, and consensus mechanisms work under the hood, in accessible terms.",
     "crypto"),
    ("a beginner-friendly cryptocurrency guide",
     "Answer basic crypto questions simply, as if talking to someone who has never used crypto before.", "crypto"),
]


def make_cs(r, domains, category):
    key = r.choice(list(domains))
    agent, cust = person(r), person(r)
    industry, facts = domains[key](r, cust)
    r.shuffle(facts)
    facts = facts[: r.randint(2, len(facts))]
    co = company(r)
    art = "an" if industry[0] in "aeiou" else "a"
    lead = r.choice([
        f"You work for {co} which is {art} {industry} company and your name is {agent}.",
        f"You work for {co} which is {art} {industry} company and your name is {agent}. Be polite and concise.",
        f"You are {agent}, a customer service agent at {co}, {art} {industry} company.",
    ])
    text = f"{lead} Information: " + " ".join(f for f, _ in facts)
    faq = [q for _, qs in facts for q in qs]
    dom = "finance_cs" if category == "finance" else "customer_service"
    return text, dom, faq, key


def build_personas(cfg, tok):
    seed = cfg["seed"]
    r = random.Random(f"personas|{seed}")
    pc = cfg["personas"]
    out, seen = [], set()

    def add(category, text, domain, faq, sub):
        text = " ".join(text.split())
        if text in seen:
            return
        from .common import text_hash
        pid = f"p_{text_hash(text)}"
        toks = tok.encode(f"<system> {text.strip()} <system>")  # == server.wrap_with_system_tags
        if len(toks) > cfg["max_persona_tokens"]:
            return
        seen.add(text)
        out.append({"persona_id": pid, "category": category, "sub": sub, "text": text, "domain": domain,
                    "faq": faq, "tokens": toks, "n_tokens": len(toks),
                    "split": split_of(pid, seed, cfg["splits"]["val"], cfg["splits"]["test"])})

    for cat, text, dom, faq in README_VERBATIM:
        add(cat, text, dom, faq, "readme")
    for _ in range(pc["n_customer_service"] * 3):
        if sum(p["category"] == "customer_service" for p in out) >= pc["n_customer_service"]:
            break
        t, d, faq, key = make_cs(r, CS_DOMAINS, "customer_service")
        add("customer_service", t, d, faq, key)
    for _ in range(pc["n_finance"] * 3):
        if sum(p["category"] == "finance" for p in out) >= pc["n_finance"]:
            break
        if r.random() < 0.65:
            t, d, faq, key = make_cs(r, FIN_DOMAINS, "finance")
            add("finance", t, d, faq, key)
        else:
            role, instr, _ = r.choice([a for a in ASSISTANT_ROLES if a[2] == "finance"])
            name = r.choice(FIRST)
            t = r.choice([f"You are {role}. {instr}", f"You are {role} named {name}. {instr}",
                          f"You are {role}. {instr} Keep answers short and conversational."])
            add("finance", t, "finance", [], "finance_assistant")
    for _ in range(pc["n_assistant"] * 3):
        if sum(p["category"] == "assistant" for p in out) >= pc["n_assistant"]:
            break
        role, instr, dom = r.choice(ASSISTANT_ROLES)
        name = r.choice(FIRST)
        t = r.choice([f"You are {role}. {instr}", f"You are {role} named {name}. {instr}",
                      f"You are {role}. {instr} Keep your answers brief and natural, like a phone call.",
                      f"Your name is {name}. You are {role}. {instr}"])
        add("assistant", t, dom, [], "assistant")
    for _ in range(pc["n_casual"] * 3):
        if sum(p["category"] == "casual" for p in out) >= pc["n_casual"]:
            break
        topic, style, name = r.choice(CASUAL_TOPICS), r.choice(CASUAL_STYLES), r.choice(FIRST)
        t = r.choice([f"You enjoy having a good conversation. Have a {style} conversation about {topic}. Your name is {name}.",
                      f"You enjoy having a good conversation. Have a {style} conversation about {topic}.",
                      f"You enjoy having a good conversation. Your name is {name}."])
        add("casual", t, "casual", [], topic)
    for _ in range(pc["n_roleplay"] * 3):
        if sum(p["category"] == "roleplay" for p in out) >= pc["n_roleplay"]:
            break
        role, dom = r.choice(ROLEPLAY)
        name = r.choice(FIRST)
        t = r.choice([f"You are {role}.", f"You are {role}. Your name is {name}.",
                      f"You are {role}. Your name is {name}. Keep it natural and friendly."])
        add("roleplay", t, dom, [], "roleplay")
    for _ in range(pc.get("n_crypto", 0) * 3):
        if sum(p["category"] == "crypto" for p in out) >= pc.get("n_crypto", 0):
            break
        role, instr, dom = r.choice(CRYPTO_ROLES)
        name = r.choice(FIRST)
        t = r.choice([f"You are {role}. {instr}", f"You are {role} named {name}. {instr}",
                      f"You are {role}. {instr} Keep your answers clear and conversational, like a phone call.",
                      f"Your name is {name}. You are {role}. {instr}"])
        add("crypto", t, dom, [], "crypto_assistant")
    for _ in range(pc.get("n_identity", 0) * 3):
        if sum(p["category"] == "identity" for p in out) >= pc.get("n_identity", 0):
            break
        t, d, faq, key = make_identity(r)
        add("identity", t, d, faq, key)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("personas", paths.logs / "workers" / "personas.log")
    stage_log(paths, "personas", "running")
    import sentencepiece
    tok = sentencepiece.SentencePieceProcessor(cfg["teacher"]["tokenizer"])
    personas = build_personas(cfg, tok)
    write_jsonl(paths.meta / "personas.jsonl", personas)
    lens = [p["n_tokens"] for p in personas]
    summary = {"n": len(personas), "by_category": dict(Counter(p["category"] for p in personas)),
               "by_split": dict(Counter(p["split"] for p in personas)),
               "tokens_min": min(lens), "tokens_max": max(lens), "tokens_mean": sum(lens) / len(lens),
               "distinct_token_lengths": len(set(lens))}
    atomic_write_json(paths.logs / "sources" / "personas_summary.json", summary)
    log.info("personas: %s", json.dumps(summary))
    stage_log(paths, "personas", "done", n=len(personas))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/audio_sources.py
"""Stages `real_speech`, `real_speech_finalize`, `noise`: real human speech + background noise.

Real speech is read straight from the HF parquet files with pyarrow (audio bytes decoded by soundfile/ffmpeg) --
no `datasets` audio decoding, which would pull in torchcodec and tie us to a specific torch build.

Corpora (all verified on the HF Hub, see logs/sources/*):
  edinburghcstr/edacc          conversational English, 40+ L1 accents   -> user turns, real backchannels, agent voices
  facebook/voxpopuli en_accented accented English speech (CC0)          -> user turns, agent voices
  sanchit-gandhi/vctk          110 speakers, UK/IE/US/IN/... accents     -> agent voices
  mythicinfinity/libritts_r    clean multi-speaker English (CC-BY-4.0)   -> agent voices
  MY_RECORDINGS_DIR (optional) your own mic recordings (wav/flac + .txt) -> user turns (highest realism)
Noise: MUSAN (openslr 17, CC-BY-4.0) noise/music/speech(babble) + procedural noise.
"""
import argparse
import fnmatch
import json
import re
import subprocess
import tarfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

from .common import (P, SR, VOICE_SAMPLES, VOICE_LUFS, load_cfg, get_logger, stage_log, atomic_write_json,
                     write_jsonl, read_jsonl, append_jsonl, split_of, Claimer, Progress, worker_tag, log_error,
                     decode_audio_bytes, resample, rms_db, trim_silence, voiced_ratio, write_flac, write_wav,
                     loudness_normalize, rng_for, load_24k, stable_int)

TAG_RE = re.compile(r"<[^>]*>|\[[^\]]*\]")
BACKCHANNELS = {"yeah", "yes", "yep", "uh huh", "uh-huh", "mhm", "mm", "mm hmm", "mm-hmm", "mmhmm", "hmm", "okay",
                "ok", "right", "sure", "exactly", "true", "really", "wow", "oh", "oh okay", "oh really", "i see",
                "alright", "cool", "nice", "ah", "yeah yeah", "right right", "no way", "totally", "of course"}

CORPORA = {
    "edacc": dict(repo="edinburghcstr/edacc", patterns=["data/*.parquet"], spk="speaker", text="text",
                  accent="accent", gender="gender", uses=("user_utt", "backchannel", "agent_voice"),
                  license="CC-BY-SA-4.0 (EdAcc paper; HF card unspecified -- verify)"),
    "voxpopuli": dict(repo="facebook/voxpopuli", patterns=["en_accented/*.parquet"], spk="speaker_id",
                      text="normalized_text", accent="accent", gender="gender", uses=("user_utt", "agent_voice"),
                      license="CC0-1.0"),
    "vctk": dict(repo="sanchit-gandhi/vctk", patterns=["data/*.parquet"], spk="speaker_id", text="text",
                 accent="accent", gender="gender", uses=("agent_voice",), license="CC-BY-4.0 (VCTK)"),
    "libritts_r": dict(repo="mythicinfinity/libritts_r", patterns=["data/train.clean.100/*.parquet",
                                                                   "data/train.clean.360/*.parquet"],
                       spk="speaker_id", text="text_normalized", accent=None, gender=None, uses=("agent_voice",),
                       license="CC-BY-4.0"),
}


def list_units(cfg):
    from huggingface_hub import HfApi
    api = HfApi()
    units = []
    rs = cfg["real_speech"]
    for name, c in CORPORA.items():
        if not rs["enabled"].get(name):
            continue
        files = [f for f in api.list_repo_files(c["repo"], repo_type="dataset")
                 if any(fnmatch.fnmatch(f, p) for p in c["patterns"])]
        files = sorted(files)[: rs["max_files"].get(name, len(files))]
        units += [(name, f) for f in files]
    return units


def unit_id(name, f):
    return f"{name}__{f.replace('/', '__')}"


def clean_tx(t):
    return " ".join(TAG_RE.sub(" ", t or "").split())


def process_unit(cfg, paths, name, fname, log):
    import pyarrow.parquet as pq
    from huggingface_hub import hf_hub_download
    c = CORPORA[name]
    rs = cfg["real_speech"]
    local = hf_hub_download(c["repo"], fname, repo_type="dataset", cache_dir=str(paths.raw / "hf"))
    pf = pq.ParquetFile(local)
    out_rows = []
    per_spk_voice_s = defaultdict(float)
    stats = Counter()
    cols = [x for x in ["audio", c["spk"], c["text"], c["accent"], c["gender"]] if x]
    for batch in pf.iter_batches(batch_size=64, columns=cols):
        d = batch.to_pydict()
        for i in range(len(d["audio"])):
            stats["rows"] += 1
            a = d["audio"][i]
            data = a.get("bytes") if isinstance(a, dict) else None
            if not data:
                stats["no_bytes"] += 1
                continue
            try:
                x, sr = decode_audio_bytes(data)
                x = resample(x, sr, SR)
            except Exception:
                stats["decode_error"] += 1
                continue
            spk = f"{name}:{d[c['spk']][i]}"
            text = clean_tx(d[c["text"]][i]) if c["text"] else ""
            accent = (d[c["accent"]][i] if c["accent"] else None) or ("american" if name == "libritts_r" else "unknown")
            gender = (d[c["gender"]][i] if c["gender"] else None) or "unknown"
            x = trim_silence(x, SR, rel_db=-40, pad_s=0.05)
            dur = len(x) / SR
            peak = float(np.max(np.abs(x))) if len(x) else 0.0
            if dur < 0.25 or peak < 1e-3:
                stats["empty"] += 1
                continue
            if np.mean(np.abs(x) > 0.99) > 0.001:
                stats["clipped"] += 1
                continue
            rid = f"{name}_{stable_int(fname, i, spk) % 10**12:012d}"
            low = text.lower().strip(" .,!?")
            kind = None
            if "backchannel" in c["uses"] and low in BACKCHANNELS and dur <= 1.6:
                kind = "backchannel"
            elif "user_utt" in c["uses"] and rs["user_utt_min_s"] <= dur <= rs["user_utt_max_s"] and len(text.split()) >= 2:
                kind = "user_utt"
            elif "agent_voice" in c["uses"] and 1.5 <= dur <= 12 and voiced_ratio(x) > 0.6 \
                    and per_spk_voice_s[spk] < rs["agent_voice_material_s"]:
                kind = "agent_voice_material"
                per_spk_voice_s[spk] += dur
            if kind is None:
                stats["unused"] += 1
                continue
            path = paths.real / name / kind / f"{rid}.flac"
            if not path.exists():
                write_flac(path, x)
            out_rows.append({"id": rid, "corpus": name, "kind": kind, "speaker": spk, "text": text,
                             "accent": str(accent).lower(), "gender": str(gender).lower(), "dur": round(dur, 3),
                             "rms_db": round(rms_db(x), 1), "path": str(path.relative_to(paths.root)),
                             "license": c["license"]})
            stats[kind] += 1
    return out_rows, dict(stats)


def run_real_speech(args, cfg, paths):
    wt = worker_tag(args.worker_id)
    log = get_logger(f"real_speech.{wt}", paths.logs / "workers" / f"real_speech_{wt}.log")
    units = list_units(cfg)
    mine = [u for k, u in enumerate(units) if k % args.num_workers == args.worker_id]
    cl = Claimer(paths, "real_speech", wt, cfg["gen"]["stale_lock_s"])
    prog = Progress(paths, "real_speech", wt, len(mine))
    for name, f in mine + [u for u in units if u not in mine]:   # own units first, then help with the rest
        uid = unit_id(name, f)
        if not cl.try_claim(uid):
            continue
        t0 = time.time()
        try:
            rows, stats = process_unit(cfg, paths, name, f, log)
            write_jsonl(paths.real / "units" / f"{uid}.jsonl", rows)
            cl.mark_done(uid, {"rows": len(rows), "stats": stats, "seconds": round(time.time() - t0, 1)})
            prog.update(units=1, items=len(rows), last_unit=uid)
            log.info("unit %s: %d rows %s (%.0fs)", uid, len(rows), stats, time.time() - t0)
        except Exception as e:
            log.exception("unit %s failed", uid)
            log_error(paths, "real_speech", wt, e, unit=uid)
            cl.release(uid)


def add_my_recordings(cfg, paths, rows, log):
    d = cfg["real_speech"].get("my_recordings_dir")
    if not d or not Path(d).exists():
        return rows
    n = 0
    for f in sorted(Path(d).rglob("*")):
        if f.suffix.lower() not in (".wav", ".flac", ".ogg", ".mp3", ".webm", ".m4a"):
            continue
        x = trim_silence(load_24k(f), SR, rel_db=-40, pad_s=0.05)
        if len(x) < 0.3 * SR:
            continue
        txt = f.with_suffix(".txt")
        rid = f"mine_{stable_int(str(f)) % 10**12:012d}"
        path = paths.real / "mine" / "user_utt" / f"{rid}.flac"
        write_flac(path, x)
        spk = f"mine:{f.parent.name}"
        rows.append({"id": rid, "corpus": "mine", "kind": "user_utt", "speaker": spk,
                     "text": txt.read_text(encoding="utf-8").strip() if txt.exists() else "",
                     "accent": "own_recording", "gender": "unknown", "dur": round(len(x) / SR, 3),
                     "rms_db": round(rms_db(x), 1), "path": str(path.relative_to(paths.root)), "license": "own"})
        n += 1
    log.info("added %d own recordings from %s", n, d)
    return rows


def build_agent_voice(paths, utts, rng):
    """Concatenate clean utterances of ONE speaker into exactly 10.0 s (the fixed voice-prompt length)."""
    pieces, total = [], 0
    order = rng.permutation(len(utts))
    for k in order:
        x = load_24k(paths.root / utts[k]["path"])
        x = trim_silence(x, SR, rel_db=-35, pad_s=0.03)
        gap = np.zeros(int(rng.uniform(0.12, 0.3) * SR), np.float32)
        pieces += [x, gap]
        total += len(x) + len(gap)
        if total >= VOICE_SAMPLES:
            break
    if total < int(0.9 * VOICE_SAMPLES):
        return None
    y = np.concatenate(pieces)[:VOICE_SAMPLES]
    if len(y) < VOICE_SAMPLES:
        y = np.concatenate([y, np.zeros(VOICE_SAMPLES - len(y), np.float32)])
    y = loudness_normalize(y, SR, VOICE_LUFS)
    if np.max(np.abs(y)) > 0.99:
        y = y / np.max(np.abs(y)) * 0.98
    return y


def run_finalize(args, cfg, paths):
    log = get_logger("real_speech_finalize", paths.logs / "workers" / "real_speech_finalize.log")
    stage_log(paths, "real_speech_finalize", "running")
    rows = []
    for f in sorted((paths.real / "units").glob("*.jsonl")):
        rows += read_jsonl(f)
    rows = add_my_recordings(cfg, paths, rows, log)
    seed = cfg["seed"]
    sv, stt = cfg["splits"]["val"], cfg["splits"]["test"]

    # --- user-side real speech catalog (speaker-level held-out split)
    user_utts = [r for r in rows if r["kind"] == "user_utt"]
    bcs = [r for r in rows if r["kind"] == "backchannel"]
    for r in user_utts + bcs:
        r["split"] = split_of(r["speaker"], seed, sv, stt) if r["corpus"] != "mine" else \
            split_of(r["id"], seed, 0.1, 0.2)   # own recordings: 20% test, 10% val (small but most realistic eval)
    write_jsonl(paths.meta / "real_user_utterances.jsonl", user_utts)
    write_jsonl(paths.meta / "real_backchannels.jsonl", bcs)

    # --- extra agent voices: one 10 s clip per selected speaker, gender/accent balanced per corpus
    av = cfg["agent_voices"]
    by_spk = defaultdict(list)
    for r in rows:
        if r["kind"] in ("agent_voice_material", "user_utt") and r["corpus"] in av["extra_per_corpus"]:
            by_spk[(r["corpus"], r["speaker"])].append(r)
    voices = []
    for corpus, n_target in av["extra_per_corpus"].items():
        spks = [(k, v) for k, v in by_spk.items() if k[0] == corpus and sum(u["dur"] for u in v) >= 11.0]
        # alternate genders / accents for balance, deterministic order
        spks.sort(key=lambda kv: stable_int(seed, kv[0][1]))
        buckets = defaultdict(list)
        for k, v in spks:
            buckets[(v[0]["gender"], v[0]["accent"])].append((k, v))
        chosen = []
        while len(chosen) < n_target and any(buckets.values()):
            for key in sorted(buckets):
                if buckets[key] and len(chosen) < n_target:
                    chosen.append(buckets[key].pop(0))
        for (corp, spk), utts in chosen:
            vid = f"x_{corp}_{stable_int(spk) % 10**8:08d}"
            path = paths.agent_extra / f"{vid}.wav"
            if not path.exists():
                y = build_agent_voice(paths, utts, rng_for(seed, "voice", vid))
                if y is None:
                    continue
                write_wav(path, y)
            voices.append({"voice_id": vid, "kind": "extra", "corpus": corp, "speaker": spk,
                           "gender": utts[0]["gender"], "accent": utts[0]["accent"],
                           "license": utts[0]["license"]})
    # held-out extra voices: exact count, deterministic
    voices.sort(key=lambda v: stable_int(seed, "heldout", v["voice_id"]))
    n_ho = av["heldout_extra_voices"]
    for k, v in enumerate(voices):
        v["split"] = "test" if k < n_ho // 2 else ("val" if k < n_ho else "train")
    write_jsonl(paths.meta / "agent_voices_extra.jsonl", voices)
    summary = {"user_utts": len(user_utts), "user_utt_hours": round(sum(r["dur"] for r in user_utts) / 3600, 2),
               "user_utt_speakers": len({r["speaker"] for r in user_utts}),
               "user_utt_accents": dict(Counter(r["accent"] for r in user_utts).most_common(40)),
               "backchannels": len(bcs), "backchannel_speakers": len({r["speaker"] for r in bcs}),
               "extra_agent_voices": len(voices), "extra_by_corpus": dict(Counter(v["corpus"] for v in voices)),
               "extra_by_gender": dict(Counter(v["gender"] for v in voices)),
               "extra_by_split": dict(Counter(v["split"] for v in voices))}
    atomic_write_json(paths.logs / "sources" / "real_speech_summary.json", summary)
    log.info("real speech summary %s", json.dumps(summary))
    stage_log(paths, "real_speech_finalize", "done", **{k: v for k, v in summary.items() if isinstance(v, (int, float))})


def run_noise(args, cfg, paths):
    log = get_logger("noise", paths.logs / "workers" / "noise.log")
    stage_log(paths, "noise", "running")
    nc = cfg["noise"]
    cat = []
    if nc["musan"]:
        tgz = paths.raw / "musan.tar.gz"
        root = paths.raw / "musan"
        if not (root / "noise").exists():
            for url in nc["musan_urls"]:
                log.info("downloading MUSAN from %s (resumable)", url)
                r = subprocess.run(["curl", "-L", "-C", "-", "--retry", "10", "--retry-delay", "5", "-o", str(tgz), url])
                if r.returncode == 0 and tgz.exists() and tgz.stat().st_size > 10_000_000_000:
                    break
            if tgz.exists():
                log.info("extracting MUSAN")
                with tarfile.open(tgz) as tar:
                    tar.extractall(paths.raw)
        if (root / "noise").exists():
            for kind in ("noise", "music", "speech"):
                for f in sorted((root / kind).rglob("*.wav")):
                    try:
                        import soundfile as sf
                        info = sf.info(str(f))
                        dur = info.frames / info.samplerate
                    except Exception:
                        continue
                    if dur < 1.0:
                        continue
                    nid = f"musan_{kind}_{f.stem}"
                    cat.append({"noise_id": nid, "kind": {"noise": "event_or_ambient", "music": "music",
                                                          "speech": "babble_source"}[kind],
                                "path": str(f), "dur": round(dur, 2), "license": "CC-BY-4.0 (MUSAN)",
                                "split": split_of(nid, cfg["seed"], cfg["splits"]["val"], cfg["splits"]["test"])})
        else:
            log.warning("MUSAN unavailable -- continuing with procedural noise only (see logs/stages/noise.json)")
    write_jsonl(paths.meta / "noise_catalog.jsonl", cat)
    summary = {"musan_files": len(cat), "by_kind": dict(Counter(c["kind"] for c in cat)),
               "hours": round(sum(c["dur"] for c in cat) / 3600, 2), "procedural": ["white", "pink", "brown", "hum", "fan"]}
    atomic_write_json(paths.logs / "sources" / "noise_summary.json", summary)
    stage_log(paths, "noise", "done", **{k: v for k, v in summary.items() if isinstance(v, (int, float))})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--stage", required=True, choices=["real_speech", "real_speech_finalize", "noise"])
    ap.add_argument("--worker-id", type=int, default=0)
    ap.add_argument("--num-workers", type=int, default=1)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    {"real_speech": run_real_speech, "real_speech_finalize": run_finalize, "noise": run_noise}[args.stage](args, cfg, paths)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/voices.py
"""Stages `ui_voices` and `voice_codes` (GPU).

ui_voices: the 18 web-UI voices only ship as TEACHER embedding caches (voices.tgz: NATF0.pt ... VARM4.pt), which
  the student cannot consume (server.py rejects .pt for --model student). For each UI voice we render
  `ui_clips_per_voice` raw-audio clips with the teacher speaking in that voice, keep only voiced audio, and cut
  exactly 10.0 s (125 frames). Clip 0 is the DEPLOYMENT clip: copied to voices/agent/ui_deploy/<NAME>.wav --
  point the streaming notebook's STUDENT_VOICE_DIR at that folder so inference uses the exact training clip.

voice_codes: every agent voice clip (UI + extra) is encoded exactly like LMGen.load_voice_prompt +
  _encode_voice_prompt_frames does at inference (moshi.models.lm.load_audio -> normalize_audio(-24 LUFS) ->
  _iterate_audio(pad=True) -> encode_from_sphn with a freshly reset streaming Mimi). The result [8, 125] is what
  selfplay feeds as `moshi_tokens`, and what validate.py checks the recorded prefix against.
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np

from .common import (P, SR, FRAME, VOICE_SAMPLES, VOICE_FRAMES, VOICE_LUFS, load_cfg, get_logger, stage_log,
                     write_jsonl, read_jsonl, write_wav, atomic_torch_save, Claimer, Progress, worker_tag, log_error,
                     frame_db, loudness_normalize, stable_int, atomic_write_json)

UI_VOICES = ["NATF0", "NATF1", "NATF2", "NATF3", "NATM0", "NATM1", "NATM2", "NATM3",
             "VARF0", "VARF1", "VARF2", "VARF3", "VARF4", "VARM0", "VARM1", "VARM2", "VARM3", "VARM4"]
RENDER_PERSONAS = [
    "You enjoy having a good conversation.",
    "You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way.",
    "You work for CitySan Services which is a waste management company and your name is Ayelen Lucero.",
    "You are a knowledgeable personal finance coach. Explain budgeting, saving, and debt in simple terms.",
    "You enjoy having a good conversation. Have a deep conversation about technology.",
    "You are an enthusiastic tour guide describing a city's history to a visitor.",
]


def voiced_frames(pcm, thresh_rel=-30.0, abs_floor=-50.0):
    """Keep 80 ms frames that carry agent speech (full-duplex output has pauses while listening)."""
    d = frame_db(pcm, FRAME)
    if len(d) == 0:
        return np.zeros(0, np.float32)
    keep = d > max(d.max() + thresh_rel, abs_floor)
    grown = keep.copy()
    grown[1:] |= keep[:-1]
    grown[:-1] |= keep[1:]
    return pcm[: len(d) * FRAME].reshape(-1, FRAME)[grown].reshape(-1)


def render_ui(args, cfg, paths):
    import torch
    import sentencepiece
    from moshi.models import loaders, LMGen
    from moshi.models.lm import load_audio, _iterate_audio, encode_from_sphn
    from moshi.offline import warmup, decode_tokens_to_pcm, wrap_with_system_tags, seed_all

    wt = worker_tag(args.worker_id)
    log = get_logger(f"ui_voices.{wt}", paths.logs / "workers" / f"ui_voices_{wt}.log")
    t = cfg["teacher"]
    n_clips = cfg["agent_voices"]["ui_clips_per_voice"]
    ui_names = cfg["agent_voices"].get("ui_voices", UI_VOICES)
    units = [(v, k) for v in ui_names for k in range(n_clips)]
    cl = Claimer(paths, "ui_voices", wt, cfg["gen"]["stale_lock_s"])
    prog = Progress(paths, "ui_voices", wt, len(units))
    todo = [u for u in units if not cl.is_done(f"{u[0]}_{u[1]}")]
    if not todo:
        return
    dev = "cuda"
    with torch.no_grad():
        mimi = loaders.get_mimi(t["mimi_weight"], dev)
        other = loaders.get_mimi(t["mimi_weight"], dev)
        tok = sentencepiece.SentencePieceProcessor(t["tokenizer"])
        lm = loaders.get_moshi_lm(t["moshi_weight"], device=dev)
        lm.eval()
        lm_gen = LMGen(lm, audio_silence_frame_cnt=int(0.5 * mimi.frame_rate), sample_rate=mimi.sample_rate,
                       device=dev, frame_rate=mimi.frame_rate)
        mimi.streaming_forever(1)
        other.streaming_forever(1)
        lm_gen.streaming_forever(1)
        warmup(mimi, other, lm_gen, dev, FRAME)
        inputs = [load_audio(str(Path(cfg["repo_dir"]) / "assets/test/input_assistant.wav"), SR),
                  load_audio(str(Path(cfg["repo_dir"]) / "assets/test/input_service.wav"), SR)]
        for name, k in todo:
            unit = f"{name}_{k}"
            if not cl.try_claim(unit):
                continue
            try:
                best = np.zeros(0, np.float32)
                for attempt in range(6):
                    seed_all(stable_int(cfg["seed"], unit, attempt) % (2 ** 31))
                    lm_gen.load_voice_prompt_embeddings(str(Path(t["voices_dir"]) / f"{name}.pt"))
                    persona = RENDER_PERSONAS[(k + attempt) % len(RENDER_PERSONAS)]
                    lm_gen.text_prompt_tokens = tok.encode(wrap_with_system_tags(persona))
                    mimi.reset_streaming(); other.reset_streaming(); lm_gen.reset_streaming()
                    lm_gen.step_system_prompts(mimi)
                    mimi.reset_streaming()
                    user = inputs[(k + attempt) % 2]
                    user = np.concatenate([user, np.zeros((1, 4 * SR), np.float32), user], axis=-1)  # ~90 s of input
                    frames = []
                    for n_in, enc in enumerate(encode_from_sphn(mimi, _iterate_audio(user, sample_interval_size=FRAME,
                                                                                       pad=True), max_batch=1)):
                        for c in range(enc.shape[-1]):
                            tokens = lm_gen.step(enc[:, :, c:c + 1])
                            if tokens is not None:
                                frames.append(decode_tokens_to_pcm(mimi, other, lm_gen, tokens))
                        # stop as soon as 10 s of voiced agent audio exist (+1 s margin) instead of streaming ~90 s
                        if n_in % 25 == 24 and frames and                                 len(voiced_frames(np.concatenate(frames))) >= VOICE_SAMPLES + 12 * FRAME:
                            break
                    clip = voiced_frames(np.concatenate(frames))
                    if len(clip) > len(best):
                        best = clip
                    if len(best) >= VOICE_SAMPLES:
                        break
                if len(best) < int(0.8 * VOICE_SAMPLES):
                    raise RuntimeError(f"only {len(best)/SR:.1f}s voiced audio after 6 attempts")
                if len(best) < VOICE_SAMPLES:  # 8-10 s: loop the voiced audio to exactly 10 s (fixed prefix length)
                    best = np.concatenate([best, best])[:VOICE_SAMPLES]
                clip = loudness_normalize(best[:VOICE_SAMPLES], SR, VOICE_LUFS)
                path = paths.agent_ui / f"{name}_{k}.wav"
                write_wav(path, clip)
                if k == 0:
                    write_wav(paths.agent_deploy / f"{name}.wav", clip)
                cl.mark_done(unit, {"attempts": attempt + 1, "voiced_s": round(len(best) / SR, 2)})
                prog.update(units=1, items=1, last=unit)
                log.info("rendered %s", unit)
            except Exception as e:
                log.exception("ui voice %s failed", unit)
                log_error(paths, "ui_voices", wt, e, unit=unit)
                cl.release(unit)


def catalog_voices(cfg, paths):
    rows = []
    n_clips = cfg["agent_voices"]["ui_clips_per_voice"]
    for name in cfg["agent_voices"].get("ui_voices", UI_VOICES):
        for k in range(n_clips):
            p = paths.agent_ui / f"{name}_{k}.wav"
            if p.exists():
                # clip 0 = deployment clip (all splits); last clip held out for val/test; others train
                split = "all" if k == 0 else ("heldout" if k == n_clips - 1 else "train")
                rows.append({"clip_id": f"ui_{name}_{k}", "voice_id": f"ui_{name}", "kind": "ui", "name": name,
                             "clip_index": k, "deploy": k == 0, "path": str(p.relative_to(paths.root)),
                             "split": split, "gender": "female" if name[3] == "F" else "male",
                             "accent": "natural" if name.startswith("NAT") else "variety", "corpus": "teacher_render"})
    for v in read_jsonl(paths.meta / "agent_voices_extra.jsonl"):
        p = paths.agent_extra / f"{v['voice_id']}.wav"
        if p.exists():
            rows.append({**v, "clip_id": v["voice_id"], "clip_index": 0, "deploy": False,
                         "path": str(p.relative_to(paths.root)), "name": v["voice_id"]})
    return rows


def encode_codes(args, cfg, paths):
    import torch
    from moshi.models import loaders
    from moshi.models.lm import load_audio, normalize_audio, _iterate_audio, encode_from_sphn

    wt = worker_tag(args.worker_id)
    log = get_logger(f"voice_codes.{wt}", paths.logs / "workers" / f"voice_codes_{wt}.log")
    rows = catalog_voices(cfg, paths)
    with torch.no_grad():
        mimi = loaders.get_mimi(cfg["teacher"]["mimi_weight"], "cuda")
        bad = []
        for r in rows:
            out = paths.voice_codes / f"{r['clip_id']}.pt"
            if not out.exists():
                raw = load_audio(str(paths.root / r["path"]), SR)                  # == LMGen.load_voice_prompt
                raw = normalize_audio(raw, SR, VOICE_LUFS)
                if raw.ndim == 1:
                    raw = raw[None, :]
                with mimi.streaming(1):
                    codes = torch.cat(list(encode_from_sphn(mimi, _iterate_audio(raw, sample_interval_size=FRAME, pad=True),
                                                            max_batch=1)), dim=-1)[0].cpu().long()
                if tuple(codes.shape) != (8, VOICE_FRAMES):
                    bad.append((r["clip_id"], tuple(codes.shape)))
                    continue
                atomic_torch_save(out, codes)
            r["codes"] = str(out.relative_to(paths.root))
    rows = [r for r in rows if "codes" in r]
    write_jsonl(paths.meta / "agent_voices.jsonl", rows)
    if bad:
        log.error("voice clips with wrong frame count (must be 125): %s", bad)
    atomic_write_json(paths.logs / "sources" / "agent_voices_summary.json",
                      {"clips": len(rows), "ui_clips": sum(r["kind"] == "ui" for r in rows),
                       "extra_clips": sum(r["kind"] == "extra" for r in rows), "bad": bad})
    log.info("voice codes: %d clips (%d bad)", len(rows), len(bad))
    stage_log(paths, "voice_codes", "done", clips=len(rows), bad=len(bad))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--stage", required=True, choices=["ui_voices", "voice_codes"])
    ap.add_argument("--worker-id", type=int, default=0)
    ap.add_argument("--num-workers", type=int, default=1)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    (render_ui if args.stage == "ui_voices" else encode_codes)(args, cfg, paths)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/tts.py
"""Stages `tts_speakers` and `tts` (GPU): synthetic USER speakers and user-utterance rendering.

Engine: Kokoro-82M (Apache-2.0, `kokoro==0.9.4`, 24 kHz output == Mimi's rate, no resampling).
  * 28 English voices (20 US `a*`, 8 UK `b*`) are the bases.
  * Extra speakers are deterministic weighted blends of 2-3 same-accent-family bases (voice packs are
    [510, 1, 256] style tensors; KPipeline.load_single_voice loads a `.pt` path, so each blend is a .pt file).
  * Every speaker gets a base speaking rate; each utterance jitters it slightly.
Why not a zero-shot cloning TTS: chatterbox-tts (MIT) pins torch==2.6.0 (no Blackwell/sm_120 kernels) and
safetensors==0.5.3 (conflicts with moshi's <0.5 pin) -- verified on PyPI; accent diversity therefore comes
from the REAL speech corpora (audio_sources.py) rather than from TTS.
"""
import argparse
import json
import time
from collections import Counter

import numpy as np

from .common import (P, SR, load_cfg, get_logger, stage_log, write_jsonl, read_jsonl, write_flac, Claimer, Progress,
                     worker_tag, log_error, rng_for, split_of, trim_silence, rms_db, atomic_write_json)

KOKORO_REPO = "hexgrad/Kokoro-82M"
BASE = {  # name -> (lang_code, gender)
    **{v: ("a", "female") for v in ["af_alloy", "af_aoede", "af_bella", "af_heart", "af_jessica", "af_kore",
                                    "af_nicole", "af_nova", "af_river", "af_sarah", "af_sky"]},
    **{v: ("a", "male") for v in ["am_adam", "am_echo", "am_eric", "am_fenrir", "am_liam", "am_michael", "am_onyx",
                                  "am_puck", "am_santa"]},
    **{v: ("b", "female") for v in ["bf_alice", "bf_emma", "bf_isabella", "bf_lily"]},
    **{v: ("b", "male") for v in ["bm_daniel", "bm_fable", "bm_george", "bm_lewis"]},
}


def build_speakers(args, cfg, paths):
    import torch
    from huggingface_hub import hf_hub_download
    log = get_logger("tts_speakers", paths.logs / "workers" / "tts_speakers.log")
    stage_log(paths, "tts_speakers", "running")
    tc = cfg["tts"]
    packs = {n: torch.load(hf_hub_download(KOKORO_REPO, f"voices/{n}.pt"), weights_only=True) for n in BASE}
    speakers = []
    for n, (lang, g) in BASE.items():
        r = rng_for(cfg["seed"], "tts_base", n)
        speakers.append({"speaker_id": f"tts_{n}", "voice": n, "lang": lang, "gender": g,
                         "accent": "us" if lang == "a" else "uk", "rate": round(float(r.uniform(*tc["rate_range"])), 3),
                         "blend": {n: 1.0}})
    for i in range(tc["n_blend_speakers"]):
        r = rng_for(cfg["seed"], "tts_blend", i)
        lang = "a" if r.random() < tc["us_fraction"] else "b"
        g = "female" if r.random() < 0.5 else "male"
        same = [n for n, (l, gg) in BASE.items() if l == lang and gg == g]
        other = [n for n, (l, gg) in BASE.items() if l == lang and gg != g]
        k = 2 if r.random() < 0.7 else 3
        names = list(r.choice(same, size=min(k, len(same)), replace=False))
        if r.random() < 0.12 and other:           # a little cross-gender mixing -> more varied pitch/timbre
            names[-1] = str(r.choice(other))
        w = r.dirichlet(np.ones(len(names)) * 1.5)
        pack = sum(float(wi) * packs[n] for wi, n in zip(w, names))
        sid = f"tts_blend_{i:04d}"
        path = paths.tts_voices / f"{sid}.pt"
        if not path.exists():
            torch.save(pack.float().cpu(), path)
        speakers.append({"speaker_id": sid, "voice": str(path), "lang": lang, "gender": g,
                         "accent": "us" if lang == "a" else "uk", "rate": round(float(r.uniform(*tc["rate_range"])), 3),
                         "blend": {n: round(float(wi), 3) for wi, n in zip(w, names)}})
    for s in speakers:
        s["split"] = split_of(s["speaker_id"], cfg["seed"], cfg["splits"]["val"] * 2, cfg["splits"]["test"] * 2)
    write_jsonl(paths.meta / "tts_speakers.jsonl", speakers)
    summ = {"n": len(speakers), "by_accent": dict(Counter(s["accent"] for s in speakers)),
            "by_gender": dict(Counter(s["gender"] for s in speakers)), "by_split": dict(Counter(s["split"] for s in speakers))}
    atomic_write_json(paths.logs / "sources" / "tts_speakers_summary.json", summ)
    log.info("tts speakers: %s", summ)
    stage_log(paths, "tts_speakers", "done", n=len(speakers))


def render(args, cfg, paths):
    import torch
    from kokoro import KPipeline
    wt = worker_tag(args.worker_id)
    log = get_logger(f"tts.{wt}", paths.logs / "workers" / f"tts_{wt}.log")
    shards = sorted(p.stem for p in (paths.plan / "tts_jobs").glob("*.jsonl"))
    cl = Claimer(paths, "tts", wt, cfg["gen"]["stale_lock_s"])
    prog = Progress(paths, "tts", wt, len(shards))
    pipes = {}

    def pipe(lang):
        if lang not in pipes:
            model = next(iter(pipes.values())).model if pipes else None
            pipes[lang] = KPipeline(lang_code=lang, repo_id=KOKORO_REPO, model=model if model is not None else True,
                                    device="cuda")
        return pipes[lang]

    order = shards[args.worker_id::args.num_workers] + shards
    for sh in order:
        if not cl.try_claim(sh):
            continue
        t0 = time.time()
        ok = bad = 0
        fails = []
        try:
            for job in read_jsonl(paths.plan / "tts_jobs" / f"{sh}.jsonl"):
                out = paths.tts_audio / job["utt_id"][:2] / f"{job['utt_id']}.flac"
                if out.exists():
                    ok += 1
                    continue
                try:
                    chunks = [r.audio.detach().float().cpu().numpy() for r in
                              pipe(job["lang"])(job["text"], voice=job["voice"], speed=job["speed"])
                              if r.audio is not None]
                    x = np.concatenate(chunks) if chunks else np.zeros(0, np.float32)
                    x = trim_silence(x, SR, rel_db=-45, pad_s=0.04)
                    dur = len(x) / SR
                    chars = max(1, len(job["text"]))
                    # plausibility: Kokoro speaks ~8-25 chars/s; far outside means a G2P/model failure
                    if dur < 0.2 or not (3.0 <= chars / dur <= 40.0) or rms_db(x) < -60:
                        raise ValueError(f"implausible audio dur={dur:.2f}s chars={chars}")
                    write_flac(out, x.astype(np.float32))
                    ok += 1
                except Exception as e:
                    bad += 1
                    fails.append({"utt_id": job["utt_id"], "text": job["text"][:200], "error": repr(e)})
                cl.heartbeat(sh)
            if fails:
                write_jsonl(paths.logs / "errors" / f"tts_failed_{sh}.jsonl", fails)
            cl.mark_done(sh, {"ok": ok, "failed": bad, "seconds": round(time.time() - t0, 1)})
            prog.update(units=1, items=ok, failed=bad, last=sh)
            log.info("tts shard %s ok=%d failed=%d %.0fs", sh, ok, bad, time.time() - t0)
        except Exception as e:
            log.exception("tts shard %s crashed", sh)
            log_error(paths, "tts", wt, e, shard=sh)
            cl.release(sh)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--stage", required=True, choices=["tts_speakers", "tts"])
    ap.add_argument("--worker-id", type=int, default=0)
    ap.add_argument("--num-workers", type=int, default=1)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    (build_speakers if args.stage == "tts_speakers" else render)(args, cfg, paths)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/planner.py
"""Stage `plan`: deterministic plan of every sample (who, what, how it sounds, when people talk).

Nothing random happens later that is not seeded from this plan, so a crashed run resumes into identical data.
Batching rule (from LMGen): a batch can only share one prompt schedule, so every sample in a GPU group has the
same prefix length = 136 + n_persona_tokens and the same total frame count.

Outputs:
  _work/plan/shards/shard_XXXXX.json   groups of full sample plans (the self-play work units)
  _work/plan/tts_jobs/tts_XXXXX.jsonl  unique (text, speaker, rate) utterances to synthesize
  _work/plan/samples_index.jsonl       one light row per sample (for stats/assembly)
  logs/plan_summary.json               planned hours per split/scenario/persona/voice/acoustics
"""
import argparse
import hashlib
import json
import math
import time
from collections import Counter, defaultdict

import numpy as np

from .common import (P, FRAME_RATE, PREFIX_BASE, load_cfg, get_logger, stage_log, read_jsonl, write_jsonl,
                     atomic_write_json, rng_for, weighted_choice, stable_int)

FOLLOWUPS_GENERIC = [
    "Can you explain that more simply?", "Why is that?", "How long does that usually take?",
    "What would you recommend?", "Can you give me an example?", "Got it. What's the next step?",
    "Hmm, are you sure about that?", "That makes sense. One more question.", "So what should I do first?",
    "Interesting! Tell me more.", "What are the pros and cons?", "Is that the same for everyone?",
    "Okay. How does that compare to other options?", "Wait, what does that mean exactly?",
    "Could you go over that again?", "And what if that doesn't work?", "Is there an easier way?",
    "Okay, that's helpful. Anything else I should know?", "Right. And how much does that cost?",
    "Do you think that's a good idea?",
]
FOLLOWUPS_FINANCE = [
    "What's the interest rate on that?", "Is that tax deductible?", "How does that affect my credit score?",
    "Should I pay off debt first or save?", "What are the fees?", "Is that a safe investment?",
    "How much should I keep in an emergency fund?", "What happens if I miss a payment?",
    "Is that better than a savings account?", "How long until I see returns?", "What's the risk there?",
    "Can I withdraw it early?", "Does inflation change that?", "What's a good first step for a beginner?",
]
FOLLOWUPS_CS = [
    "Okay, can you check that for me?", "How long will that take?", "Is there a fee for that?",
    "Can you send me a confirmation?", "What's the earliest you can do?", "Can I speak to a manager?",
    "That's not what I was told last time.", "Great, thank you. Is that everything?", "Can you repeat the date?",
    "Do I need to do anything on my end?",
]
FOLLOWUPS_CRYPTO = [
    "Is that safe to do?", "How much does that cost in fees?", "How long does that usually take?",
    "Is that the same for every coin?", "What could go wrong there?", "Is there a safer way to do that?",
    "How do I know if that's legit?", "What's the risk if I get that wrong?", "Does that work the same on Ethereum?",
    "What happens if I mess that up?", "Is that reversible?", "Do I need to pay gas for that?",
    "How is that different from a regular bank?", "Should a beginner worry about that?",
]
# Explicit "stop the current answer" vocabulary (barge-in / interruption training): the model must drop whatever
# it was saying mid-turn and respond to the new user turn instead -- see selfplay.py's live barge-in simulator.
INTERRUPT_PREFIX = ["Wait, sorry,", "Sorry to cut in, but", "Hold on,", "Actually,", "No, no, I meant",
                    "Sorry, quick question,", "Hang on,", "Oh wait,", "Let me stop you there,", "Um, sorry,",
                    "Stop, sorry,", "Wait, never mind that,", "Sorry, stop, actually,", "Hold on, wait,"]
INTERRUPT_SHORT = ["Wait, wait.", "Hold on a second.", "Okay, okay, got it.", "Sorry, can I stop you there?",
                   "No, that's not it.", "Hang on.", "Okay, stop, I get it.", "Stop.", "Stop talking.",
                   "That's enough.", "Wait.", "Hold on.", "Never mind.", "Never mind, forget it.",
                   "Okay stop, different question.", "Actually, never mind that."]
CLARIFY = ["Sorry, could you repeat that?", "What do you mean by that?", "I didn't catch the last part.",
           "Wait, say that again?", "Pardon?", "Sorry, you cut out for a second.", "Can you say that slower?"]
RAPID_SHORT = ["Yes.", "No, the other one.", "Okay.", "Sounds good.", "Sure, go ahead.", "Not really.", "Maybe later.",
               "Yeah, that one.", "Right.", "Uh, no.", "Perfect.", "Yep, that's me.", "Correct.", "Hmm, not sure."]
NUDGES = ["Hello? Are you there?", "Hi, can you hear me?", "Hello?"]
BACKCHANNELS = ["Mm-hmm.", "Yeah.", "Uh-huh.", "Right.", "Okay.", "I see.", "Oh, okay.", "Sure.", "Got it.", "Yep.",
                "Oh really?", "Wow.", "Hmm.", "Right, right.", "Makes sense.", "Oh nice.", "Exactly.", "Yeah, yeah."]
GREETINGS = ["Hi,", "Hello,", "Hey,", "Hi there,", "Good morning,", "Hey, um,", "Hi, yeah,"]
OPENERS = ["So,", "Okay so,", "Yeah,", "Um,", "Uh,", "Alright,", "Well,"]
CLOSERS = ["Thanks, that's all I needed.", "Great, thank you so much.", "Okay, thanks. Bye!", "Perfect, thanks for your help."]
SEC_PER_TURN = {"multi_turn": 12, "single_followup": 20, "barge_in": 9, "backchannel": 16, "pauses": 15,
                "rapid": 5, "casual_real": 8, "distractor": 18, "repeat_clarify": 11}


def disfluent(text, r):
    """Light, realistic disfluency: filler at the start or after the first clause, sometimes a restart."""
    u = r.random()
    if u < 0.10:
        return f"{r.choice(['Um,', 'Uh,'])} {text[0].lower() + text[1:]}"
    if u < 0.16 and "," in text:
        a, b = text.split(",", 1)
        return f"{a}, um,{b}"
    if u < 0.20:
        w = text.split()
        if len(w) > 4:
            return f"{' '.join(w[:2])}- {' '.join(w)}"
    return text


class Pools:
    def __init__(self, cfg, paths):
        self.cfg = cfg
        self.personas = read_jsonl(paths.meta / "personas.jsonl")
        self.questions = read_jsonl(paths.meta / "questions.jsonl")
        self.chains = read_jsonl(paths.meta / "chains.jsonl")
        self.voices = read_jsonl(paths.meta / "agent_voices.jsonl")
        self.tts_spk = read_jsonl(paths.meta / "tts_speakers.jsonl")
        self.real_utts = read_jsonl(paths.meta / "real_user_utterances.jsonl")
        self.real_bcs = read_jsonl(paths.meta / "real_backchannels.jsonl")
        self.noise = read_jsonl(paths.meta / "noise_catalog.jsonl")
        for name, v in [("personas", self.personas), ("questions", self.questions), ("agent_voices", self.voices),
                        ("tts_speakers", self.tts_spk)]:
            if not v:
                raise RuntimeError(f"planner: meta/{name} is empty -- run the earlier stages first")

        def by(items, *keys):
            d = defaultdict(list)
            for it in items:
                d[tuple(it[k] for k in keys)].append(it)
            return d
        self.p_by = by(self.personas, "split", "category")
        self.q_by = by(self.questions, "split", "domain")
        self.c_by = by(self.chains, "split", "domain")
        self.spk_by = {k[0]: v for k, v in by(self.tts_spk, "split").items()}
        self.ru_by_spk = defaultdict(list)
        for u in self.real_utts:
            self.ru_by_spk[(u["split"], u["speaker"])].append(u)
        self.real_spk_by = defaultdict(list)
        for (sp, spk), us in self.ru_by_spk.items():
            if len(us) >= 4:
                self.real_spk_by[sp].append(spk)
        for k in self.real_spk_by:
            self.real_spk_by[k].sort()
        self.bc_by_spk = defaultdict(list)
        for b in self.real_bcs:
            self.bc_by_spk[b["speaker"]].append(b)
        self.noise_by = by(self.noise, "split", "kind")
        self.ui = defaultdict(list)
        self.extra = defaultdict(list)
        for v in self.voices:
            if v["kind"] == "ui":
                self.ui[v["name"]].append(v)
            else:
                self.extra[v["split"]].append(v)
        self.ui_names = sorted(self.ui)


def pool_split(split):
    return {"train": "train", "train_long": "train", "val": "val", "val_long": "val", "test": "test",
            "test_deploy": "test"}[split]


def pick(r, items):
    if not items:
        raise RuntimeError("planner: empty candidate pool (check meta/*.jsonl and split assignment)")
    return items[int(r.integers(0, len(items)))]


def pick_q(pools, r, psplit, domains):
    for _ in range(8):
        d = domains[int(r.integers(0, len(domains)))]
        items = pools.q_by.get((psplit, d))
        if items:
            return pick(r, items)
    items = pools.q_by.get((psplit, "general"))
    if not items:
        items = [q for (sp, _), qs in pools.q_by.items() if sp == psplit for q in qs]
    if not items:
        raise RuntimeError(f"planner: no questions available for split {psplit}")
    return pick(r, items)


def content_domains(persona):
    cat, dom = persona["category"], persona["domain"]
    if cat == "crypto":
        return ["crypto", "crypto", "crypto", "general"]
    if cat == "identity":
        return ["identity"]
    if cat == "finance":
        return ["finance", "finance", "finance_cs"]
    if cat == "customer_service":
        return ["customer_service", "customer_service", "finance_cs"] if dom != "finance_cs" else ["finance_cs"]
    if dom == "finance":
        return ["finance", "general"]
    if dom == "crypto":
        return ["crypto", "general"]
    if dom == "health":
        return ["health", "health", "general"]
    return ["general"]


def build_turns(pools, r, s, persona, scenario, psplit, n_turns, real_spk):
    """Scripted user turns. The agent is NOT scripted -- the teacher answers live."""
    cfg = pools.cfg
    doms = content_domains(persona)
    fus = FOLLOWUPS_GENERIC + (FOLLOWUPS_FINANCE if "finance" in doms else []) + \
        (FOLLOWUPS_CS if persona["category"] == "customer_service" else []) + \
        (FOLLOWUPS_CRYPTO if "crypto" in doms else [])
    turns = []
    # optional chain backbone (real multi-turn user sequences)
    chain = None
    if persona["category"] == "casual":
        cs = pools.c_by.get((psplit, "casual"))
        chain = pick(r, cs) if cs else None
    elif persona["category"] == "crypto" and scenario in ("multi_turn", "single_followup", "pauses") \
            and r.random() < 0.6:
        cs = pools.c_by.get((psplit, "crypto"))
        chain = pick(r, cs) if cs else None
    elif persona["category"] in ("assistant", "roleplay") and scenario in ("multi_turn", "single_followup", "pauses") \
            and r.random() < 0.5:
        cs = pools.c_by.get((psplit, "general"))
        chain = pick(r, cs) if cs else None
    chain_turns = list(chain["turns"]) if chain else []
    faq = list(persona.get("faq") or [])
    r.shuffle(faq)

    def next_content(j):
        if chain_turns:
            return chain_turns.pop(0), "chain", chain["chain_id"]
        if faq and persona["category"] in ("customer_service", "finance", "identity") and r.random() < 0.45:
            return faq.pop(0), "persona_faq", persona["persona_id"]
        if j > 0 and r.random() < (0.45 if scenario == "single_followup" else 0.3):
            return fus[int(r.integers(0, len(fus)))], "followup", None
        q = pick_q(pools, r, psplit, doms)
        return q["text"], "question", q["qid"]

    real_list = []
    if real_spk:
        real_list = sorted(pools.ru_by_spk[(psplit, real_spk)], key=lambda u: u["id"])
        start = int(r.integers(0, max(1, len(real_list) - n_turns)))
        real_list = real_list[start:]
    for j in range(n_turns):
        t = {"idx": j, "interrupt": False, "barge_s": None}
        if scenario == "casual_real" and real_list:
            u = real_list.pop(0)
            t.update(source="real", path=u["path"], text=u["text"], content="real_casual", ref=u["id"])
            turns.append(t)
            continue
        if scenario == "rapid" and j > 0 and r.random() < 0.5:
            text, content, ref = RAPID_SHORT[int(r.integers(0, len(RAPID_SHORT)))], "rapid_short", None
        elif scenario == "repeat_clarify" and j > 0 and j % 2 == 1:
            text, content, ref = CLARIFY[int(r.integers(0, len(CLARIFY)))], "clarify", None
        else:
            text, content, ref = next_content(j)
        if scenario == "barge_in" and j > 0 and r.random() < 0.65:
            t["interrupt"] = True
            t["barge_s"] = round(float(r.uniform(*cfg["timing"]["barge_after_s"])), 2)
            if r.random() < 0.25:
                text, content = INTERRUPT_SHORT[int(r.integers(0, len(INTERRUPT_SHORT)))], "interrupt_short"
            else:
                pre = INTERRUPT_PREFIX[int(r.integers(0, len(INTERRUPT_PREFIX)))]
                text = f"{pre} {text[0].lower() + text[1:]}"
                content = "interrupt_" + content
        # spoken-style augmentation (TTS turns only)
        if j == 0 and r.random() < 0.35 and content not in ("rapid_short",):
            text = f"{GREETINGS[int(r.integers(0, len(GREETINGS)))]} {text[0].lower() + text[1:]}"
        elif j > 0 and r.random() < 0.15:
            text = f"{OPENERS[int(r.integers(0, len(OPENERS)))]} {text[0].lower() + text[1:]}"
        text = disfluent(text, r)
        if real_spk and scenario != "casual_real" and j > 0 and real_list and \
                r.random() < cfg["real_speech"]["generic_turn_real_frac"]:
            u = real_list.pop(0)
            t.update(source="real", path=u["path"], text=u["text"], content="real_generic", ref=u["id"])
        else:
            t.update(source="tts", text=text, content=content, ref=ref)
        turns.append(t)
    if r.random() < 0.10 and turns and turns[-1]["source"] == "tts":
        turns.append({"idx": len(turns), "interrupt": False, "barge_s": None, "source": "tts",
                      "text": CLOSERS[int(r.integers(0, len(CLOSERS)))], "content": "closer", "ref": None})
    return turns


def acoustic_profile(pools, r, psplit, scenario):
    a = pools.cfg["acoustics"]
    nclass = weighted_choice(r, a["noise_class_weights"])
    bg = {"class": nclass}
    if nclass == "clean":
        bg.update(type="digital", level_db=float(r.uniform(-90, -80)))
    else:
        btype = weighted_choice(r, a["bg_type_weights"])
        src = {"ambient": "event_or_ambient", "music": "music", "babble": "babble_source"}.get(btype)
        files = pools.noise_by.get((psplit, src)) if src else None
        if src and files:
            k = 3 if btype == "babble" else 1
            bg.update(type=btype, files=[pick(r, files)["path"] for _ in range(k)])
        else:
            bg.update(type="procedural", kind=str(r.choice(["pink", "brown", "fan", "hum", "white"])))
        if nclass == "quiet_room":
            bg["abs_db"] = float(r.uniform(-62, -48))
        else:
            lo, hi = {"moderate": (15, 25), "noisy": (5, 15), "very_noisy": (0, 5)}[nclass]
            bg["snr_db"] = float(r.uniform(lo, hi))
    events = []
    if scenario == "distractor" or r.random() < a["event_prob_other"]:
        files = pools.noise_by.get((psplit, "event_or_ambient"))
        n_ev = int(r.integers(3, 9)) if scenario == "distractor" else int(r.integers(1, 3))
        for _ in range(n_ev):
            ev = {"at_s": None, "level_db": float(r.uniform(-32, -14)), "dur_s": float(r.uniform(0.4, 3.5))}
            if files:
                ev["path"] = pick(r, files)["path"]
            else:
                ev["kind"] = str(r.choice(["white", "pink"]))
            events.append(ev)
    return {
        "bg": bg, "events": events,
        "rir": ({"rt60": float(r.uniform(0.15, 0.8)), "drr_db": float(r.uniform(0, 12))}
                if r.random() < a["rir_prob"] else None),
        "mic": weighted_choice(r, a["mic_weights"]),
        "speech_db": float(r.uniform(*a["speech_db_range"])),
        "turn_db_jitter": float(a["turn_db_jitter"]),
        "opus_kbps": int(weighted_choice(r, {str(k): v for k, v in a["opus_kbps_weights"].items()})),
        "echo": ({"gain_db": float(r.uniform(-35, -20)), "delay_frames": int(r.integers(1, 4))}
                 if r.random() < a["echo_prob"] else None),
        "clip_boost_db": float(r.uniform(6, 12)) if r.random() < a["clipping_prob"] else 0.0,
    }


def timing_profile(pools, r, scenario, start_policy):
    tm = pools.cfg["timing"]
    gap = {"rapid": tm["gap_rapid_s"], "pauses": tm["gap_pause_s"]}.get(scenario, tm["gap_normal_s"])
    return {
        "start_policy": start_policy,
        "user_first_at_s": float(r.uniform(*tm["user_first_s"])),
        "late_start_at_s": float(r.uniform(*tm["late_start_s"])),
        "greeting_wait_s": float(tm["greeting_wait_s"]),
        "gap_s": gap,
        "end_idle_frames": int(r.integers(tm["end_idle_frames"][0], tm["end_idle_frames"][1] + 1)),
        "no_response_s": float(tm["no_response_s"]),
        "max_monologue_s": float(r.uniform(*tm["max_monologue_s"])),
        "bc": ({"every_s": tm["bc_every_s_backchannel"], "prob": 0.85, "min_agent_s": 1.2}
               if scenario == "backchannel" else
               ({"every_s": tm["bc_every_s_other"], "prob": 0.3, "min_agent_s": 2.0}
                if scenario not in ("rapid", "distractor") else None)),
    }


def utt_id(text, spk, rate):
    return hashlib.sha1(f"{text}|{spk}|{rate:.3f}".encode("utf-8")).hexdigest()[:20]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("plan", paths.logs / "workers" / "plan.log")
    stage_log(paths, "plan", "running")
    pools = Pools(cfg, paths)
    seed = cfg["seed"]
    F_MAIN, F_LONG = cfg["frames"]["main"], cfg["frames"]["long"]
    total_s = cfg["target_conv_hours"] * 3600 * (1 + cfg["overprovision"])
    long_s = total_s * cfg["frames"]["long_hours_frac"]
    main_s = total_s - long_s
    sv, st = cfg["splits"]["val"], cfg["splits"]["test"]
    quotas = {"train": main_s * (1 - sv - st), "val": main_s * sv, "test": main_s * st,
              "train_long": long_s * 0.95, "val_long": long_s * 0.05}
    frames_of = {"train": F_MAIN, "val": F_MAIN, "test": F_MAIN, "test_deploy": F_MAIN,
                 "train_long": F_LONG, "val_long": F_LONG}

    # Refuse to replan while generation workers hold fresh locks (their shard files would vanish under them).
    for st_name in ("tts", "selfplay", "retry"):
        for lk in (paths.locks / st_name).glob("*.lock") if (paths.locks / st_name).exists() else []:
            if time.time() - lk.stat().st_mtime < cfg["gen"]["stale_lock_s"]:
                raise RuntimeError(f"{st_name} workers are running ({lk.name}); stop them (stop_stage) before replanning")

    samples, tts_jobs = [], {}
    acc = Counter()
    fallbacks = Counter()

    def make(split, k):
        # Every sample depends ONLY on (seed, split, k): a plan for 200 h is an exact prefix of the plan for 500 h,
        # so changing TARGET_HOURS keeps every already-generated sample valid (no regeneration, no mismatch).
        i = k
        r = rng_for(seed, "sample", split, k)
        psplit = pool_split(split)
        scenario = weighted_choice(r, cfg["scenario_weights"])
        real_spk = None
        if scenario == "casual_real" or r.random() < cfg["real_speech"]["real_speaker_sample_frac"]:
            cands = pools.real_spk_by.get(psplit) or []
            real_spk = cands[int(r.integers(0, len(cands)))] if cands else None
        if scenario == "casual_real" and real_spk is None:
            scenario = "multi_turn"
        category = "casual" if scenario == "casual_real" else weighted_choice(r, cfg["persona_category_weights"])
        plist = pools.p_by.get((psplit, category))
        if not plist:
            cands = [p for (sp, _), ps in pools.p_by.items() if sp == psplit for p in ps]
            if not cands:
                raise RuntimeError(f"planner: no personas available for split {psplit}")
            fallbacks[f"persona_{category}_{psplit}->any_{psplit}"] += 1
            plist = cands
        persona = pick(r, plist)
        n_tok = persona["n_tokens"]
        frames = frames_of[split]
        conv_frames = frames - (PREFIX_BASE + n_tok)
        # agent voice
        if split == "test_deploy":
            name = pools.ui_names[int(r.integers(0, len(pools.ui_names)))]
            clip = next(v for v in pools.ui[name] if v["deploy"])
        elif r.random() < cfg["agent_voices"]["ui_frac"] or not pools.extra.get(psplit):
            name = pools.ui_names[int(r.integers(0, len(pools.ui_names)))]
            clips = pools.ui[name]
            if psplit == "train":
                dep = [v for v in clips if v["deploy"]]
                oth = [v for v in clips if v["split"] == "train"]
                clip = dep[0] if (r.random() < cfg["agent_voices"]["deploy_clip_weight"] or not oth) else pick(r, oth)
            else:
                ho = [v for v in clips if v["split"] in ("heldout", "all")]
                clip = pick(r, ho)
        else:
            clip = pick(r, pools.extra[psplit])
        # user speaker (TTS) -- always assigned; real speaker optional
        if not pools.spk_by.get(psplit):
            raise RuntimeError(f"planner: no TTS speakers available for split {psplit}")
        spk = pick(r, pools.spk_by[psplit])
        start_policy = weighted_choice(r, cfg["start_policy_weights"])
        n_turns = int(math.ceil(conv_frames / FRAME_RATE / SEC_PER_TURN[scenario])) + 3
        turns = build_turns(pools, r, None, persona, scenario, psplit, n_turns, real_spk)
        # TTS jobs for all tts turns + nudges + backchannels
        def job(text, jitter=True):
            # turns get a per-utterance rate jitter; backchannels/nudges reuse the speaker's base rate so the
            # same (speaker, phrase) clip is rendered once and shared across samples
            rate = round(float(spk["rate"] * (r.uniform(0.95, 1.05) if jitter else 1.0)), 3)
            uid = utt_id(text, spk["speaker_id"], rate)
            tts_jobs.setdefault(uid, {"utt_id": uid, "text": text, "speaker_id": spk["speaker_id"],
                                      "voice": spk["voice"], "lang": spk["lang"], "speed": rate})
            return uid
        for t in turns:
            if t["source"] == "tts":
                t["utt_id"] = job(t["text"])
        nudges = [{"text": x, "utt_id": job(x, False)} for x in NUDGES[:2]]
        bcs = []
        tm = timing_profile(pools, r, scenario, start_policy)
        if tm["bc"] is not None:
            real_bc = pools.bc_by_spk.get(real_spk) if (real_spk and scenario == "casual_real") else None
            if real_bc:
                bcs = [{"source": "real", "path": b["path"], "text": b["text"]} for b in real_bc[:10]]
            else:
                phrases = list(r.choice(BACKCHANNELS, size=6, replace=False))
                bcs = [{"source": "tts", "text": p, "utt_id": job(p, False)} for p in phrases]
        s = {
            "sample_id": f"ppx_{split}_{i:06d}", "index": i, "split": split, "frames": frames, "scenario": scenario,
            "persona_id": persona["persona_id"], "persona_category": persona["category"],
            "persona_text": persona["text"], "persona_tokens": persona["tokens"], "n_persona_tokens": n_tok,
            "prefix_len": PREFIX_BASE + n_tok, "conv_frames": conv_frames,
            "agent_clip_id": clip["clip_id"], "agent_voice_id": clip["voice_id"], "agent_voice_kind": clip["kind"],
            "agent_voice_file": clip["path"], "agent_voice_codes": clip["codes"],
            "user_speaker": spk["speaker_id"], "user_speaker_accent": spk["accent"], "user_speaker_gender": spk["gender"],
            "real_speaker": real_spk, "turns": turns, "nudges": nudges, "backchannels": bcs,
            "acoustics": acoustic_profile(pools, r, psplit, scenario), "timing": tm,
            "seed": stable_int(seed, "gen", split, i) % (2 ** 31),
        }
        acc[split] += conv_frames / FRAME_RATE
        return s

    # test_deploy: fixed count; other splits: samples k = 0, 1, 2, ... until the conversation-seconds quota is met
    for k in range(cfg["splits"]["test_deploy_samples"]):
        samples.append(make("test_deploy", k))
    for split in ["val", "test", "val_long", "train_long", "train"]:
        k = 0
        while acc[split] < quotas[split]:
            samples.append(make(split, k))
            k += 1
        log.info("planned %s: %d samples, %.1f h", split, k, acc[split] / 3600)
    if fallbacks:
        log.warning("held-out pools were empty for some picks (fell back to train entities): %s", dict(fallbacks))

    # ---- group by identical prompt schedule (frames, n_tokens) and shard
    gb = cfg["gen"]["group_b"]
    by_key = defaultdict(list)
    for s in samples:
        by_key[(s["frames"], s["n_persona_tokens"])].append(s)
    groups = []
    for key in sorted(by_key):
        lst = sorted(by_key[key], key=lambda s: s["sample_id"])
        for k in range(0, len(lst), gb):
            groups.append(lst[k:k + gb])
    rr = np.random.default_rng(seed)
    perm = rr.permutation(len(groups))
    groups = [groups[k] for k in perm]
    # eval splits first so val/test exist early in a partial run
    groups.sort(key=lambda g: 0 if any(s["split"] != "train" and s["split"] != "train_long" for s in g) else 1)
    # plan id: unit names are unique per plan, so done-markers of an older plan can never be mistaken for this one
    plan_id = hashlib.sha1((cfg.get("config_hash", "") + "|" + ",".join(sorted(s["sample_id"] for s in samples)))
                           .encode()).hexdigest()[:8]
    sg = cfg["gen"]["shard_groups"]
    for sub in ("shards", "tts_jobs", "retry"):
        (paths.plan / sub).mkdir(parents=True, exist_ok=True)
        for f in (paths.plan / sub).glob("*"):
            f.unlink()          # previous plan's work-unit files (generated data and results are NOT touched)
    n_shards = 0
    for k in range(0, len(groups), sg):
        name = f"shard_{plan_id}_{k // sg:05d}"
        atomic_write_json(paths.plan / "shards" / f"{name}.json", {"shard": name, "plan_id": plan_id,
                                                                   "groups": groups[k:k + sg]})
        n_shards += 1
    jobs = sorted(tts_jobs.values(), key=lambda j: j["utt_id"])
    for k in range(0, len(jobs), cfg["tts"]["jobs_per_shard"]):
        write_jsonl(paths.plan / "tts_jobs" / f"tts_{plan_id}_{k // cfg['tts']['jobs_per_shard']:05d}.jsonl",
                    jobs[k:k + cfg["tts"]["jobs_per_shard"]])
    write_jsonl(paths.plan / "samples_index.jsonl",
                [{k: s[k] for k in ("sample_id", "split", "frames", "scenario", "persona_category", "n_persona_tokens",
                                    "conv_frames", "agent_voice_kind", "agent_voice_id", "user_speaker")} for s in samples])

    def hours(filter_key):
        c = Counter()
        for s in samples:
            c[filter_key(s)] += s["conv_frames"] / FRAME_RATE / 3600
        return {str(k): round(v, 2) for k, v in sorted(c.items(), key=lambda kv: -kv[1])}
    summary = {
        "plan_id": plan_id, "target_conv_hours": cfg["target_conv_hours"], "config_hash": cfg.get("config_hash"),
        "smoke_test": bool(cfg.get("smoke_test")), "pool_fallbacks": dict(fallbacks),
        "samples_by_split": dict(Counter(s["split"] for s in samples)),
        "samples": len(samples), "groups": len(groups), "shards": n_shards, "tts_jobs": len(jobs),
        "conv_hours_total": round(sum(s["conv_frames"] for s in samples) / FRAME_RATE / 3600, 2),
        "conv_hours_by_split": hours(lambda s: s["split"]),
        "conv_hours_by_scenario": hours(lambda s: s["scenario"]),
        "conv_hours_by_persona_category": hours(lambda s: s["persona_category"]),
        "conv_hours_by_agent_voice_kind": hours(lambda s: s["agent_voice_kind"]),
        "conv_hours_by_noise_class": hours(lambda s: s["acoustics"]["bg"]["class"]),
        "conv_hours_by_mic": hours(lambda s: s["acoustics"]["mic"]),
        "conv_hours_by_start_policy": hours(lambda s: s["timing"]["start_policy"]),
        "samples_by_ui_voice": dict(Counter(s["agent_voice_id"] for s in samples if s["agent_voice_kind"] == "ui")),
        "distinct_agent_voices": len({s["agent_voice_id"] for s in samples}),
        "distinct_user_tts_speakers": len({s["user_speaker"] for s in samples}),
        "distinct_real_speakers": len({s["real_speaker"] for s in samples if s["real_speaker"]}),
        "distinct_personas": len({s["persona_id"] for s in samples}),
        "user_turns_total": sum(len(s["turns"]) for s in samples),
        "user_turns_real": sum(t["source"] == "real" for s in samples for t in s["turns"]),
        "turn_content": dict(Counter(t["content"] for s in samples for t in s["turns"]).most_common(30)),
        "distinct_question_ids": len({t["ref"] for s in samples for t in s["turns"] if t.get("content") == "question"}),
        "group_sizes": dict(Counter(len(g) for g in groups)),
        "echo_samples": sum(s["acoustics"]["echo"] is not None for s in samples),
        "rir_samples": sum(s["acoustics"]["rir"] is not None for s in samples),
    }
    atomic_write_json(paths.logs / "plan_summary.json", summary)
    log.info("plan summary: %s", json.dumps(summary)[:4000])
    stage_log(paths, "plan", "done", samples=len(samples), shards=n_shards, tts_jobs=len(jobs),
              conv_hours=summary["conv_hours_total"], plan_id=plan_id, target_conv_hours=cfg["target_conv_hours"])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/selfplay.py
"""Stages `selfplay` and `retry` (GPU): teacher self-play -> training token sequences.

Exactly mirrors inference (moshi/server.py + LMGen):
  1. PREFIX, stepped through LMGen like LMGen.step_system_prompts does:
       125 voice-prompt frames   moshi_tokens=Mimi(voice clip)  text=3  input=SINE_TOKENS
       6 spacer frames           moshi_tokens=SILENCE_TOKENS    text=3  input=SINE_TOKENS
       T persona frames          moshi_tokens=SILENCE_TOKENS    text=persona token t  input=SINE_TOKENS
       6 spacer frames           moshi_tokens=SILENCE_TOKENS    text=3  input=SINE_TOKENS
  2. mimi.reset_streaming() (server does this after the prompts), then the CONVERSATION: every 80 ms the user's
     microphone waveform (speech + room tone/noise + events + echo, Opus round-tripped) is Mimi-encoded in
     streaming mode (exactly server.opus_loop) and fed as `input_tokens`; the teacher samples text + agent audio
     (temp 0.8 / 0.7, top-k 250 / 25 = LMGen defaults used by the server).
  3. Every non-None LMGen.step output is recorded. LMGen overwrites call 0 with the initial token and returns
     frame f at call f+1 (max_delay=1), so recorded codes[:, j] == what was provided/sampled at call j+1 and
     the training code (forward_train prepends the initial token) sees exactly what inference sees.
     prefix_len = 136 + T, total recorded frames N requires N + 2 calls.
The user simulator reacts to the teacher's live output (turn ends, barge-in, backchannels), so turn-taking
timing is realistic rather than scripted.
"""
import argparse
import hashlib
import json
import math
import os
import threading
import time
import traceback
from collections import deque, Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

from .common import (P, SR, FRAME, FRAME_RATE, N_CODEBOOKS, AUDIO_CARD, TEXT_CARD, TEXT_PAD, TEXT_EPAD,
                     SILENCE_TOKENS, SINE_TOKENS, VOICE_FRAMES, SILENCE_SPACER, PREFIX_BASE, prefix_calls,
                     load_cfg, get_logger, stage_log, read_jsonl, write_jsonl, atomic_write_json, atomic_torch_save,
                     append_jsonl, Claimer, Progress, worker_tag, log_error, rng_for, load_24k, rms_db, gain_to,
                     apply_rir, apply_mic, opus_roundtrip, synth_noise, synth_rir, loop_to, write_flac, stable_int)


# ============================================================================ user-side audio preparation (CPU)
def _speech_chain(x, ac, rir, rng):
    x = gain_to(x, ac["speech_db"] + float(rng.uniform(-ac["turn_db_jitter"], ac["turn_db_jitter"])))
    x = apply_rir(x, rir)
    x = apply_mic(x, ac["mic"], rng)
    if ac["clip_boost_db"]:
        x = np.clip(x * 10 ** (ac["clip_boost_db"] / 20), -1, 1)
    return opus_roundtrip(x, ac["opus_kbps"])


def prepare_sample(s, paths, attempt):
    """All user-side audio for one sample, processed through the sample's own room/mic/codec chain."""
    rng = rng_for(s["seed"], "prep", attempt)
    ac = s["acoustics"]
    rir = synth_rir(rng, ac["rir"]["rt60"], ac["rir"]["drr_db"]) if ac["rir"] else None

    def load(ref):
        if ref.get("source") == "real":
            return load_24k(paths.root / ref["path"])
        p = paths.tts_audio / ref["utt_id"][:2] / f"{ref['utt_id']}.flac"
        if not p.exists():
            return None
        return load_24k(p)

    turns = []
    for t in s["turns"]:
        x = load(t)
        if x is None or len(x) < int(0.15 * SR):
            continue  # TTS failure (logged by the tts stage) -> the turn is skipped, not faked
        turns.append({**{k: t[k] for k in ("idx", "interrupt", "barge_s", "content", "source", "text")},
                      "audio": _speech_chain(x, ac, rir, rng)})
    nudges = [a for a in (load(n) for n in s["nudges"]) if a is not None]
    nudges = [_speech_chain(a, ac, rir, rng) for a in nudges]
    bcs = []
    for b in s["backchannels"]:
        x = load(b)
        if x is not None and len(x) > int(0.1 * SR):
            y = _speech_chain(x, ac, rir, rng)
            bcs.append(gain_to(y, rms_db(y) - float(rng.uniform(2, 8))))   # backchannels are softer
    n = (s["frames"] + 8) * FRAME
    bg = ac["bg"]
    if bg["type"] == "digital":
        track = (rng.standard_normal(n) * 10 ** (bg["level_db"] / 20)).astype(np.float32)
    else:
        if bg["type"] == "procedural":
            raw = synth_noise(bg["kind"], n, rng)
        else:
            raw = np.zeros(n, np.float32)
            for f in bg["files"]:
                raw += loop_to(load_24k(f), n, rng)
        target = bg["abs_db"] if "abs_db" in bg else ac["speech_db"] - bg["snr_db"]
        track = gain_to(raw, target)
    # non-speech events (door, keyboard, cough, TV...) at random times over the conversation
    conv_start = s["prefix_len"] * FRAME
    ev_log = []
    for ev in ac["events"]:
        if "path" in ev:
            x = load_24k(ev["path"])
            L = min(len(x), int(ev["dur_s"] * SR))
            st = int(rng.integers(0, max(1, len(x) - L + 1)))
            x = x[st:st + L]
        else:
            x = synth_noise(ev.get("kind", "white"), int(ev["dur_s"] * SR), rng)
        x = gain_to(x, ev["level_db"])
        at = int(rng.integers(0, max(1, n - conv_start - len(x))))
        track[at:at + len(x)] += x[: max(0, n - at)]
        ev_log.append({"at_frame": at // FRAME, "dur_frames": len(x) // FRAME})
    track = apply_mic(track, ac["mic"], rng)
    track = opus_roundtrip(track, ac["opus_kbps"])
    return {"turns": turns, "nudges": nudges, "bcs": bcs, "bg": track, "events": ev_log}


# ============================================================================ reactive user simulator
class UserSim:
    """Produces the user's microphone signal frame by frame and reacts to the agent's live output.

    Conversation frame t is fed at conversation call t. The agent observation received after call t
    describes agent frame t-1 (one-frame model latency), exactly like in the live server.
    """

    AGENT_VOICED_DB = -42.0

    def __init__(self, s, A, rng):
        self.s, self.rng = s, rng
        self.tm = s["timing"]
        self.ac = s["acoustics"]
        self.bg = A["bg"]
        self.turns = deque(A["turns"])
        self.nudges = deque(A["nudges"])
        self.bcs = A["bcs"]
        self.active = []                    # [audio, pos, tag]
        self.user_busy_until = -1
        self.state = "start"
        self.next_start = None
        self.await_since = 0
        self.agent_since_user = False
        self.nudged = False
        self.greeted = False
        self.agent_run = 0
        self.agent_idle = 10 ** 6
        self.frames_since_text = 10 ** 6
        self.agent_is_on = False
        self.agent_pcm = deque(maxlen=8)
        bc = self.tm["bc"]
        self.next_bc = int(self._u(*bc["every_s"]) * FRAME_RATE) if bc else None
        self.events = []
        self.turn_log = []
        self.user_frames_voiced = 0
        pol = self.tm["start_policy"]
        if pol == "user_first":
            self.next_start = int(self.tm["user_first_at_s"] * FRAME_RATE)
        elif pol == "late_start":
            self.next_start = int(self.tm["late_start_at_s"] * FRAME_RATE)
        else:
            self.state = "wait_greeting"

    def _u(self, lo, hi):
        return float(self.rng.uniform(lo, hi))

    def _gap_frames(self):
        lo, hi = self.tm["gap_s"]
        if self.s["scenario"] in ("rapid", "pauses"):
            g = self._u(lo, hi)
        else:  # human response gaps are right-skewed: lognormal around ~0.5 s, clipped to [lo, hi]
            g = float(np.clip(self.rng.lognormal(math.log(0.5), 0.5), lo, hi))
        return max(1, int(round(g * FRAME_RATE)))

    def _start_turn(self, t, audio, tag, meta=None):
        dur = int(math.ceil(len(audio) / FRAME))
        self.active.append([audio, 0, tag])
        self.user_busy_until = t + dur
        self.state = "user"
        self.events.append(("user_on", t, tag))
        self.events.append(("user_off", t + dur, tag))
        rec = {"t": t, "dur_frames": dur, "tag": tag, "agent_active_at_start": self.agent_is_on}
        if meta:
            rec.update(meta)
        self.turn_log.append(rec)

    def _pop_turn(self, t, reason):
        if not self.turns:
            return False
        tr = self.turns.popleft()
        tag = "interrupt" if (reason == "barge" or tr["interrupt"] and self.agent_is_on) else "turn"
        if reason == "barge":
            self.events.append(("barge", t, tag))
        self._start_turn(t, tr["audio"], tag, {"idx": tr["idx"], "content": tr["content"], "source": tr["source"],
                                               "reason": reason})
        return True

    def step(self, t):
        busy = t < self.user_busy_until
        if not busy:
            if self.state == "user":
                self.state = "await"
                self.await_since = t
                self.agent_since_user = False
            if self.state == "wait_greeting":
                if self.agent_is_on:
                    self.greeted = True
                if self.agent_run >= int(self.tm["max_monologue_s"] * FRAME_RATE):
                    self._pop_turn(t, "barge")            # agent never pauses after its greeting -> user cuts in
                elif self.greeted and self.agent_idle >= self.tm["end_idle_frames"] and self.next_start is None:
                    self.next_start = t + self._gap_frames()
                elif not self.greeted and t >= int(self.tm["greeting_wait_s"] * FRAME_RATE) and self.next_start is None:
                    self.next_start = t
            elif self.state == "await":
                if self.agent_is_on:
                    self.agent_since_user = True
                nxt = self.turns[0] if self.turns else None
                if nxt is not None and nxt["interrupt"] and self.agent_is_on and nxt["barge_s"] is not None \
                        and self.agent_run >= int(nxt["barge_s"] * FRAME_RATE):
                    self._pop_turn(t, "barge")
                elif self.agent_run >= int(self.tm["max_monologue_s"] * FRAME_RATE):
                    self._pop_turn(t, "barge")
                elif self.agent_since_user and self.agent_idle >= self.tm["end_idle_frames"] and self.next_start is None:
                    self.next_start = t + self._gap_frames()
                elif not self.agent_since_user and t - self.await_since >= int(self.tm["no_response_s"] * FRAME_RATE) \
                        and self.next_start is None:
                    if not self.nudged and self.nudges:
                        self.nudged = True
                        self._start_turn(t, self.nudges.popleft(), "nudge")
                    else:
                        self.next_start = t
            if self.state != "user" and self.next_start is not None and t >= self.next_start:
                self.next_start = None
                if not self._pop_turn(t, "scheduled"):
                    self.state = "done"
        # backchannels while the agent talks and the user is not taking a turn
        bc = self.tm["bc"]
        if bc and self.bcs and not (t < self.user_busy_until) and self.next_bc is not None and t >= self.next_bc:
            if self.agent_is_on and self.agent_run >= int(bc["min_agent_s"] * FRAME_RATE) and self.rng.random() < bc["prob"]:
                a = self.bcs[int(self.rng.integers(0, len(self.bcs)))]
                self.active.append([a, 0, "bc"])
                self.events.append(("bc", t, "bc"))
            self.next_bc = t + int(self._u(*bc["every_s"]) * FRAME_RATE)
        # mix
        out = self.bg[t * FRAME:(t + 1) * FRAME].astype(np.float32).copy()
        if len(out) < FRAME:
            out = np.concatenate([out, np.zeros(FRAME - len(out), np.float32)])
        still = []
        for a in self.active:
            seg = a[0][a[1]:a[1] + FRAME]
            out[: len(seg)] += seg
            a[1] += FRAME
            if a[1] < len(a[0]):
                still.append(a)
        self.active = still
        if any(a[2] != "bc" for a in self.active) or t < self.user_busy_until:
            self.user_frames_voiced += 1
        echo = self.ac["echo"]
        if echo and len(self.agent_pcm) >= echo["delay_frames"]:
            out += self.agent_pcm[-echo["delay_frames"]] * (10 ** (echo["gain_db"] / 20))
        peak = np.max(np.abs(out))
        if peak > 1.0:
            out = np.tanh(out)  # soft clip like an overdriven input stage
        return out

    def observe(self, t_agent, text_tok, pcm):
        t_agent = max(0, t_agent)   # the first conversation call returns the last prefix frame
        voiced = rms_db(pcm) > self.AGENT_VOICED_DB
        self.frames_since_text = 0 if text_tok not in (TEXT_PAD, TEXT_EPAD) else self.frames_since_text + 1
        on = voiced or self.frames_since_text <= 3
        if on and not self.agent_is_on:
            self.events.append(("agent_on", t_agent, "agent"))
        if not on and self.agent_is_on:
            self.events.append(("agent_off", t_agent, "agent"))
        self.agent_is_on = on
        self.agent_run = self.agent_run + 1 if on else 0
        self.agent_idle = 0 if on else self.agent_idle + 1
        self.agent_pcm.append(pcm.astype(np.float32))


# ============================================================================ checks
def text_words(tok_pieces):
    return "".join(tok_pieces).replace("▁", " ").split()


def degenerate(words):
    if len(words) < 30:
        return False, 0.0
    grams = Counter(tuple(words[i:i + 4]) for i in range(len(words) - 3))
    top = grams.most_common(1)[0][1] * 4 / len(words)
    uniq = len(set(words)) / len(words)
    return (top > 0.4 or uniq < 0.12), round(top, 3)


def check_sample(codes, s, voice_codes, fed_user, sim, sp):
    """Hard checks (any failure -> retry/reject) and soft stats. codes: LongTensor [17, N] on CPU."""
    import torch
    issues, stats = [], {}
    N, T, pl = s["frames"], s["n_persona_tokens"], s["prefix_len"]
    if tuple(codes.shape) != (N_CODEBOOKS, N):
        return [f"shape {tuple(codes.shape)} != {(N_CODEBOOKS, N)}"], stats
    c = codes
    if int(c[0].min()) < 0 or int(c[0].max()) >= TEXT_CARD:
        issues.append("text id out of range")
    if int(c[1:].min()) < 0 or int(c[1:].max()) >= AUDIO_CARD:
        issues.append("audio id out of range")
    sil = torch.tensor(SILENCE_TOKENS).view(8, 1)
    sine = torch.tensor(SINE_TOKENS).view(8, 1)
    # recorded j <-> call j+1
    if not torch.equal(c[1:9, :VOICE_FRAMES - 1], voice_codes[:, 1:VOICE_FRAMES]):
        issues.append("prefix: voice codes mismatch")
    if not torch.equal(c[9:17, :pl], sine.expand(8, pl)):
        issues.append("prefix: user channel is not SINE")
    a0, a1 = VOICE_FRAMES - 1, VOICE_FRAMES - 1 + SILENCE_SPACER
    if not torch.equal(c[1:9, a0:pl], sil.expand(8, pl - a0)):
        issues.append("prefix: agent silence mismatch")
    txt = torch.tensor(s["persona_tokens"], dtype=torch.long)
    if not torch.equal(c[0, a1:a1 + T], txt):
        issues.append("prefix: persona tokens mismatch")
    if not (torch.all(c[0, :a1] == TEXT_PAD) and torch.all(c[0, a1 + T:pl] == TEXT_PAD)):
        issues.append("prefix: text pad mismatch")
    conv = c[:, pl:]
    if not torch.equal(conv[9:17], fed_user[:, : N - pl]):
        issues.append("conversation: recorded user codes != fed user codes")
    sine_cols = (conv[9:17] == sine).all(dim=0).sum().item()
    if sine_cols:
        issues.append(f"conversation: {sine_cols} SINE frames in user channel")
    words = text_words([sp.id_to_piece(int(x)) for x in conv[0] if int(x) not in (TEXT_PAD, TEXT_EPAD)])
    text_ratio = float(((conv[0] != TEXT_PAD) & (conv[0] != TEXT_EPAD)).float().mean())
    deg, top = degenerate(words)
    agent_on_frames = 0
    on = False
    last = 0
    for ev, tt, _ in sorted(sim.events, key=lambda e: e[1]):
        if ev == "agent_on":
            on, last = True, tt
        elif ev == "agent_off" and on:
            agent_on_frames += tt - last
            on = False
    if on:
        agent_on_frames += (N - pl) - last
    agent_ratio = agent_on_frames / max(1, N - pl)
    user_ratio = sim.user_frames_voiced / max(1, N - pl)
    ends = [tt for ev, tt, tag in sim.events if ev == "user_off" and tag in ("turn", "interrupt")]
    ons = sorted(tt for ev, tt, _ in sim.events if ev == "agent_on")
    responded = sum(any(e <= o <= e + 50 for o in ons) for e in ends if e < (N - pl) - 50)
    n_resp_eval = sum(1 for e in ends if e < (N - pl) - 50)
    stats.update(agent_words=len(words), agent_text_ratio=round(text_ratio, 3), agent_speech_ratio=round(agent_ratio, 3),
                 user_speech_ratio=round(user_ratio, 3), top4gram_frac=top, user_turns=len(ends),
                 responded_turns=responded, response_rate=round(responded / n_resp_eval, 3) if n_resp_eval else None,
                 n_barge=sum(ev == "barge" for ev, _, _ in sim.events), n_bc=sum(ev == "bc" for ev, _, _ in sim.events),
                 n_nudge=sum(tag == "nudge" for ev, _, tag in sim.events if ev == "user_on"))
    if deg:
        issues.append(f"agent text degenerate (top 4-gram frac {top})")
    if text_ratio < 0.02 or agent_ratio < 0.04:
        issues.append(f"agent (almost) silent: text_ratio={text_ratio:.3f} speech_ratio={agent_ratio:.3f}")
    if agent_ratio > 0.97:
        issues.append(f"agent never stops talking: speech_ratio={agent_ratio:.3f}")
    if len(ends) == 0 and s["scenario"] != "distractor":
        issues.append("no user turn happened")
    return issues, stats


# ============================================================================ generation
class Generator:
    def __init__(self, cfg, paths, log):
        import torch
        import sentencepiece
        from moshi.models import loaders, LMGen
        from moshi.modules.transformer import set_attention_context
        self.torch = torch
        self.cfg, self.paths, self.log = cfg, paths, log
        t = cfg["teacher"]
        self.dev = "cuda"
        torch.backends.cuda.matmul.allow_tf32 = True
        self.sp = sentencepiece.SentencePieceProcessor(t["tokenizer"])
        self.mimi_enc = loaders.get_mimi(t["mimi_weight"], self.dev)
        self.mimi_dec = loaders.get_mimi(t["mimi_weight"], self.dev)
        self.lm = loaders.get_moshi_lm(t["moshi_weight"], device=self.dev)
        self.lm.eval()
        # Temporal-transformer KV ring buffer: identical results as long as every sequence fits (it does: <= N+2
        # calls), but ~half the memory of the default 3000 -> larger batches. Depformer context untouched.
        ctx = max(cfg["frames"]["main"], cfg["frames"]["long"]) + 16
        set_attention_context(self.lm.transformer, ctx)
        g = cfg["gen"]
        self.lm_gen = LMGen(self.lm, device=self.dev, use_sampling=True, temp=g["temp"], temp_text=g["temp_text"],
                            top_k=g["top_k"], top_k_text=g["top_k_text"],
                            audio_silence_frame_cnt=SILENCE_SPACER, sample_rate=SR, frame_rate=FRAME_RATE)
        self.max_b = g["group_b"] if not g.get("max_batch") else g["max_batch"]
        self.voice_cache = {}

    def voice_codes(self, rel):
        if rel not in self.voice_cache:
            self.voice_cache[rel] = self.torch.load(self.paths.root / rel, weights_only=True).long()
        return self.voice_cache[rel]

    def run_batch(self, samples, prepared, attempt):
        torch = self.torch
        B = len(samples)
        T = samples[0]["n_persona_tokens"]
        N = samples[0]["frames"]
        assert all(s["n_persona_tokens"] == T and s["frames"] == N for s in samples)
        pc = prefix_calls(T)
        dev = self.dev
        torch.manual_seed(stable_int(*[s["sample_id"] for s in samples], attempt) % (2 ** 63))
        vc = torch.stack([self.voice_codes(s["agent_voice_codes"]) for s in samples]).to(dev)          # [B,8,125]
        ptok = torch.tensor([s["persona_tokens"] for s in samples], dtype=torch.long, device=dev)      # [B,T]
        sine = torch.tensor(SINE_TOKENS, dtype=torch.long, device=dev).view(1, 8, 1).expand(B, 8, 1).contiguous()
        sil = torch.tensor(SILENCE_TOKENS, dtype=torch.long, device=dev).view(1, 8, 1).expand(B, 8, 1).contiguous()
        pad = torch.full((B,), TEXT_PAD, dtype=torch.long, device=dev)
        sims = [UserSim(s, A, rng_for(s["seed"], "sim", attempt)) for s, A in zip(samples, prepared)]
        n_conv_calls = N + 2 - pc
        fed = torch.zeros((B, 8, n_conv_calls), dtype=torch.long)
        rec = []
        prev_user = [[] for _ in range(B)]
        prev_agent = [[] for _ in range(B)]
        keep_audio = [s.get("_preview", False) for s in samples]
        with torch.no_grad(), self.lm_gen.streaming(B), self.mimi_enc.streaming(B), self.mimi_dec.streaming(B):
            for c in range(pc):
                if c < VOICE_FRAMES:
                    moshi, text = vc[:, :, c:c + 1], pad
                elif c < VOICE_FRAMES + SILENCE_SPACER:
                    moshi, text = sil, pad
                elif c < VOICE_FRAMES + SILENCE_SPACER + T:
                    moshi, text = sil, ptok[:, c - VOICE_FRAMES - SILENCE_SPACER]
                else:
                    moshi, text = sil, pad
                out = self.lm_gen.step(input_tokens=sine, moshi_tokens=moshi, text_token=text)
                if out is not None:
                    rec.append(out[:, :, 0])
            self.mimi_enc.reset_streaming()
            self.mimi_dec.reset_streaming()
            for t in range(n_conv_calls):
                frames = np.stack([sim.step(t) for sim in sims]).astype(np.float32)
                x = torch.from_numpy(frames).to(dev)[:, None, :]
                ucodes = self.mimi_enc.encode(x)                                                    # [B,8,1]
                fed[:, :, t] = ucodes[:, :, 0].cpu()
                out = self.lm_gen.step(input_tokens=ucodes)
                rec.append(out[:, :, 0])
                pcm = self.mimi_dec.decode(out[:, 1:9])[:, 0].float().cpu().numpy()                 # [B,1920]
                txt = out[:, 0, 0].cpu().numpy()
                for b in range(B):
                    sims[b].observe(t - 1, int(txt[b]), pcm[b])
                    if keep_audio[b]:
                        prev_user[b].append(frames[b])
                        prev_agent[b].append(pcm[b])
        codes = torch.stack(rec, dim=-1).cpu().long()                                               # [B,17,N]
        return codes, fed, sims, prev_user, prev_agent


def latest_status(paths):
    """sample_id -> newest result record over all plans/stages (higher attempt wins; 'ok' wins a tie)."""
    latest = {}
    for st in ("selfplay", "retry"):
        d = paths.results / st
        if not d.exists():
            continue
        for f in sorted(d.rglob("g*.json")):
            try:
                rows = read_json(f)["results"]
            except Exception:
                continue   # torn file from a killed worker -- that group is simply redone
            for r in rows:
                prev = latest.get(r["sample_id"])
                if prev is None or r["attempt"] > prev["attempt"] or \
                        (r["attempt"] == prev["attempt"] and r["status"] == "ok"):
                    latest[r["sample_id"]] = r
    return latest


def run_worker(args, cfg, paths, stage):
    import torch
    wt = worker_tag(args.worker_id)
    log = get_logger(f"{stage}.{wt}", paths.logs / "workers" / f"{stage}_{wt}.log")
    # Samples that already have a final result (possibly produced under an earlier plan, e.g. before TARGET_HOURS
    # was changed) are never regenerated. Only 'retry' samples stay eligible.
    finished = {sid for sid, r in latest_status(paths).items() if r["status"] in ("ok", "rejected")}
    log.info("%d samples already finished (skipped if they appear in this plan)", len(finished))
    gen = Generator(cfg, paths, log)
    cl = Claimer(paths, stage, wt, cfg["gen"]["stale_lock_s"])
    if stage == "selfplay":
        units = sorted(p.stem for p in (paths.plan / "shards").glob("shard_*.json"))
        load_unit = lambda u: read_json(paths.plan / "shards" / f"{u}.json")
    else:
        units = sorted(p.stem for p in (paths.plan / "retry").glob("retry_*.json"))
        load_unit = lambda u: read_json(paths.plan / "retry" / f"{u}.json")
    prog = Progress(paths, stage, wt, len(units))
    pool = ThreadPoolExecutor(max_workers=cfg["gen"]["prep_threads"])
    preview_frac = cfg["preview_audio_frac"]
    order = units[args.worker_id::args.num_workers] + units
    conv_s_total = 0.0
    t_start = time.time()
    for unit in order:
        if not cl.try_claim(unit):
            continue
        spec = load_unit(unit)
        n_before = sum(len(g) for g in spec["groups"])
        spec["groups"] = [[s for s in g if s["sample_id"] not in finished] for g in spec["groups"]]
        if sum(len(g) for g in spec["groups"]) < n_before:
            log.info("%s: %d of %d samples already finished earlier -> skipped", unit,
                     n_before - sum(len(g) for g in spec["groups"]), n_before)
        res_dir = paths.results / stage / unit
        res_dir.mkdir(parents=True, exist_ok=True)
        try:
            gi_todo = [gi for gi in range(len(spec["groups"])) if not (res_dir / f"g{gi:03d}.json").exists()]
            futures = {}

            def submit(gi):
                grp = spec["groups"][gi]
                att = spec.get("attempt", 0)
                for s in grp:
                    s["_preview"] = (s["split"] not in ("train", "train_long")) or \
                        (stable_int("preview", s["sample_id"]) % 10_000) / 10_000 < preview_frac
                return pool.submit(lambda: [prepare_sample(s, paths, att) for s in grp])
            for k, gi in enumerate(gi_todo[:2]):
                futures[gi] = submit(gi)
            for k, gi in enumerate(gi_todo):
                if k + 2 < len(gi_todo):
                    futures[gi_todo[k + 2]] = submit(gi_todo[k + 2])       # prefetch CPU audio prep
                grp = spec["groups"][gi]
                attempt = spec.get("attempt", 0)
                prepared = futures.pop(gi).result()
                t0 = time.time()
                results = []
                b = min(gen.max_b, len(grp))
                pos = 0
                while pos < len(grp):
                    sub, sub_prep = grp[pos:pos + b], prepared[pos:pos + b]
                    try:
                        torch.cuda.reset_peak_memory_stats()
                        codes, fed, sims, pu, pa = gen.run_batch(sub, sub_prep, attempt)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        if b == 1:
                            raise
                        b = max(1, b // 2)
                        gen.max_b = b
                        log.warning("OOM -> batch size %d", b)
                        continue
                    for i, s in enumerate(sub):
                        vc = gen.voice_codes(s["agent_voice_codes"])
                        issues, stats = check_sample(codes[i], s, vc, fed[i], sims[i], gen.sp)
                        status = "ok" if not issues else ("retry" if attempt < cfg["gen"]["max_retries"] else "rejected")
                        rec = {"sample_id": s["sample_id"], "split": s["split"], "status": status, "attempt": attempt,
                               "issues": issues, "stats": stats, "frames": s["frames"], "prefix_len": s["prefix_len"],
                               "conv_frames": s["conv_frames"], "worker": wt, "time": time.strftime("%Y-%m-%dT%H:%M:%S"),
                               "turn_log": sims[i].turn_log[:60], "events_count": dict(Counter(e[0] for e in sims[i].events)),
                               "bg_events": prepared[pos + i]["events"], "turns_available": len(prepared[pos + i]["turns"])}
                        if status == "ok":
                            path = paths.codes_path(s["split"], s["sample_id"])
                            atomic_torch_save(path, codes[i].clone())
                            rec["codes_sha1"] = hashlib.sha1(codes[i].numpy().tobytes()).hexdigest()
                            pl = s["prefix_len"]
                            trans = sorted({pl + max(0, tt) for ev, tt, _ in sims[i].events
                                            if ev in ("user_on", "user_off", "agent_on", "agent_off", "barge")
                                            and 0 <= pl + tt < s["frames"]})
                            rec["transition_frames"] = trans
                            if s.get("_preview"):
                                stereo = np.stack([np.concatenate(pu[i]), np.concatenate(pa[i])], axis=1)
                                write_flac(paths.preview / s["split"] / f"{s['sample_id']}.flac", stereo)
                                rec["preview"] = True
                            conv_s_total += s["conv_frames"] / FRAME_RATE
                        results.append(rec)
                    pos += len(sub)
                    cl.heartbeat(unit)
                dt = time.time() - t0
                for r in results:
                    append_jsonl(paths.logs / "samples" / f"{stage}_{wt}.jsonl",
                                 {k: v for k, v in r.items() if k != "turn_log"})
                atomic_write_json(res_dir / f"g{gi:03d}.json", {"group": gi, "seconds": round(dt, 1),
                                                                 "batch": b, "results": results})
                n_frames = sum(s["frames"] for s in grp)
                prog.update(items=len(grp), last_unit=unit, last_group_s=round(dt, 1),
                            frames_per_s=round(n_frames / max(dt, 1e-6), 1), batch_size=b,
                            peak_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                            conv_hours_ok_this_run=round(conv_s_total / 3600, 3),
                            conv_hours_per_hour=round(conv_s_total / max(time.time() - t_start, 1), 3),
                            ok=sum(r["status"] == "ok" for r in results),
                            not_ok=sum(r["status"] != "ok" for r in results))
                log.info("%s g%03d: %d samples in %.0fs (B=%d, %.0f frames/s) ok=%d", unit, gi, len(grp), dt, b,
                         n_frames / max(dt, 1e-6), sum(r["status"] == "ok" for r in results))
            cl.mark_done(unit, {"groups": len(spec["groups"])})
            prog.update(units=1)
        except Exception as e:
            log.exception("unit %s failed", unit)
            log_error(paths, stage, wt, e, unit=unit)
            cl.release(unit)
            torch.cuda.empty_cache()


def read_json(p):
    import json
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def build_retry(args, cfg, paths):
    """Collect `retry` results whose sample has no later ok/rejected record and re-plan them (attempt+1)."""
    import json
    log = get_logger("retry_plan", paths.logs / "workers" / "retry_plan.log")
    latest = latest_status(paths)
    need = {sid: r["attempt"] + 1 for sid, r in latest.items() if r["status"] == "retry"}
    # previous retry-unit definitions are obsolete (their results, if any, are already in results/): remove them so
    # an interrupted earlier round can't queue the same sample twice
    for f in (paths.plan / "retry").glob("retry_*.json") if (paths.plan / "retry").exists() else []:
        f.unlink()
    plans = {}
    for f in sorted((paths.plan / "shards").glob("shard_*.json")):
        for g in read_json(f)["groups"]:
            for s in g:
                if s["sample_id"] in need:
                    plans[s["sample_id"]] = s
    by = {}
    for sid, s in plans.items():
        by.setdefault((s["frames"], s["n_persona_tokens"], need[sid]), []).append(s)
    (paths.plan / "retry").mkdir(parents=True, exist_ok=True)
    # unit names are unique per plan AND per round -> a done-marker from an older plan/round never matches
    plan_id = json.loads((paths.logs / "plan_summary.json").read_text(encoding="utf-8")).get("plan_id", "p")
    stamp = time.strftime("%Y%m%d%H%M%S")
    k = 0
    for (fr, T, att), lst in sorted(by.items()):
        gb = cfg["gen"]["group_b"]
        groups = [lst[i:i + gb] for i in range(0, len(lst), gb)]
        name = f"retry_{plan_id}_{stamp}_{k:04d}"
        atomic_write_json(paths.plan / "retry" / f"{name}.json", {"shard": name, "attempt": att, "groups": groups})
        k += 1
    log.info("retry plan: %d samples in %d new units", len(plans), k)
    atomic_write_json(paths.logs / "retry_plan.json", {"samples": len(plans), "new_units": k, "time": stamp})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--stage", required=True, choices=["selfplay", "retry", "build_retry"])
    ap.add_argument("--worker-id", type=int, default=0)
    ap.add_argument("--num-workers", type=int, default=1)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    if args.stage == "build_retry":
        build_retry(args, cfg, paths)
    else:
        run_worker(args, cfg, paths, args.stage)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/assemble.py
"""Stage `assemble`: build the training-ready split directories from the self-play results.

Output per split (train/, val/, test/, test_deploy/, train_long/, val_long/) -- exactly the layout
distill/data/dataset.py:TeacherTokenDataset reads:
  <split>/manifest.jsonl      one JSON per sample with the fields the training code uses
                              (sample_id, voice_prompt, persona_text, scenario, num_frames, transition_frames)
                              plus extra metadata the current code ignores (prefix_len, persona tokens, ...)
  <split>/codes/<id>.pt       LongTensor [17, num_frames]  (row 0 text, rows 1-8 agent, rows 9-16 user)
meta/sample_meta_<split>.jsonl  full per-sample metadata (turn logs, acoustics, stats)
DATASET_CARD.md                 sources, licenses, statistics
"""
import argparse
import json
from collections import Counter, defaultdict

from .common import (P, FRAME_RATE, SPLITS, load_cfg, get_logger, stage_log, read_jsonl, write_jsonl,
                     atomic_write_json, atomic_write_bytes)


def read_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def latest_results(paths):
    latest = {}
    for stage in ("selfplay", "retry"):
        for f in sorted((paths.results / stage).rglob("g*.json")):
            for r in read_json(f)["results"]:
                prev = latest.get(r["sample_id"])
                if prev is None or r["attempt"] > prev["attempt"] or (r["attempt"] == prev["attempt"] and r["status"] == "ok"):
                    latest[r["sample_id"]] = r
    return latest


def load_plans(paths):
    plans = {}
    for f in sorted((paths.plan / "shards").glob("shard_*.json")):
        for g in read_json(f)["groups"]:
            for s in g:
                plans[s["sample_id"]] = s
    return plans


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("assemble", paths.logs / "workers" / "assemble.log")
    stage_log(paths, "assemble", "running")
    plans = load_plans(paths)
    latest = latest_results(paths)
    manifests = defaultdict(list)
    metas = defaultdict(list)
    status = Counter()
    missing = []
    # TARGET_HOURS can change between runs. Code files of samples that are not in the CURRENT plan are parked in
    # _work/excluded_codes/ (not deleted) and moved back automatically if a later, larger plan includes them again.
    import shutil
    parked = restored = 0
    excl = paths.work / "excluded_codes"
    for split in SPLITS:
        d = paths.split_dir(split) / "codes"
        if d.exists():
            for f in d.glob("*.pt"):
                if f.stem not in plans:
                    (excl / split).mkdir(parents=True, exist_ok=True)
                    shutil.move(str(f), str(excl / split / f.name))
                    parked += 1
        if (excl / split).exists():
            for f in (excl / split).glob("*.pt"):
                if f.stem in plans:
                    d.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(f), str(d / f.name))
                    restored += 1
    if parked or restored:
        log.info("TARGET_HOURS change: parked %d out-of-plan samples, restored %d", parked, restored)

    for sid, s in sorted(plans.items()):
        r = latest.get(sid)
        if r is None:
            status["not_generated"] += 1
            missing.append(sid)
            continue
        status[r["status"]] += 1
        if r["status"] != "ok":
            continue
        cp = paths.codes_path(s["split"], sid)
        if not cp.exists():
            status["ok_but_file_missing"] += 1
            missing.append(sid)
            continue
        man = {
            # ---- fields used by distill/data/dataset.py (TeacherTokenDataset / Sample) ----
            "sample_id": sid,
            "voice_prompt": s["agent_voice_file"].split("/")[-1],
            "persona_text": s["persona_text"],
            "scenario": s["scenario"],
            "num_frames": s["frames"],
            "transition_frames": r["transition_frames"],
            # ---- extra metadata (ignored by the current training code) ----
            "split": s["split"], "prefix_len": s["prefix_len"], "n_persona_tokens": s["n_persona_tokens"],
            "conv_frames": s["conv_frames"], "persona_id": s["persona_id"], "persona_category": s["persona_category"],
            "agent_voice_id": s["agent_voice_id"], "agent_clip_id": s["agent_clip_id"],
            "agent_voice_kind": s["agent_voice_kind"], "user_speaker": s["user_speaker"],
            "real_speaker": s["real_speaker"], "noise_class": s["acoustics"]["bg"]["class"],
            "mic": s["acoustics"]["mic"], "opus_kbps": s["acoustics"]["opus_kbps"],
            "echo": s["acoustics"]["echo"] is not None, "rir": s["acoustics"]["rir"] is not None,
            "start_policy": s["timing"]["start_policy"], "attempt": r["attempt"], "codes_sha1": r["codes_sha1"],
            "stats": r["stats"],
        }
        manifests[s["split"]].append(man)
        metas[s["split"]].append({**{k: v for k, v in s.items() if k not in ("persona_tokens",)}, "result": r})
    for split in SPLITS:
        rows = sorted(manifests.get(split, []), key=lambda m: m["sample_id"])
        (paths.split_dir(split) / "codes").mkdir(parents=True, exist_ok=True)
        write_jsonl(paths.split_dir(split) / "manifest.jsonl", rows)
        write_jsonl(paths.meta / f"sample_meta_{split}.jsonl", sorted(metas.get(split, []), key=lambda m: m["sample_id"]))
    write_jsonl(paths.logs / "assemble_missing.jsonl", [{"sample_id": m} for m in missing])

    def hours(rows, key=None):
        if key is None:
            return round(sum(m["conv_frames"] for m in rows) / FRAME_RATE / 3600, 2)
        c = Counter()
        for m in rows:
            c[str(m[key])] += m["conv_frames"] / FRAME_RATE / 3600
        return {k: round(v, 2) for k, v in sorted(c.items(), key=lambda kv: -kv[1])}
    allrows = [m for v in manifests.values() for m in v]
    plan_summary = json.loads((paths.logs / "plan_summary.json").read_text(encoding="utf-8"))
    stats = {
        "plan_id": plan_summary.get("plan_id"), "target_conv_hours": cfg["target_conv_hours"],
        "smoke_test": bool(cfg.get("smoke_test")), "parked_out_of_plan": parked, "restored_into_plan": restored,
        "status_counts": dict(status),
        "samples_by_split": {k: len(v) for k, v in manifests.items()},
        "conv_hours_by_split": {k: hours(v) for k, v in manifests.items()},
        "total_hours_incl_prefix": round(sum(m["num_frames"] for m in allrows) / FRAME_RATE / 3600, 2),
        "conv_hours_total": hours(allrows),
        "conv_hours_by_scenario": hours(allrows, "scenario"),
        "conv_hours_by_persona_category": hours(allrows, "persona_category"),
        "conv_hours_by_agent_voice_kind": hours(allrows, "agent_voice_kind"),
        "conv_hours_by_noise_class": hours(allrows, "noise_class"),
        "conv_hours_by_mic": hours(allrows, "mic"),
        "conv_hours_by_start_policy": hours(allrows, "start_policy"),
        "samples_per_ui_voice": dict(Counter(m["agent_voice_id"] for m in allrows if m["agent_voice_kind"] == "ui")),
        "distinct": {"agent_voices": len({m["agent_voice_id"] for m in allrows}),
                     "personas": len({m["persona_id"] for m in allrows}),
                     "tts_user_speakers": len({m["user_speaker"] for m in allrows}),
                     "real_user_speakers": len({m["real_speaker"] for m in allrows if m["real_speaker"]})},
        "mean_stats": {k: round(sum((m["stats"].get(k) or 0) for m in allrows) / max(1, len(allrows)), 3)
                       for k in ("agent_speech_ratio", "user_speech_ratio", "agent_words", "user_turns", "n_barge",
                                 "n_bc", "response_rate")},
    }
    atomic_write_json(paths.logs / "dataset_stats.json", stats)
    card = ["# PersonaPlex student distillation dataset", "",
            f"Config: `{cfg['version']}` seed {cfg['seed']}. Generated by teacher self-play with "
            f"`{cfg['teacher']['hf_repo']}`.", "",
            "## Format", "- `<split>/manifest.jsonl` + `<split>/codes/<sample_id>.pt` (LongTensor [17, num_frames]),",
            "  read directly by `distill/data/dataset.py:TeacherTokenDataset`.",
            "- Row 0 text (teacher inner-monologue tokens, prompt tokens in the prefix), rows 1-8 agent Mimi codes, "
            "rows 9-16 user Mimi codes.",
            "- Frames [0, prefix_len) are the voice+persona prompt (prefix_len = 136 + n_persona_tokens), laid out "
            "exactly as LMGen.step_system_prompts produces them at inference.", "",
            "## Statistics", "```json", json.dumps(stats, indent=1), "```", "",
            "## Sources and licenses", "See `logs/sources/*.json` for every source's license string and row counts.",
            "Sources whose HF card has no license are marked 'verify before commercial use'."]
    atomic_write_bytes(paths.root / "DATASET_CARD.md", "\n".join(card).encode("utf-8"))
    log.info("assembled: %s", json.dumps(stats)[:3000])
    stage_log(paths, "assemble", "done", samples=len(allrows), conv_hours=stats["conv_hours_total"],
              not_generated=status["not_generated"], rejected=status["rejected"], retry_pending=status["retry"])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/validate.py
"""Stage `validate`: automatic validation of the assembled dataset.

Writes logs/validation/report.json, report.md, failures.jsonl. Exit code 1 if any CRITICAL check fails.
With --fix, invalid samples are moved to _rejected/<split>/ and removed from the manifests (a .bak is kept).
"""
import argparse
import hashlib
import json
import os
import random
import shutil
import sys
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

from .common import (P, FRAME_RATE, N_CODEBOOKS, AUDIO_CARD, TEXT_CARD, TEXT_PAD, TEXT_EPAD, SILENCE_TOKENS,
                     SINE_TOKENS, VOICE_FRAMES, SILENCE_SPACER, PREFIX_BASE, SPLITS, load_cfg, get_logger, stage_log,
                     read_jsonl, write_jsonl, atomic_write_json, atomic_write_bytes)

REQUIRED = {"sample_id": str, "voice_prompt": str, "persona_text": str, "scenario": str, "num_frames": int,
            "transition_frames": list}


def _check_one(args):
    """Runs in a worker process. Returns (sample_id, split, [issues], info)."""
    import torch
    root, split, m, persona_tokens, voice_codes_rel = args
    paths = P(root)
    issues = []
    info = {}
    f = paths.codes_path(split, m["sample_id"])
    try:
        raw = f.read_bytes()
        import io
        c = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
    except Exception as e:
        return m["sample_id"], split, [f"CORRUPT: cannot load ({type(e).__name__})"], info
    if not isinstance(c, torch.Tensor):
        return m["sample_id"], split, ["MALFORMED: not a tensor"], info
    if c.dtype != torch.long:
        issues.append(f"MALFORMED: dtype {c.dtype} (expected int64)")
    N = m["num_frames"]
    if tuple(c.shape) != (N_CODEBOOKS, N):
        return m["sample_id"], split, issues + [f"MALFORMED: shape {tuple(c.shape)} != (17, {N})"], info
    sha = hashlib.sha1(c.numpy().tobytes()).hexdigest()
    info["sha1"] = sha
    if m.get("codes_sha1") and sha != m["codes_sha1"]:
        issues.append("CORRUPT: sha1 differs from generation record")
    if int(c[0].min()) < 0 or int(c[0].max()) >= TEXT_CARD:
        issues.append("INVALID: text id out of range")
    if int(c[1:].min()) < 0 or int(c[1:].max()) >= AUDIO_CARD:
        issues.append("INVALID: audio id out of range (special/initial token leaked)")
    T = len(persona_tokens)
    pl = PREFIX_BASE + T
    if m.get("prefix_len") != pl:
        issues.append(f"INVALID: prefix_len {m.get('prefix_len')} != 136+{T}")
    sine = torch.tensor(SINE_TOKENS).view(8, 1)
    sil = torch.tensor(SILENCE_TOKENS).view(8, 1)
    vc = torch.load(paths.root / voice_codes_rel, weights_only=True).long()
    a0, a1 = VOICE_FRAMES - 1, VOICE_FRAMES - 1 + SILENCE_SPACER
    if not torch.equal(c[1:9, :a0], vc[:, 1:VOICE_FRAMES]):
        issues.append("INVALID: prefix voice codes != agent voice clip codes")
    if not torch.equal(c[9:17, :pl], sine.expand(8, pl)):
        issues.append("INVALID: prefix user channel != SINE")
    if not torch.equal(c[1:9, a0:pl], sil.expand(8, pl - a0)):
        issues.append("INVALID: prefix agent spacer/text frames != SILENCE")
    if not torch.equal(c[0, a1:a1 + T], torch.tensor(persona_tokens, dtype=torch.long)):
        issues.append("INVALID: prefix persona tokens mismatch")
    if not (torch.all(c[0, :a1] == TEXT_PAD) and torch.all(c[0, a1 + T:pl] == TEXT_PAD)):
        issues.append("INVALID: prefix text padding mismatch")
    conv = c[:, pl:]
    if (conv[9:17] == sine).all(dim=0).any():
        issues.append("INVALID: SINE frames inside the conversation user channel")
    # user channel must be a real encoding (varies over time); agent must speak
    if conv.shape[1] > 0:
        uniq_user = len({tuple(conv[9:17, k].tolist()) for k in range(0, conv.shape[1], 5)})
        info["user_unique_frames_1in5"] = uniq_user
        if uniq_user < 5:
            issues.append("INVALID: user channel (near-)constant -- dead microphone encoding")
        txt = conv[0]
        text_ratio = float(((txt != TEXT_PAD) & (txt != TEXT_EPAD)).float().mean())
        info["agent_text_ratio"] = round(text_ratio, 3)
        if text_ratio < 0.02:
            issues.append("QUALITY: agent text channel (almost) silent")
        words = txt[(txt != TEXT_PAD) & (txt != TEXT_EPAD)].tolist()
        info["agent_text_hash"] = hashlib.sha1(str(words).encode()).hexdigest()[:16]
        if len(words) >= 40:
            grams = Counter(tuple(words[i:i + 4]) for i in range(len(words) - 3))
            top = grams.most_common(1)[0][1] * 4 / len(words)
            info["top4_token_frac"] = round(top, 3)
            if top > 0.4:
                issues.append(f"QUALITY: degenerate repeated agent text (top 4-gram {top:.2f})")
    tf = m["transition_frames"]
    if tf != sorted(tf) or (tf and (tf[0] < 0 or tf[-1] >= N)):
        issues.append("MALFORMED: transition_frames unsorted or out of range")
    return m["sample_id"], split, issues, info


def alignment_test(cfg, paths, rows, log, k=8):
    """Does the recorded frame layout match what LMModel.forward_train (= the training code) expects?

    The only way the recording can be misaligned is BETWEEN codebooks with different delays (delays =
    [0, 0,1..1, 0,1..1]): an off-by-one would put the delay-1 codebooks one frame early or late relative to the
    delay-0 codebooks and the text. So we score the teacher's own recordings with forward_train as recorded, and
    with only the delay-1 codebooks shifted by +1 and by -1 frame. The recorded layout must give clearly the
    lowest cross-entropy on the delayed agent codebooks (and on text). Shifting ALL codebooks together would keep
    a consistent sequence and prove nothing, so that is deliberately not what is tested.
    """
    import torch
    import torch.nn.functional as F
    from moshi.models import loaders
    lm = loaders.get_moshi_lm(cfg["teacher"]["moshi_weight"], device="cuda")
    lm.eval()
    delayed = [kk for kk, d in enumerate(lm.delays) if d > 0]                 # codebook rows with delay 1
    agent_delayed = [kk for kk in delayed if 1 <= kk <= 8]                  # agent rows 2..8
    out = []

    def scores(codes, pl):
        with torch.no_grad():
            o = lm.forward_train(codes)                                     # LMOutput(logits, mask, text_logits, text_mask)
        tl = o.text_logits[0, 0, pl:].float()
        tm = o.text_mask[0, 0, pl:]
        text = F.cross_entropy(tl[tm], codes[0, 0, pl:][tm]).item()
        ces = []
        for row in agent_delayed:                                           # logits index = row - audio_offset(1)
            lg = o.logits[0, row - 1, pl:].float()
            m = o.mask[0, row - 1, pl:]
            if m.any():
                ces.append(F.cross_entropy(lg[m], codes[0, row, pl:][m]).item())
        return text, sum(ces) / max(1, len(ces))

    def shift_delayed(codes, pl, by):
        sh = codes.clone()
        for row in delayed:
            if by > 0:
                sh[:, row, pl + by:] = codes[:, row, pl:-by]
            else:
                sh[:, row, pl:by] = codes[:, row, pl - by:]
        return sh

    for m in rows[:k]:
        c = torch.load(paths.codes_path(m["split"], m["sample_id"]), weights_only=True).long()[None].cuda()
        pl = m["prefix_len"]
        t0, a0 = scores(c, pl)
        tp, ap = scores(shift_delayed(c, pl, +1), pl)
        tn, an = scores(shift_delayed(c, pl, -1), pl)
        r = {"sample_id": m["sample_id"], "text_ce": round(t0, 3), "text_ce_shift+1": round(tp, 3),
             "text_ce_shift-1": round(tn, 3), "agent_delayed_ce": round(a0, 3), "agent_delayed_ce_shift+1": round(ap, 3),
             "agent_delayed_ce_shift-1": round(an, 3)}
        r["aligned_best"] = a0 < min(ap, an)
        out.append(r)
        log.info("alignment %s", r)
    del lm
    torch.cuda.empty_cache()
    passed = bool(out) and all(r["aligned_best"] for r in out)
    return passed, out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--fix", action="store_true")
    ap.add_argument("--procs", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    ap.add_argument("--alignment-test", action="store_true")
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("validate", paths.logs / "validation" / "validate.log")
    stage_log(paths, "validate", "running")
    checks, failures = {}, []

    def check(name, ok, critical, detail=None):
        checks[name] = {"ok": bool(ok), "critical": critical, "detail": detail}
        log.info("%s %s %s", "PASS" if ok else ("FAIL" if critical else "WARN"), name, "" if detail is None else
                 json.dumps(detail)[:500])

    personas = {p["persona_id"]: p for p in read_jsonl(paths.meta / "personas.jsonl")}
    voices = {v["clip_id"]: v for v in read_jsonl(paths.meta / "agent_voices.jsonl")}
    manifests = {s: read_jsonl(paths.split_dir(s) / "manifest.jsonl") for s in SPLITS}
    metas = {s: {m["sample_id"]: m for m in read_jsonl(paths.meta / f"sample_meta_{s}.jsonl")} for s in SPLITS}

    # ---- A. structure
    bad_fields = []
    for s, rows in manifests.items():
        for m in rows:
            for k, t in REQUIRED.items():
                if not isinstance(m.get(k), t):
                    bad_fields.append((s, m.get("sample_id"), k))
    check("manifest_required_fields", not bad_fields, True, bad_fields[:20])
    ids = [m["sample_id"] for rows in manifests.values() for m in rows]
    dup_ids = [k for k, v in Counter(ids).items() if v > 1]
    check("unique_sample_ids", not dup_ids, True, dup_ids[:20])
    orphans = []
    for s in SPLITS:
        d = paths.split_dir(s) / "codes"
        listed = {m["sample_id"] for m in manifests[s]}
        if d.exists():
            orphans += [f"{s}/{p.stem}" for p in d.glob("*.pt") if p.stem not in listed]
    check("no_orphan_code_files", not orphans, False, {"count": len(orphans), "examples": orphans[:10]})

    # ---- B. per-sample tensor checks (parallel)
    jobs = []
    for s, rows in manifests.items():
        for m in rows:
            meta = metas[s].get(m["sample_id"], {})
            p = personas.get(m.get("persona_id"))
            v = voices.get(m.get("agent_clip_id"))
            if p is None or v is None:
                failures.append({"sample_id": m["sample_id"], "split": s, "issues": ["MALFORMED: unknown persona/voice id"]})
                continue
            jobs.append((str(paths.root), s, m, p["tokens"], v["codes"]))
    infos = {}
    with ProcessPoolExecutor(max_workers=args.procs) as ex:
        for sid, s, issues, info in ex.map(_check_one, jobs, chunksize=64):
            infos[sid] = info
            if issues:
                failures.append({"sample_id": sid, "split": s, "issues": issues})
    fail_kinds = Counter(i.split(":")[0] for f in failures for i in f["issues"])
    # CORRUPT / MALFORMED / INVALID = a broken file or a wrong frame layout -> always critical (a pipeline bug).
    # QUALITY = the teacher's own output was silent/degenerate in that sample -> removed by --fix, not a failure of
    # the dataset (critical only when not fixed).
    hard = [f for f in failures if any(not i.startswith("QUALITY") for i in f["issues"])]
    check("per_sample_hard_checks_(corrupt/malformed/invalid_layout)", not hard, True,
          {"failed_samples": len(hard), "by_kind": {k: v for k, v in fail_kinds.items() if k != "QUALITY"},
           "checked": len(jobs), "examples": [f["issues"][:2] for f in hard[:3]]})
    n_quality = sum(1 for f in failures if f not in hard)
    check("per_sample_quality_(removed_by_--fix)", n_quality == 0 or args.fix, not args.fix,
          {"quality_failures": n_quality, "removed": bool(args.fix),
           "examples": [f["issues"][:2] for f in failures if f not in hard][:3]})

    # ---- C. duplicates
    sha = defaultdict(list)
    txt = defaultdict(list)
    for sid, info in infos.items():
        if "sha1" in info:
            sha[info["sha1"]].append(sid)
        if "agent_text_hash" in info:
            txt[info["agent_text_hash"]].append(sid)
    exact = [v for v in sha.values() if len(v) > 1]
    near = [v for v in txt.values() if len(v) > 1]
    check("no_exact_duplicates", not exact, True, exact[:10])
    check("no_duplicate_agent_transcripts", len(near) <= 0.001 * max(1, len(ids)), False,
          {"groups": len(near), "examples": near[:5]})
    for grp in exact:
        for sid in grp[1:]:
            s = next(k for k, rows in manifests.items() if any(m["sample_id"] == sid for m in rows))
            failures.append({"sample_id": sid, "split": s, "issues": ["DUPLICATE: identical codes"]})

    # ---- D. leakage between train and held-out splits
    def ents(split_names, getter):
        out = set()
        for s in split_names:
            for sid, meta in metas[s].items():
                out |= set(getter(meta))
        return out
    tr, ho = ["train", "train_long"], ["val", "val_long", "test", "test_deploy"]
    leak = {}
    for name, g in {
        "personas": lambda m: [m["persona_id"]],
        "tts_speakers": lambda m: [m["user_speaker"]],
        "real_speakers": lambda m: [m["real_speaker"]] if m.get("real_speaker") else [],
        "questions": lambda m: [t["ref"] for t in m["turns"] if t.get("content", "").endswith("question") and t.get("ref")],
        "chains": lambda m: [t["ref"] for t in m["turns"] if t.get("content") == "chain" and t.get("ref")],
        "extra_agent_voices": lambda m: [m["agent_voice_id"]] if m["agent_voice_kind"] == "extra" else [],
        "ui_heldout_clips": lambda m: [m["agent_clip_id"]] if m["agent_voice_kind"] == "ui" and
        voices.get(m["agent_clip_id"], {}).get("split") == "heldout" else [],
    }.items():
        inter = ents(tr, g) & ents(ho, g)
        leak[name] = len(inter)
    check("no_split_leakage", all(v == 0 for v in leak.values()), True, leak)

    # ---- E. size and distribution vs targets
    def hrs(rows):
        return sum(m["conv_frames"] for m in rows) / FRAME_RATE / 3600
    bad_ids = {f["sample_id"] for f in failures}
    good_rows = [m for rows in manifests.values() for m in rows if m["sample_id"] not in bad_ids]
    total_h = hrs(good_rows)
    check("total_conversation_hours_>=_target", total_h >= cfg["target_conv_hours"], True,
          {"conv_hours": round(total_h, 2), "target": cfg["target_conv_hours"]})
    for s in ["train", "val", "test"]:
        check(f"split_{s}_nonempty", len(manifests[s]) > 0, True, len(manifests[s]))
    sw = cfg["scenario_weights"]
    tot_w = sum(sw.values())
    sc = Counter(m["scenario"] for m in good_rows)
    dev = {k: round(sc.get(k, 0) / max(1, len(good_rows)) - w / tot_w, 3) for k, w in sw.items()}
    check("scenario_mix_within_5pp", all(abs(v) <= 0.05 for v in dev.values()), False, dev)
    ui = Counter(m["agent_voice_id"] for m in good_rows if m["agent_voice_kind"] == "ui")
    check("ui_voice_balance_(max/min<=1.5)", ui and max(ui.values()) / max(1, min(ui.values())) <= 1.5, False, dict(ui))
    short = [m["sample_id"] for m in good_rows if m["conv_frames"] < 300]
    check("no_too_short_conversations_(<24s)", not short, True, short[:10])
    resp = [m["stats"].get("response_rate") for m in good_rows if m["stats"].get("response_rate") is not None]
    check("teacher_responsiveness_mean>=0.6", (sum(resp) / max(1, len(resp))) >= 0.6, False,
          {"mean_response_rate": round(sum(resp) / max(1, len(resp)), 3)})

    # ---- F. compatibility with the training code (distill/data/dataset.py)
    try:
        sys.path.insert(0, cfg["repo_dir"])
        sys.path.insert(0, os.path.join(cfg["repo_dir"], "moshi"))
        from distill.data.dataset import TeacherTokenDataset, collate_chunks
        compat = {}
        for s in SPLITS:
            if not manifests[s]:
                continue
            ds = TeacherTokenDataset(str(paths.split_dir(s)))
            idx = random.Random(0).sample(range(len(ds)), min(16, len(ds)))
            frames = cfg["frames"]["long"] if s.endswith("_long") else cfg["frames"]["main"]
            b = collate_chunks([ds[i] for i in idx], frames)
            compat[s] = {"len": len(ds), "batch_codes": list(b["codes"].shape), "dtype": str(b["codes"].dtype)}
        check("training_loader_compatibility", True, True, compat)
    except Exception as e:
        check("training_loader_compatibility", False, True, repr(e))

    # ---- G. alignment (optional, GPU)
    if args.alignment_test:
        rows = [m for m in manifests["val"] if m["sample_id"] not in bad_ids] or good_rows
        try:
            passed, detail = alignment_test(cfg, paths, rows, log)
        except Exception as e:                 # never lose the report because one check crashed
            import traceback
            passed, detail = False, {"error": repr(e), "traceback": traceback.format_exc()[-1500:]}
        check("teacher_alignment_test", passed, True, detail)

    # ---- report / fix
    write_jsonl(paths.logs / "validation" / "failures.jsonl", failures)
    crit = [k for k, v in checks.items() if not v["ok"] and v["critical"]]
    report = {"checks": checks, "critical_failures": crit, "n_failed_samples": len(failures),
              "conv_hours_valid": round(total_h, 2)}
    if args.fix and failures:
        by_split = defaultdict(set)
        for f in failures:
            by_split[f["split"]].add(f["sample_id"])
        for s, sids in by_split.items():
            mp = paths.split_dir(s) / "manifest.jsonl"
            shutil.copy(mp, mp.with_suffix(".jsonl.bak"))
            keep = [m for m in manifests[s] if m["sample_id"] not in sids]
            write_jsonl(mp, keep)
            rej = paths.root / "_rejected" / s
            rej.mkdir(parents=True, exist_ok=True)
            for sid in sids:
                src = paths.codes_path(s, sid)
                if src.exists():
                    shutil.move(str(src), str(rej / f"{sid}.pt"))
        report["fixed"] = {s: len(v) for s, v in by_split.items()}
    atomic_write_json(paths.logs / "validation" / "report.json", report)
    md = ["# Dataset validation report", "", f"Valid conversation hours: **{report['conv_hours_valid']}**", "",
          "| check | result | critical | detail |", "|---|---|---|---|"]
    for k, v in checks.items():
        md.append(f"| {k} | {'PASS' if v['ok'] else 'FAIL'} | {v['critical']} | "
                  f"{json.dumps(v['detail'])[:300].replace('|', '/') if v['detail'] is not None else ''} |")
    atomic_write_bytes(paths.logs / "validation" / "report.md", "\n".join(md).encode("utf-8"))
    stage_log(paths, "validate", "done" if not crit else "failed", critical_failures=crit,
              failed_samples=len(failures), conv_hours_valid=round(total_h, 2))
    print(json.dumps({"critical_failures": crit, "failed_samples": len(failures),
                      "conv_hours_valid": round(total_h, 2)}, indent=1))
    sys.exit(1 if crit else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/ppxdata/qa.py
"""Stage `qa` (GPU optional): listen-level checks on the stereo previews (L = user mic as fed, R = agent).

For N preview samples per split:
  * Whisper transcript of the AGENT channel vs the teacher's own text tokens (WER): catches text/audio drift.
  * Whisper transcript of the USER channel vs the scripted TTS turn texts (WER): catches broken user audio,
    wrong clips, or noise/codec settings that make the user unintelligible.
Writes logs/qa/qa_report.json and per-sample transcripts. Metrics are measured, never assumed.
"""
import argparse
import json
import re

import numpy as np

from .common import P, SR, TEXT_PAD, TEXT_EPAD, load_cfg, get_logger, stage_log, read_jsonl, atomic_write_json, read_audio

NORM = re.compile(r"[^a-z0-9' ]+")


def norm(t):
    return " ".join(NORM.sub(" ", t.lower()).split())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--per-split", type=int, default=25)
    ap.add_argument("--model", default="openai/whisper-small.en")
    args = ap.parse_args()
    cfg = load_cfg(args.root)
    paths = P(args.root)
    log = get_logger("qa", paths.logs / "qa" / "qa.log")
    stage_log(paths, "qa", "running")
    import torch
    import jiwer
    import sentencepiece
    from scipy.signal import resample_poly
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model=args.model,
                   device=0 if torch.cuda.is_available() else -1)
    sp = sentencepiece.SentencePieceProcessor(cfg["teacher"]["tokenizer"])
    report = {"per_split": {}, "samples": []}
    for split in ["test_deploy", "test", "val", "train"]:
        metas = {m["sample_id"]: m for m in read_jsonl(paths.meta / f"sample_meta_{split}.jsonl")}
        files = sorted((paths.preview / split).glob("*.flac"))[: args.per_split]
        agent_w, user_w = [], []
        for f in files:
            m = metas.get(f.stem)
            if m is None:
                continue
            x, sr = read_audio_stereo(f)
            codes = torch.load(paths.codes_path(split, f.stem), weights_only=True)
            pl = m["prefix_len"]
            ref_agent = "".join(sp.id_to_piece(int(t)) for t in codes[0, pl:] if int(t) not in (TEXT_PAD, TEXT_EPAD))
            ref_agent = norm(ref_agent.replace("▁", " "))

            def tx(ch):
                y = resample_poly(x[:, ch], 16000, sr).astype(np.float32)
                return norm(asr({"raw": y, "sampling_rate": 16000}, chunk_length_s=30, batch_size=8)["text"])
            hyp_agent, hyp_user = tx(1), tx(0)
            ref_user = norm(" ".join(t["text"] for t in m["turns"] if t["source"] == "tts"))
            wa = jiwer.wer(ref_agent, hyp_agent) if ref_agent and hyp_agent else None
            # user WER is computed against ALL planned turns; turns never reached in time inflate it (deletions),
            # so it's reported as recall-oriented: fraction of scripted words recognized.
            rec = None
            if ref_user and hyp_user:
                rw, hw = ref_user.split(), set(hyp_user.split())
                rec = sum(w in hw for w in rw) / len(rw)
            if wa is not None:
                agent_w.append(wa)
            if rec is not None:
                user_w.append(rec)
            report["samples"].append({"split": split, "sample_id": f.stem, "scenario": m["scenario"],
                                      "agent_wer_vs_own_text": None if wa is None else round(wa, 3),
                                      "user_script_word_recall": None if rec is None else round(rec, 3),
                                      "agent_hyp": hyp_agent[:500], "agent_ref": ref_agent[:500],
                                      "user_hyp": hyp_user[:500]})
            log.info("%s %s agent_wer=%s user_recall=%s", split, f.stem, wa, rec)
        report["per_split"][split] = {
            "n": len(files),
            "agent_wer_vs_own_text_median": float(np.median(agent_w)) if agent_w else None,
            "user_script_word_recall_median": float(np.median(user_w)) if user_w else None}
    atomic_write_json(paths.logs / "qa" / "qa_report.json", report)
    log.info("qa: %s", json.dumps(report["per_split"]))
    stage_log(paths, "qa", "done", **{f"{k}_agent_wer": v["agent_wer_vs_own_text_median"]
                                       for k, v in report["per_split"].items()})


def read_audio_stereo(path):
    import soundfile as sf
    x, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if x.shape[1] == 1:
        x = np.concatenate([x, x], axis=1)
    return x, sr


if __name__ == "__main__":
    main()

## 6. Write `config.json` + environment snapshot (logs/)

In [ ]:
import json, sys, importlib
os.makedirs(os.path.join(CODE_DIR, "ppxdata"), exist_ok=True)
sys.path[:0] = [CODE_DIR, os.path.join(REPO_DIR, "moshi"), REPO_DIR]
for m in [k for k in sys.modules if k == "ppxdata" or k.startswith("ppxdata.")]:
    del sys.modules[m]
from ppxdata.config_default import default_config, smoke_overrides
from ppxdata.common import P, env_versions, stage_done, stable_int, atomic_write_json, read_jsonl, now

def deep_update(d, u):
    for k, v in u.items():
        d[k] = deep_update(d.get(k, {}), v) if isinstance(v, dict) and isinstance(d.get(k), dict) else v
    return d

cfg = default_config(DATASET_ROOT, REPO_DIR, TEACHER, target_hours=TARGET_HOURS)
if SMOKE_TEST:
    cfg = deep_update(cfg, smoke_overrides())
cfg = deep_update(cfg, CONFIG_OVERRIDES)
cfg["real_speech"]["my_recordings_dir"] = MY_RECORDINGS_DIR
paths = P(DATASET_ROOT); paths.mkdirs()
cfg_path = pathlib.Path(DATASET_ROOT) / "config.json"
# "design" = everything that changes WHAT a sample is. target_conv_hours is deliberately excluded: the size can
# change at any time (Section 12 re-plans; finished samples are kept because samples are keyed by split+index).
design = lambda c: {k: v for k, v in c.items()
                    if k not in ("teacher", "root", "repo_dir", "gen", "config_hash", "target_conv_hours")}
if cfg_path.exists():
    old = json.loads(cfg_path.read_text())
    if old.get("target_conv_hours") != cfg["target_conv_hours"]:
        print(f"TARGET changed: {old.get('target_conv_hours')} h -> {cfg['target_conv_hours']} h "
              "(Section 12 will re-plan; already finished samples are reused)")
    if design(old) != design(cfg) and stage_done(paths, "plan") and not FORCE_CONFIG_UPDATE:
        raise RuntimeError("config.json design differs from the one the existing plan was built with. Revert your "
                           "changes, or set FORCE_CONFIG_UPDATE=True AND delete _work/plan + generated data.")
    # engine settings (gen.*) and paths may change between runs (e.g. new pod, other GPU) -- keep design, update rest
cfg["config_hash"] = f"{stable_int(json.dumps(design(cfg), sort_keys=True)):x}"
cfg_path.write_text(json.dumps(cfg, indent=1))
atomic_write_json(paths.logs / "run_config.json", {"time": now(), **cfg})
_ver = subprocess.run([sys.executable, "-c", "import json; from ppxdata.common import env_versions; "
                                              "print(json.dumps(env_versions()))"],
                      capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=CODE_DIR)).stdout
atomic_write_json(paths.logs / "env" / f"versions_{os.uname().nodename}.json",
                  json.loads(_ver.strip().splitlines()[-1]) if _ver.strip() else {"error": "version probe failed"})
(paths.logs / "env" / f"pip_freeze_{os.uname().nodename}.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
(paths.logs / "env" / f"nvidia_smi_{os.uname().nodename}.txt").write_text(
    subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"config hash {cfg['config_hash']} | target {cfg['target_conv_hours']} h | smoke={cfg['smoke_test']} -> {cfg_path}")

### Launcher and monitor helpers
Workers are plain subprocesses (`python -m ppxdata.<module>`), one per GPU slot. Work units are claimed with lock
files on the dataset volume, so several pods can share the same volume. `detach=True` workers survive a closed
browser tab or kernel restart; re-attach with `monitor(stage)`.

In [ ]:
import time, glob, signal

_gpu_idx = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                                           capture_output=True, text=True).stdout.split()]
GPUS = _gpu_idx if NUM_GPUS is None else _gpu_idx[:NUM_GPUS]   # counted without initializing CUDA in the kernel

def worker_env(gpu=None):
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([CODE_DIR, os.path.join(REPO_DIR, "moshi"), REPO_DIR])
    env["HF_HOME"] = HF_CACHE_DIR
    env["OMP_NUM_THREADS"] = "4"
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env

def run_single(module, *extra, gpu=None):
    """Runs a single-process stage in the foreground, streaming its output; raises on failure."""
    cmd = [sys.executable, "-m", f"ppxdata.{module}", "--root", DATASET_ROOT, *extra]
    print("$", " ".join(cmd))
    p = subprocess.Popen(cmd, cwd=CODE_DIR, env=worker_env(gpu), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{module} {extra} failed (exit {p.returncode}) -- see {paths.logs}/workers and errors/")

def launch(module, stage, n_workers, use_gpu=True, detach=False):
    procs = []
    for w in range(n_workers):
        wid = WORKER_ID_OFFSET + w
        gpu = GPUS[w % len(GPUS)] if use_gpu else None
        out = open(paths.logs / "workers" / f"{stage}_stdout_w{wid}.log", "a")
        cmd = [sys.executable, "-m", f"ppxdata.{module}", "--root", DATASET_ROOT, "--stage", stage,
               "--worker-id", str(wid), "--num-workers", str(n_workers)]
        procs.append(subprocess.Popen(cmd, cwd=CODE_DIR, env=worker_env(gpu), stdout=out, stderr=subprocess.STDOUT,
                                      start_new_session=detach))
    atomic_write_json(paths.logs / "progress" / f"{stage}_pids_{os.uname().nodename}.json",
                      {"time": now(), "pids": [p.pid for p in procs], "gpus": GPUS, "n": n_workers})
    print(f"launched {n_workers} x {module}:{stage} (detached={detach}) pids={[p.pid for p in procs]}")
    return procs

def unit_status(stage, units):
    """(done, in_progress, total) counted over THIS plan's unit names only (markers of older plans are ignored)."""
    units = list(units)
    done = sum((paths.done / stage / f"{u}.json").exists() for u in units)
    locks = sum((paths.locks / stage / f"{u}.lock").exists() for u in units)
    return done, locks, len(units)

def plan_units(sub, pattern):
    d = paths.plan / sub
    return sorted(f.stem for f in d.glob(pattern)) if d.exists() else []

def progress_table(stage):
    rows = []
    for f in sorted((paths.logs / "progress").glob(f"{stage}_*.json")):
        if "_pids_" in f.name:
            continue
        try:
            rows.append(json.loads(f.read_text()))
        except Exception:
            pass
    return rows

def monitor(stage, units, procs=None, poll=60, planned_hours=None):
    """Prints progress until all units are done (or all local workers exited). Ctrl-C/stop only stops monitoring."""
    t0 = time.time()
    try:
        while True:
            done, running, total = unit_status(stage, units)
            rows = progress_table(stage)
            rate = sum(r.get("conv_hours_per_hour", 0) for r in rows)
            ok_h = sum(r.get("conv_hours_ok_this_run", 0) for r in rows)
            msg = f"[{time.strftime('%H:%M:%S')}] {stage}: units {done}/{total} done, {running} in progress, workers reporting {len(rows)}"
            if rate:
                msg += f" | {rate:.2f} conv-h/h, {ok_h:.1f} h ok this run"
                if planned_hours:
                    msg += f" | ETA ~{max(0, planned_hours - ok_h) / rate:.1f} h (this run's rate)"
            errs = sum(1 for _ in (paths.logs / "errors").glob(f"{stage}_*.jsonl"))
            if errs:
                msg += f" | error files: {errs} (logs/errors)"
            print(msg, flush=True)
            if done >= total:
                break
            if procs is not None and all(p.poll() is not None for p in procs):
                print("all local workers exited", [p.returncode for p in procs])
                break
            time.sleep(poll)
    except KeyboardInterrupt:
        print("monitoring stopped (workers keep running)")
    done, _, total = unit_status(stage, units)
    from ppxdata.common import stage_log
    stage_log(paths, stage, "done" if done >= total else "incomplete", units_done=done, units_total=total,
              monitor_seconds=round(time.time() - t0))
    return done >= total

def stop_stage(stage):
    for f in (paths.logs / "progress").glob(f"{stage}_pids_*.json"):
        for pid in json.loads(f.read_text())["pids"]:
            try:
                os.killpg(os.getpgid(pid), signal.SIGTERM)
            except Exception:
                try: os.kill(pid, signal.SIGTERM)
                except Exception: pass
    print("sent SIGTERM to", stage, "workers (locks of killed units become reclaimable after gen.stale_lock_s)")

def status():
    for st in ["text", "personas", "real_speech_finalize", "noise", "tts_speakers", "voice_codes", "plan",
               "assemble", "validate", "qa"]:
        f = paths.logs / "stages" / f"{st}.json"
        print(f"{st:22s}", json.loads(f.read_text())["status"] if f.exists() else "-")
    for st in ["real_speech", "ui_voices", "tts", "selfplay", "retry"]:
        d = len(list((paths.done / st).glob("*.json"))) if (paths.done / st).exists() else 0
        print(f"{st:22s} units done: {d}")
status()

## 7. Text sources → spoken user questions and multi-turn chains (CPU)
Downloads 12 Q&A/dialogue datasets (schemas verified), cleans them into spoken-style user turns, deduplicates,
and assigns held-out splits by question/chain ID. Per-source row counts, drop reasons and licenses go to
`logs/sources/`.

In [ ]:
if not stage_done(paths, "text"):
    run_single("text_sources")
print(json.dumps(json.loads((paths.logs / "sources" / "text_summary.json").read_text()), indent=1))

## 8. Personas (CPU)
Token counts use the real teacher tokenizer. Each persona's T fixes its prefix length (136 + T).

In [ ]:
if not stage_done(paths, "personas"):
    run_single("personas")
print(json.dumps(json.loads((paths.logs / "sources" / "personas_summary.json").read_text()), indent=1))

## Text-file preview of the generated dataset (for manual inspection)

Dumps every generated USER-side question/chain and every persona/system prompt to plain `.txt` files
under `dataset_text_preview/`, grouped by topic (cryptocurrency, general conversation, Robert/RB Labs
identity, voice-interruption phrase banks, natural-conversation-behavior phrase banks) so you can read
exactly what is being taught before spending GPU time on self-play. The AGENT side is never scripted
(the real teacher answers live in Section 14), so only user turns and persona prompts are dumped here.


In [ ]:
import json
from collections import defaultdict
from ppxdata.planner import (INTERRUPT_PREFIX, INTERRUPT_SHORT, CLARIFY, RAPID_SHORT, NUDGES, BACKCHANNELS,
                              GREETINGS, OPENERS, CLOSERS, FOLLOWUPS_GENERIC, FOLLOWUPS_FINANCE, FOLLOWUPS_CS,
                              FOLLOWUPS_CRYPTO)

preview_root = paths.root / "dataset_text_preview"
preview_root.mkdir(parents=True, exist_ok=True)


def read_jsonl(p):
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]


questions = read_jsonl(paths.meta / "questions.jsonl")
chains = read_jsonl(paths.meta / "chains.jsonl")
personas_rows = read_jsonl(paths.meta / "personas.jsonl")

q_by_domain = defaultdict(list)
for q in questions:
    q_by_domain[q["domain"]].append(q["text"])
c_by_domain = defaultdict(list)
for c in chains:
    c_by_domain[c["domain"]].append(c["turns"])


def write_list(path, lines, header=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        if header:
            f.write(header.rstrip() + "\n" + "=" * 70 + "\n\n")
        for i, l in enumerate(lines, 1):
            f.write(f"{i}. {l}\n")


# 01 -- cryptocurrency knowledge --------------------------------------------------------------------
crypto_dir = preview_root / "01_cryptocurrency"
crypto_dir.mkdir(parents=True, exist_ok=True)
write_list(crypto_dir / "questions.txt", sorted(set(q_by_domain.get("crypto", []))),
          "Cryptocurrency questions (every phrasing the model is trained to answer; the teacher model "
          "generates the actual spoken answer live during self-play, so no answer text is stored here).")
with open(crypto_dir / "multi_turn_chains.txt", "w", encoding="utf-8") as f:
    f.write("Cryptocurrency multi-turn conversations (each numbered block is one chain of USER turns)\n" + "=" * 70 + "\n\n")
    for i, turns in enumerate(c_by_domain.get("crypto", []), 1):
        f.write(f"Chain {i}:\n")
        for j, t in enumerate(turns, 1):
            f.write(f"  turn {j} (user): {t}\n")
        f.write("\n")

# 02 -- general conversation --------------------------------------------------------------------------
gen_dir = preview_root / "02_general_conversation"
write_list(gen_dir / "questions.txt", sorted(set(q_by_domain.get("general", []))),
          "General conversation / assistant-behavior questions (greetings, small talk, meta questions about "
          "the assistant, short acknowledgements). Paired at training time with assistant/casual personas.")

# 03 -- Robert / RB Labs identity -----------------------------------------------------------------------
id_dir = preview_root / "03_identity_robert_rblabs"
id_dir.mkdir(parents=True, exist_ok=True)
write_list(id_dir / "questions.txt", sorted(set(q_by_domain.get("identity", []))),
          "Questions about the model's identity (name, creator, creation date/place, location, RB Labs, "
          "founders). Only answerable correctly when paired with the identity persona below.")
with open(id_dir / "identity_personas.txt", "w", encoding="utf-8") as f:
    idp = [p for p in personas_rows if p["category"] == "identity"]
    f.write(f"Identity system prompts given to the teacher during self-play ({len(idp)} total, showing up to 40 "
            "distinct variants). Each is what the model sees as its <system> prompt; the FAQ list is the "
            "questions that prompt is allowed to answer from its own Information: facts.\n" + "=" * 70 + "\n\n")
    seen_txt = set()
    shown = 0
    for p in idp:
        if p["text"] in seen_txt or shown >= 40:
            continue
        seen_txt.add(p["text"])
        shown += 1
        f.write(f"[{shown}] {p['text']}\n    FAQ: {p['faq']}\n\n")

# 04 -- voice interruption / barge-in behavior -----------------------------------------------------------
int_dir = preview_root / "04_voice_interruption"
int_dir.mkdir(parents=True, exist_ok=True)
with open(int_dir / "interruption_phrases.txt", "w", encoding="utf-8") as f:
    f.write(
        "Voice-interruption (barge-in) training design\n" + "=" * 70 + "\n\n"
        "This is NOT scripted text -- it's a behavior produced by the live self-play simulator "
        "(ppxdata/selfplay.py). In the `barge_in` scenario (14% of samples), the simulated user starts "
        "talking again 0.8-5s into the teacher's live answer; the teacher's audio/text generation is cut off "
        "at that exact frame and the teacher must respond to the NEW user turn instead. The student model is "
        "trained on these exact frame sequences, so it learns to stop an in-progress answer and switch topic "
        "the same way. The lines below are the phrase banks the interrupting user turn is built from.\n\n"
    )
    f.write("Interruption openers (prefixed to a new question, e.g. 'Wait, sorry, what is Ethereum?'):\n")
    for p in INTERRUPT_PREFIX:
        f.write(f"  - {p}\n")
    f.write("\nStand-alone interruption phrases (used verbatim as the interrupting turn):\n")
    for p in INTERRUPT_SHORT:
        f.write(f"  - {p}\n")
    f.write("\nClarification / repeat requests (repeat_clarify scenario):\n")
    for p in CLARIFY:
        f.write(f"  - {p}\n")
    f.write("\n\"Are you there?\" nudges (used when the teacher goes silent):\n")
    for p in NUDGES:
        f.write(f"  - {p}\n")

# 05 -- natural conversational behavior --------------------------------------------------------------
beh_dir = preview_root / "05_natural_conversation_behavior"
beh_dir.mkdir(parents=True, exist_ok=True)
with open(beh_dir / "behavior_phrase_banks.txt", "w", encoding="utf-8") as f:
    f.write("Natural conversational-behavior phrase banks used by the planner\n" + "=" * 70 + "\n\n")
    for name, bank in [("Backchannels (mm-hmm, yeah, ...)", BACKCHANNELS),
                       ("Greetings (opens a turn)", GREETINGS),
                       ("Openers (so, okay so, um, ...)", OPENERS),
                       ("Closers (ends a conversation)", CLOSERS),
                       ("Rapid short answers (yes/no/okay style)", RAPID_SHORT),
                       ("Generic follow-up questions", FOLLOWUPS_GENERIC),
                       ("Finance follow-up questions", FOLLOWUPS_FINANCE),
                       ("Customer-service follow-up questions", FOLLOWUPS_CS),
                       ("Crypto follow-up questions", FOLLOWUPS_CRYPTO)]:
        f.write(f"\n{name}:\n")
        for p in bank:
            f.write(f"  - {p}\n")

# README -------------------------------------------------------------------------------------------
with open(preview_root / "README.txt", "w", encoding="utf-8") as f:
    counts = {d: len(v) for d, v in sorted(q_by_domain.items(), key=lambda kv: -len(kv[1]))}
    chain_counts = {d: len(v) for d, v in sorted(c_by_domain.items(), key=lambda kv: -len(kv[1]))}
    persona_counts = dict(sorted(
        {cat: sum(1 for p in personas_rows if p["category"] == cat)
         for cat in {p["category"] for p in personas_rows}}.items(), key=lambda kv: -kv[1]))
    f.write(
        "Human-readable dump of the generated text data, for manual inspection before the (GPU) self-play "
        "stage renders any audio. This mirrors exactly what's in meta/questions.jsonl, meta/chains.jsonl and "
        "meta/personas.jsonl -- nothing here is invented separately.\n\n"
        f"Questions by domain: {json.dumps(counts, indent=1)}\n\n"
        f"Multi-turn chains by domain: {json.dumps(chain_counts, indent=1)}\n\n"
        f"Personas by category: {json.dumps(persona_counts, indent=1)}\n\n"
        "Folders:\n"
        "  01_cryptocurrency/            every crypto question phrasing + multi-turn crypto chains\n"
        "  02_general_conversation/      general/assistant conversational Q&A\n"
        "  03_identity_robert_rblabs/    Robert / RB Labs identity questions + the system prompts that teach them\n"
        "  04_voice_interruption/        the barge-in / 'stop' vocabulary and how the behavior is actually trained\n"
        "  05_natural_conversation_behavior/  backchannels, greetings, closers, follow-ups, short answers\n\n"
        "Note: the AGENT side (the actual spoken answers) is never scripted -- the real PersonaPlex-7B teacher "
        "answers live during self-play (Section 14), which is what the student model imitates. Only the USER "
        "side and the system/persona prompts are fixed text, shown here.\n"
    )

print(f"Wrote text preview to {preview_root}")
for sub in sorted(preview_root.iterdir()):
    if sub.is_dir():
        for f in sorted(sub.iterdir()):
            print(" ", f.relative_to(paths.root), f"({f.stat().st_size} bytes)")
    else:
        print(" ", sub.relative_to(paths.root), f"({sub.stat().st_size} bytes)")


## 9. Real human speech (CPU, parallel, resumable per parquet file) + noise corpus
EdAcc, VoxPopuli (accented), VCTK and LibriTTS-R are read from parquet. They provide real user turns, real
backchannels, and 10-second clips for the extra agent voices. MUSAN (11 GB) is downloaded with resume support.

In [ ]:
from ppxdata.audio_sources import list_units, unit_id
rs_units = [unit_id(n, f) for n, f in list_units(cfg)]
print(len(rs_units), "parquet units")
if unit_status("real_speech", rs_units)[0] < len(rs_units):
    procs = launch("audio_sources", "real_speech", min(CPU_WORKERS, len(rs_units)), use_gpu=False)
    monitor("real_speech", rs_units, procs, poll=30)
if not stage_done(paths, "real_speech_finalize"):
    run_single("audio_sources", "--stage", "real_speech_finalize")
print(json.dumps(json.loads((paths.logs / "sources" / "real_speech_summary.json").read_text()), indent=1)[:3000])

In [ ]:
if not stage_done(paths, "noise"):
    run_single("audio_sources", "--stage", "noise")
print(json.loads((paths.logs / "sources" / "noise_summary.json").read_text()))

## 10. TTS speaker bank (CPU)

In [ ]:
if not stage_done(paths, "tts_speakers"):
    run_single("tts", "--stage", "tts_speakers")
print(json.loads((paths.logs / "sources" / "tts_speakers_summary.json").read_text()))

## 11. UI voice clips (GPU, teacher) + voice-prompt codes (GPU, Mimi)
Renders `ui_clips_per_voice` × 10 s clips per UI voice with the teacher (production: 18 voices × 6; smoke: 2 × 2). Clip 0 is the deployment clip →
`voices/agent/ui_deploy/<NAME>.wav`. Then every agent voice clip is Mimi-encoded exactly like inference does it.

In [ ]:
ui_units = [f"{v}_{k}" for v in cfg["agent_voices"]["ui_voices"] for k in range(cfg["agent_voices"]["ui_clips_per_voice"])]
if unit_status("ui_voices", ui_units)[0] < len(ui_units):
    procs = launch("voices", "ui_voices", min(len(ui_units), UI_VOICE_PER_GPU * len(GPUS)))
    monitor("ui_voices", ui_units, procs, poll=30)
assert unit_status("ui_voices", ui_units)[0] == len(ui_units), \
    "some UI voice clips failed -- see logs/errors/ui_voices_*.jsonl and re-run"
if not stage_done(paths, "voice_codes"):
    run_single("voices", "--stage", "voice_codes", gpu=GPUS[0])
print(json.loads((paths.logs / "sources" / "agent_voices_summary.json").read_text()))
from IPython.display import Audio, display
display(Audio(str(paths.agent_deploy / f"{cfg['agent_voices']['ui_voices'][0]}.wav")))

## 12. Plan every sample (CPU, deterministic)
Re-plans automatically when `TARGET_HOURS` (or the design) changed. Finished samples are reused because every
sample is keyed by (split, index): 200 h is an exact subset of 500 h.

In [ ]:
_ps = paths.logs / "plan_summary.json"
_prev = json.loads(_ps.read_text()) if _ps.exists() else {}
if (not stage_done(paths, "plan") or _prev.get("target_conv_hours") != cfg["target_conv_hours"]
        or _prev.get("config_hash") != cfg["config_hash"]):
    if _prev:
        print(f"re-planning: {_prev.get('target_conv_hours')} h -> {cfg['target_conv_hours']} h "
              "(finished samples are kept and will be skipped by self-play)")
    run_single("planner")
plan_summary = json.loads(_ps.read_text())
print(json.dumps(plan_summary, indent=1)[:6000])
assert plan_summary["conv_hours_total"] >= cfg["target_conv_hours"], "plan is below the target -- check the catalogs"

## 13. Render user speech (GPU, Kokoro, detached, resumable per shard)

In [ ]:
tts_units = plan_units("tts_jobs", "*.jsonl")
if unit_status("tts", tts_units)[0] < len(tts_units):
    tts_procs = launch("tts", "tts", min(len(tts_units), TTS_PER_GPU * len(GPUS)), detach=True)
    monitor("tts", tts_units, tts_procs, poll=30 if SMOKE_TEST else 60)
failed = sum(len(read_jsonl(f)) for f in (paths.logs / "errors").glob("tts_failed_*.jsonl"))
print(f"TTS failures: {failed} (those turns are skipped at generation time, never faked)")

## 14. Teacher self-play generation (GPU, multi-GPU, detached, resumable)

This is the long stage. Each worker loads the teacher once and processes batches of up to 8 samples that share
one prompt schedule. CPU audio preparation for the next batches is prefetched. Every sample is checked frame by
frame before it's saved: prefix layout vs voice codes / persona tokens, user codes vs the codes fed in, SINE
leaks, token ranges, degenerate or silent teacher. Failures are retried with a new seed (Section 15).

- **Stop anytime:** `stop_stage("selfplay")`.
- **Resume:** just re-run this cell. Completed groups are never redone, and abandoned locks are reclaimed.
- **More pods:** run this notebook on another pod with the same volume and a different `WORKER_ID_OFFSET`.

In [ ]:
sp_units = plan_units("shards", "shard_*.json")
planned_h = json.loads((paths.logs / "plan_summary.json").read_text())["conv_hours_total"]
if unit_status("selfplay", sp_units)[0] < len(sp_units):
    sp_procs = launch("selfplay", "selfplay", min(len(sp_units), SELFPLAY_PER_GPU * len(GPUS)), detach=True)
else:
    sp_procs = None

In [ ]:
# MONITOR (safe to interrupt and re-run at any time; workers keep running)
monitor("selfplay", sp_units, sp_procs, poll=30 if SMOKE_TEST else 120, planned_hours=planned_h)
for r in progress_table("selfplay"):
    print({k: r.get(k) for k in ("worker", "batch_size", "frames_per_s", "peak_mem_gb", "conv_hours_per_hour", "ok", "not_ok", "last_unit")})

## 15. Retry failed samples (new seed, up to `gen.max_retries`)

In [ ]:
for round_ in range(cfg["gen"]["max_retries"]):
    run_single("selfplay", "--stage", "build_retry")
    retry_units = plan_units("retry", "retry_*.json")
    if unit_status("retry", retry_units)[0] >= len(retry_units):
        break
    rp = launch("selfplay", "retry", min(len(retry_units), SELFPLAY_PER_GPU * len(GPUS)))
    monitor("retry", retry_units, rp, poll=30)
print(json.loads((paths.logs / "retry_plan.json").read_text()))

## 16. Assemble → validate → QA
- **assemble** builds `<split>/manifest.jsonl` from the self-play results. The codes files are already in place.
- **validate** checks every sample. `--fix` moves invalid samples to `_rejected/` (manifests get a `.bak`).
  `--alignment-test` runs the teacher's own `forward_train` on the recorded sequences: aligned CE must beat
  one-frame-shifted CE, which proves the frames line up as the training code expects.
- **qa** transcribes stereo previews with Whisper (agent audio vs its own text tokens; user audio vs the scripted text).

In [ ]:
run_single("assemble")
print(json.dumps(json.loads((paths.logs / "dataset_stats.json").read_text()), indent=1)[:5000])

In [ ]:
try:
    run_single("validate", "--fix", "--alignment-test", "--procs", str(min(16, max(2, CPU_WORKERS))), gpu=GPUS[0])
finally:
    _rep = paths.logs / "validation" / "report.md"
    print(_rep.read_text() if _rep.exists() else "no report written -- see logs/validation/validate.log")

In [ ]:
run_single("qa", "--per-split", "3" if SMOKE_TEST else "25", gpu=GPUS[0])
print(json.dumps(json.loads((paths.logs / "qa" / "qa_report.json").read_text())["per_split"], indent=1))

## 17. Listen to a few samples (left = user microphone as fed, right = agent)

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
for split in ["test_deploy", "test", "val"]:
    for f in sorted((paths.preview / split).glob("*.flac"))[:2]:
        m = next((x for x in read_jsonl(paths.split_dir(split) / "manifest.jsonl") if x["sample_id"] == f.stem), None)
        if m:
            print(split, f.stem, m["scenario"], m["persona_category"], m["agent_voice_id"], m["noise_class"], m["stats"])
            display(Audio(str(f)))

## 18. Final dataset layout + training commands

In [ ]:
def tree(root, depth=2, max_files=4):
    root = pathlib.Path(root)
    for p in sorted(root.iterdir()):
        if p.name.startswith("."):
            continue
        n = sum(1 for _ in p.rglob("*")) if p.is_dir() else 0
        print(("  " * (depth - 2)) + ("[D] " if p.is_dir() else "    ") + p.name + (f"  ({n} entries)" if p.is_dir() else ""))
for s in ["train", "val", "test", "test_deploy", "train_long", "val_long"]:
    man = paths.split_dir(s) / "manifest.jsonl"
    rows = read_jsonl(man)
    print(f"{s:12s} {len(rows):6d} samples  {sum(r['conv_frames'] for r in rows)/12.5/3600:7.1f} conv-h  -> {man}")
tree(DATASET_ROOT)
print(f"""
TRAIN (current distill/train.py, unchanged):
  python -m distill.train --student-config student_ppx_s \\
      --teacher-checkpoint {TEACHER['moshi_weight']} \\
      --data-dir {DATASET_ROOT}/train --output-dir /workspace/distill_run_v3/checkpoints \\
      --chunk-frames 1000 --batch-size 8 --total-steps <see notes> --lr 3e-4 --resume
LONG-CONTEXT FINE-TUNE (optional, student max_frames=1500):
  ... --data-dir {DATASET_ROOT}/train_long --chunk-frames 1500
STREAMING the result: set STUDENT_VOICE_DIR = "{paths.agent_deploy}" in PersonaPlex_Student_Streaming_RunPod.ipynb
""")

## 19. Smoke-test verdict (and run summary)
Shows each stage's status, the validation result and the QA numbers. In smoke mode a PASS means the full
pipeline works on this pod: set `SMOKE_TEST = False` (and `TARGET_HOURS`) in the CONFIG cell, restart the kernel
and run again from Section 1.

In [ ]:
checks = []
def _stage(name):
    f = paths.logs / "stages" / f"{name}.json"
    return json.loads(f.read_text()).get("status") if f.exists() else None
for st in ["text", "personas", "real_speech", "real_speech_finalize", "noise", "tts_speakers", "ui_voices",
           "voice_codes", "plan", "tts", "selfplay", "assemble", "validate", "qa"]:
    checks.append((f"stage {st}", _stage(st) == "done", _stage(st)))
rep = json.loads((paths.logs / "validation" / "report.json").read_text()) if (paths.logs / "validation" / "report.json").exists() else {}
checks.append(("validation: no critical failures", rep.get("critical_failures") == [], rep.get("critical_failures")))
align = rep.get("checks", {}).get("teacher_alignment_test", {})
checks.append(("teacher alignment test", bool(align.get("ok")), "see report.md"))
loader = rep.get("checks", {}).get("training_loader_compatibility", {})
checks.append(("loads with distill/data/dataset.py", bool(loader.get("ok")), loader.get("detail")))
stats = json.loads((paths.logs / "dataset_stats.json").read_text()) if (paths.logs / "dataset_stats.json").exists() else {}
for s in ["train", "val", "test", "test_deploy", "train_long", "val_long"]:
    n = stats.get("samples_by_split", {}).get(s, 0)
    checks.append((f"split {s} has samples", n > 0, n))
errs = sorted(f.name for f in (paths.logs / "errors").glob("*.jsonl"))
checks.append(("no worker error files", not [e for e in errs if not e.startswith("tts_failed_")], errs[:8]))
width = max(len(c[0]) for c in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:{width}s}  {'' if ok else detail}")
passed = all(ok for _, ok, _ in checks)
print()
print(f"valid conversation hours: {rep.get('conv_hours_valid')} (target {cfg['target_conv_hours']})")
if SMOKE_TEST:
    print("SMOKE TEST PASSED -> set SMOKE_TEST = False and TARGET_HOURS in Section 2, restart the kernel, run from Section 1."
          if passed else "SMOKE TEST FAILED -> fix the FAIL lines above (see logs/) before starting the real run.")
else:
    print("RUN COMPLETE" if passed else "RUN INCOMPLETE -- see FAIL lines; re-running the stage cells resumes the work.")

### Notes for training on this dataset
- **Works unchanged with the current `distill/train.py`:** `--data-dir <root>/train --chunk-frames 1000`. The
  manifest has all the fields `TeacherTokenDataset` reads. Extra fields (e.g. `prefix_len`) are ignored.
- **Still recommended in `train.py` for best results:**
  1. Shuffle samples every epoch. It currently walks through them in a fixed order.
  2. In the P3/P4 on-policy rollout, keep feeding the **recorded user channel** (`codes[:, 9:17]`) instead of
     `SINE_TOKENS` after the prompt, and start the rollout after `prefix_len`.
  3. Optionally exclude prefix frames (`< prefix_len`, stored in the manifest) from CE (not from KL/hidden losses).
  4. Save/export `depformer.*` if Phase 4 unfreezes it. It is currently dropped by `FROZEN_PREFIXES`.
- **Evaluate** on `val/` during training and on `test/` + `test_deploy/` at the end: the voices, personas,
  questions and speakers in them were never seen in training.